In [ ]:
from IPython.display import HTML, display,clear_output, FileLink,Markdown, Javascript
import numpy as np
import ipysheet
from ipysheet import current, sheet, cell, to_dataframe, from_dataframe
import requests
import ipywidgets as widgets
import requests as r
from bs4 import BeautifulSoup
import functools
import hashlib
import pandas as pd
from pandas import option_context
import io
import os
from os import path
from ipywidgets import FileUpload, Output, Checkbox, Layout, Button, Box, VBox,HBox, HTML,interact, interactive, fixed, interact_manual,interactive_output
import docx
from docx import Document
from docx.shared import Inches, Cm, Pt, Mm
from docx.enum.section import WD_ORIENT,WD_SECTION
import pymysql
import warnings
warnings.filterwarnings('ignore')
#from ipywidgets import interact, interactive, fixed, interact_manual,interactive_output
#from ipyvuetify.extra import FileInput
#from datetime import datetime
#import shutil
#import os.pathF
#from os import path
#import subprocess
#global vcf_dis
#import lxml.html
#from lxml import etree
#import re
#import Bio
#from Bio import Entrez
#from functools import reduce
#import time
#from bs4 import *
#import threading
#from IPython.display import display

#User Input file upload path
USER_INPUT_FOLDER_PATH='/home/ubuntu/clinical_reporting/s3/input_files/'

#Temporary foler path 
TEMP_FOLDER_PATH = USER_INPUT_FOLDER_PATH+'Annotations_temporary/'

#External data files path (dir for  Clinvar, Gene_Description , Variant Summary...)
EXTERNAL_FILES_FOLDER_PATH='/home/ubuntu/clinical_reporting/s3/installations/external_datafiles/'

#SIFT Indel installation path
SIFT_INDEL_INSTALL_PATH=' /home/ubuntu/SIFT'
#SIFT_INDEL_INSTALL_PATH='/home/ubuntu/clinical_reporting/s3/installations/SIFT'
#SIFT_Indel path for hg37
#SIFT_INDEL_DB_PATH='/home/ubuntu/clinical_reporting/s3/installations/SIFT/SIFT_INDEL_HG37'
SIFT_INDEL_DB_PATH='/home/ubuntu/SIFT/SIFT_INDEL_HG37'

#SIFT_INDEL_DB_PATH='/home/ubuntu/SIFT/SIFT_INDEL_HG37'
#SIFT 4GPATH
SIFT_4G_INSTALL_PATH = '/home/ubuntu/clinical_reporting/s3/installations/SIFT_4G'

#dbNSFP installation path
dbNSFP_PATH='$HOME/clinical_reporting/s3/installations/dbNSFP/dbNSFP4.3a'


# Preview of the input file and process button functions: Previews the input file that the user uploads and the process button starts annotating the input VCF the process button appears only when the input file is not present in the token_filename_map.txt file 

In [ ]:
#preview the input file taken from the user, 10 rows are displayed
def preview_input_file(input_file):
    def read_vcf(path):
        with open(path, 'r') as f:
            lines = [l for l in f if not l.startswith('##')]
        vcf_dis= pd.read_csv(
            io.StringIO(''.join(lines)),
            dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str,
                   'QUAL': str, 'FILTER': str, 'INFO': str},
            sep='\t'
        )
        return vcf_dis
  #process button appears only when the input file from the user is not present in the token_filename_map.txt file
    def on_process_button_clicked(b):
        clear_output(wait=True)
        token(input_file)
        #warnings.filterwarnings('ignore')
        previewButton1 = widgets.Button(description = "Refresh")
        previewButton1.on_click(functools.partial(click_refresh, file_name=input_file))
        display(previewButton1)
        annotations(input_file)
        #previewButton2 = widgets.Button(description = "Upload another vcf file")
        #previewButton2.on_click(anothervcffile)
        #display(previewButton2)
        #submit_another_file()

    vcf_display=read_vcf(USER_INPUT_FOLDER_PATH+input_file)
    count_row = vcf_display.shape[0]
    print('PREVIEW OF THE VCF FILE:',input_file)
    print('Total no. of rows: ',count_row)

    vcf_display=vcf_display.head(10)
    vcf_display.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
    display(vcf_display.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7'),('text-align', 'left')]}]).hide_index())
    #filesubmission(file)
    string1 = input_file
    file1 = open(USER_INPUT_FOLDER_PATH+"token_filename_map.txt", "r")
    readfile = file1.read()
    if string1 in readfile:
        status(input_file)
    else: 
        print('Start the file annotations')
        previewButton = widgets.Button(description = "Process File")
        previewButton.on_click(on_process_button_clicked)
        display(previewButton)
    file1.close() 
   
    return

# Upload input file function: user can input the VCF

In [ ]:
#Global variable used to store user input file name
input_file_name=''

def uploadInputFile():
    def show_content(change):
        #f = open('input1.vcf','wb')
        #print('=================',change['new'])
        #print('-----',list(change['new']['metadata']['name']))
        #print(list(change['new'])[0]['content'])
        #print(list(change['new'].values())[0]['content'])
        for key in list(change['new']):
            #print (key)
            #print(change['new'][key])
            #print(change['new'][key]['metadata'])
            #print(change['new'][key]['content'])
            global input_file_name
            y=hashlib.md5(change['new'][key]['content']).hexdigest()
            #print(y)
            input_file_name=key[:key.rfind('.')]+'_'+y+'.vcf'
            input_file_name=input_file_name.replace('(', '').replace(')', '').replace(' ', '_')
            input_file_name=input_file_name.replace(" ", "_")
            #print(input_file_name)
            f = open(USER_INPUT_FOLDER_PATH+input_file_name,'wb')

            f.write(change['new'][key]['content'])
            f.close()
        vcf_display=read_vcf(USER_INPUT_FOLDER_PATH+input_file_name)
        vcf_display= vcf_display.rename(columns={"#CHROM": "CHROM"})
        vcf_display.CHROM = pd.to_numeric(vcf_display.CHROM, errors='coerce')
        vcf_display = vcf_display.sort_values(by =['CHROM', 'POS'])
        vcf_display.CHROM = vcf_display.CHROM.fillna(-1)
        vcf_display = vcf_display.astype({'CHROM': 'int'})
        vcf_display['CHROM'] = vcf_display['CHROM'].astype(str) 
        vcf_display['CHROM']=vcf_display['CHROM'].replace(str(-1),'X')
        vcf_display= vcf_display.rename(columns={"CHROM": "#CHROM"})
        vcf_display.columns= vcf_display.columns.str.rstrip()
        with open(USER_INPUT_FOLDER_PATH+input_file_name, "r") as rf, open(USER_INPUT_FOLDER_PATH+'sampletext_copy'+input_file_name+'.txt', "w") as wf:
            for line in rf:
                if '#CHROM' in line:
                    break
                wf.write(line)
            f.close()
      
        vcf_display.to_csv(USER_INPUT_FOLDER_PATH+'sampletext_copy'+input_file_name+'.txt',sep="\t" , mode='a', index=False, header=True)
        file_new=input_file_name.replace('.vcf','')
        os.rename(USER_INPUT_FOLDER_PATH+'sampletext_copy'+input_file_name+'.txt',USER_INPUT_FOLDER_PATH+file_new+'_sorted.vcf')
        os.remove(USER_INPUT_FOLDER_PATH+input_file_name)
        os.rename(USER_INPUT_FOLDER_PATH+file_new+'_sorted.vcf',USER_INPUT_FOLDER_PATH+file_new+'.vcf')
                #print("Input File Written")
        #preview_input_file(file_new+'_sorted.vcf')
        preview_input_file(file_new+'.vcf')
        #print('Done')
        
        #return input_file_name
        #return file_new+'_sorted.vcf'
        return file_new+'.vcf'
    
    w = FileUpload(multiple=False,accept='.vcf')
    w.observe(show_content, 'value')

    display(w)
    return

# Reading the vcf file, SIFT indel, SIFT 4G and dbNSFP annotation functions:

In [ ]:
def read_vcf(input_file):
        with open(input_file, 'r') as f:
            lines = [l for l in f if not l.startswith('##')]
        data_frame= pd.read_csv(
            io.StringIO(''.join(lines)),
            dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str,
                   'QUAL': str, 'FILTER': str, 'INFO': str},
            sep='\t'
        )
        
        return data_frame

#SIFT_Indel input file genration and running the process for given input file
def SIFT_indel(input_file):
    data_frame=read_vcf(USER_INPUT_FOLDER_PATH+input_file)
    ref=list(data_frame['REF'])
    alt=list(data_frame['ALT'])
    pos=list(data_frame['POS'])
    chrom=list(data_frame['#CHROM'])

    indel_list_pos=[]
    indel_list_chrom=[]
    indel_list_ref=[]
    indel_list_alt=[]

    for i in range(len(ref)):
        if len(ref[i])!=len(alt[i])and ','not in alt[i]:
            indel_list_pos.append(pos[i])
            indel_list_chrom.append(chrom[i])
            indel_list_ref.append(ref[i])
            indel_list_alt.append(alt[i])

    insertion_list_pos=[]
    insertion_list_ref=[]
    insertion_list_alt=[]
    insertion_list_chrom=[]

    deletion_list_chrom=[]
    deletion_list_pos=[]
    deletion_list_ref=[]
    deletion_list_alt=[]

    for i in range(len(indel_list_ref)):
        if len(indel_list_ref[i])< len(indel_list_alt[i]):
            insertion_list_chrom.append(indel_list_chrom[i])
            insertion_list_pos.append(indel_list_pos[i])
            insertion_list_ref.append(indel_list_ref[i])
            insertion_list_alt.append(indel_list_alt[i])
        else:
            deletion_list_chrom.append(indel_list_chrom[i])
            deletion_list_pos.append(indel_list_pos[i])
            deletion_list_alt.append(indel_list_alt[i])
            deletion_list_ref.append(indel_list_ref[i])

    def split(word):
        return [char for char in word]

    sift_format_deletion=[]
    for i in range(len(deletion_list_ref)):
        sub=len(deletion_list_ref[i])-len(deletion_list_alt[i])
        del_ref=split(deletion_list_ref[i])
        del_alt=split(deletion_list_alt[i])
        for j in range(len(del_alt)):
            del_ref.remove(del_alt[j])

        pos2=deletion_list_pos[i]+sub
        sift_format_deletion.append(deletion_list_chrom[i]+','+str(deletion_list_pos[i])+','+str(pos2)+','+'1'+','+'/')

    inserts=[]
    sift_format_insertion=[]

    for i in range(len(insertion_list_alt)):
        ins_strings=''
        sub=len(insertion_list_alt[i])-len(insertion_list_ref[i])
        ins_ref=split(insertion_list_ref[i])
        ins_alt=split(insertion_list_alt[i])
        for j in range(len(ins_ref)):
            ins_alt.remove(ins_ref[j])
        for k in range(len(ins_alt)):
            ins_strings=ins_strings+ins_alt[k]
        
        sift_format_insertion.append(insertion_list_chrom[i]+','+str(insertion_list_pos[i])+','+str(insertion_list_pos[i])+','+'1'+','+ins_strings)

    final_list=sift_format_insertion+sift_format_deletion

    #input file prepared for SIFT Indel process
    sift_input_file = TEMP_FOLDER_PATH+"indel_file_"+input_file+".txt" 
    textfile = open(sift_input_file, "w")
    for element in final_list:
        textfile.write(element + "\n")
    textfile.close()
    
    #path="indel_file"+file+".txt"
    sift_process_output=USER_INPUT_FOLDER_PATH+input_file+"_sift_indel_process_out.txt"
    sift_prediction_file=USER_INPUT_FOLDER_PATH+'SIFT_indel'+'_'+input_file+'.tsv'
    SIFTIndel_out = USER_INPUT_FOLDER_PATH+input_file+'_SIFTIndel_out.txt'
    SIFTIndel_err = USER_INPUT_FOLDER_PATH+input_file+'_SIFTIndel_err.txt'
    
    command='nohup sh SIFT_indel.sh '+SIFT_INDEL_INSTALL_PATH +' '+sift_input_file+' '+SIFT_INDEL_DB_PATH +' '+TEMP_FOLDER_PATH +' '+ sift_process_output + ' ' + sift_prediction_file +' '+USER_INPUT_FOLDER_PATH+' >'+SIFTIndel_out + ' 2>'+SIFTIndel_err +' &'
    #print("Command :",command)
    r1=os.system(command)

#SIFT 4G processing
def SIFT4G_1(input_file):
    file_new=file.replace('.vcf','')
    
    SIFT4G_out = USER_INPUT_FOLDER_PATH+input_file+'_SIFT4G_out.txt'
    SIFT4G_err = USER_INPUT_FOLDER_PATH+input_file+'_SIFT4G_err.txt'
    file_new=input_file.replace('.vcf','')
    SIFT4G_annotation_file=TEMP_FOLDER_PATH+file_new+'_SIFTannotations.xls'
    SIFT4G='nohup sh SIFT_4G.sh '+SIFT_4G_INSTALL_PATH +' '+USER_INPUT_FOLDER_PATH+' '+input_file+' '+TEMP_FOLDER_PATH+' '+SIFT4G_annotation_file+' >'+SIFT4G_out + ' 2>'+SIFT4G_err +' &'
    #print(s4g)
    r=os.system(SIFT4G)
#dbNSFP processing
def dbNSFP_1(input_file):
    
    dbnsfp_out = USER_INPUT_FOLDER_PATH+input_file+'_dbNSFP_out.txt'
    dbnsfp_err = USER_INPUT_FOLDER_PATH+input_file+'_dbNSFP_err.txt'
    db='nohup sh dbNSFP4.3a.sh '+dbNSFP_PATH +' '+USER_INPUT_FOLDER_PATH+input_file+' >'+dbnsfp_out + ' 2>'+dbnsfp_err +' &'
   
    result=os.system(db)
    
'''
def polyphen_1(file):
    output = open(file_path+'Annotations_temporary'+'/'+file+'_PolyphenInput.txt', 'w')
    input_vcf = open(file_path+file,'r')
    #Expecting CHROM  POS ID  REF ALT
    #Using CHROM , POS, REF and ALT
    for line in input_vcf:
        if(line[0] == '#'):
            continue
        token = line.split('\t')
        output.write('chr'+token[0]+':'+token[1]+' '+token[3]+'/'+token[4]+'\r')

    input_vcf.close()
    output.close()
    #print("DONE")

    #output = file_path+'/Annotations_temporary'+'/PolyphenInput_'+file+'.txt'
    output = file_path+'Annotations_temporary'+'/'+file+'_PolyphenInput.txt'
    input = open(output,'r').read()


    files = {
        '_ggi_project': (None, 'PPHWeb2'),
        '_ggi_origin': (None, 'query'),
        '_ggi_target_pipeline': (None, '1'),
        'MODELNAME': (None, 'HumVar'),
        'UCSCDB': (None, 'hg19'),
        'SNPFILTER': (None, '0'),
        'SNPFUNC': (None, ''),
        '_ggi_batch_file': (output, open(output, 'rb')),
    }

    response = requests.post('http://genetics.bwh.harvard.edu/ggi/cgi-bin/ggi2.cgi', files=files)

    #print(response.content)

    soup = BeautifulSoup(response.content,"html.parser")
    inp = soup.find("input")
    token = inp.get("value")
    #print(token)

    while True:
        time.sleep(20)
        baseUrl = 'http://genetics.bwh.harvard.edu/ggi/pph2/'
        #token 
        addUrl1 = '/1/completed.txt'
        r1 = r.get(baseUrl+token+addUrl1)
        r1f = BeautifulSoup(r1.content,"html.parser")
        inp1 = r1f.find("h1")
        #val = inp1.get("value")
        if inp1 is None:
            #If completed.txt file created, process is done.
            break

    addUrl2 = '/1/pph2-short.txt'
    addUrl3 = '/1/pph2-full.txt'
    addUrl4 = '/1/pph2-snps.txt'
    addUrl5 = '/1/pph2-log.txt'

    r2 = r.get(baseUrl+token+addUrl2)
    r3 = r.get(baseUrl+token+addUrl3)
    r4 = r.get(baseUrl+token+addUrl4)
    r5 = r.get(baseUrl+token+addUrl5)   

    
    print(r1.content)
    print('-------------------------------------------------------------------------')
    print(r2.content)
    print('-------------------------------------------------------------------------')
    print(r3.content)
    print('-------------------------------------------------------------------------')
    print(r4.content)
    print('-------------------------------------------------------------------------')
    print(r5.content)
    
    with open(file_path+"/"+file+"_Polyphen_short.txt", "w") as f:
        f.write(r2.text)
    with open(file_path+"/"+file+"_Polyphen_full.txt", "w") as f:
        f.write(r3.text)
    with open(file_path+"/"+file+"_Polyphen_snps.txt", "w") as f:
        f.write(r4.text)
    with open(file_path+"Annotations_temporary"+"/"+file+"_Polyphen_log.txt", "w") as f:
        f.write(r5.text)

    #print snps dataframe

    df4 = pd.read_csv(r'snps.txt', sep="\t")

    df4.style.set_properties(**{'border': '1.3px solid green',
                              'color': 'magenta'})
    df4.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}])
'''


# annotation function runs SIFT indel, SIFT 4G and dbNSFP processes. Refresh function displays the current status of the process. Status function checks the status of the annotated file. Anothervcffile function helps the user to upload another vcf when the process is complete. 

In [ ]:
#annotations runs SIFT indel, SIFT4G and dbNSFP processes
def annotations(input_file):
    print('SIFT Indel annotations have begun for ', input_file)
    SIFT_indel(input_file)
    print('SIFT 4G annotations have begun for',input_file)
    SIFT4G_1(input_file)
    print('dbNSFP annotations have begun for',input_file)
    dbNSFP_1(input_file)
    #polyphen_1(file)
    
#token stores the user input file names into token_filename_map.txt file
def token(input_file):
    with open(USER_INPUT_FOLDER_PATH+"token_filename_map.txt", "r+") as f:
        if input_file in f.read():
            #print("file ",file,"already exists")
            f.close()
        else:
            with open(USER_INPUT_FOLDER_PATH+"token_filename_map.txt", "a+") as f:
                f.write('\n'+input_file)
                f.close()
#refresh function gives current status of the annotated file
def refresh(input_file):
    clear_output(wait=True)
    status(input_file)
#staus checks whether or not the file annotations is complete
def status(input_file):
    file_new=input_file.replace('.vcf','')
    SIFT_indel_annoation_file=os.path.exists(USER_INPUT_FOLDER_PATH+'SIFT_indel_'+input_file+'.tsv')
    SIFT4G_annoation_file=os.path.exists(USER_INPUT_FOLDER_PATH+file_new+'_SIFTannotations.xls')
    dbNSFP_annoation_file=os.path.exists(USER_INPUT_FOLDER_PATH+file_new+'.vcf_dbNSFP.tsv')
    display_refresh=False
    uploadotherfile=False
    
    if SIFT_indel_annoation_file==True and SIFT4G_annoation_file ==True and dbNSFP_annoation_file==True:
        print('Annotations are complete. Obtain the results from: https://3.132.17.87:8888/notebooks/jupyter_notebooks/Clinical%20reporting%20workflow-MERGED/Notebook2-MERGED.ipynb')
        uploadotherfile=True
        #print('Annotations are complete. Obtain the results from: https://3.132.17.87:8888/notebooks/jupyter_notebooks/Clinical%20reporting%20workflow-MERGED/Notebook2-MERGED.ipynb')
    elif SIFT_indel_annoation_file==False:
        print('SIFT, SIFT 4G and dbNSFP annotations are going on')
        print('Please come back after 40/60 minutes')
        display_refresh=True
    elif SIFT_indel_annoation_file==True and SIFT4G_annoation_file ==False and dbNSFP_annoation_file==False:
        print('SIFT annotations are completed , SIFT4G and dbNSFP annotations are going on')
        display_refresh=True
    elif SIFT_indel_annoation_file==True and SIFT4G_annoation_file ==True and dbNSFP_annoation_file==False:
        print('SIFT and SIFT 4G annotations are completed , dbNSFP annotations is going on')
        display_refresh=True
    
    if display_refresh==True:
        previewButton1 = widgets.Button(description = "Refresh")
        previewButton1.on_click(functools.partial(click_refresh, file_name=file))
        display(previewButton1)
    if uploadotherfile==True:
        previewButton2 = widgets.Button(description = "Upload another vcf file")
        previewButton2.on_click(anothervcffile)
        display(previewButton2)
        
#anothervcffile user can input another file after the process is done        
def anothervcffile(b):
    uploadInputFile()
#click_refresh is the refresh button 
def click_refresh(b,file_name):
        refresh(file_name)

# Method to select a VCF file from the dropdown to view Variants (Display Table) and filter button allows the display table to be filtered using Widgets, IntSlider, Textbox and SelectMultiple dropdown

In [ ]:

file = ''
#def displayVariants(fname):
    #print("inside displayVariants ::::",fname)

def Select_VCF():
    def Select_VCF_Button(b):
        global file
        #displayVariants(file)
        #print(fileDropdown.value)
        displayVariants(fileDropdown.value)
        file=fileDropdown.value
        #file=fileDropdown.value
        #print("inside MMR_Variants_Button1 ::::",file)
        '''
        progress = widgets.FloatProgress(value=0.0, min=0.0, max=1.0)
    
        def work(progress):
            total = 100
            for i in range(total):
                time.sleep(0.2)
                progress.value = float(i+1)/total

        thread = threading.Thread(target=work, args=(progress,))
        display(progress)
        thread.start()
        
        def Filter_Button(b):
            clear_output(wait=True)
            filter_variants(fileDropdown.value)

        buttonM = widgets.Button(description="Filter Variants")
        buttonM.on_click(Filter_Button)
        display(buttonM)
        return
        '''
    df = pd.read_csv(USER_INPUT_FOLDER_PATH+'token_filename_map.txt', header=0)
    
    #layout = widgets.Layout(width='auto', height='40px')
    fileDropdown= widgets.Dropdown(
        options=df.iloc[:,0],
        #value='2',
        description='Select VCF file:',
        disabled=False,style= {'description_width': 'initial'},layout={'width': 'max-content'})
    display(fileDropdown)
    '''
    fileDropdown = widgets.RadioButtons(#options=rows,
    options=df.iloc[:,0],
    description='Select VCF file:',
    disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    display(fileDropdown)
    '''
    buttonM = widgets.Button(description="Display Variants")
    buttonM.on_click(Select_VCF_Button)
    display(buttonM)
    
    
    return
    

# method to Filter the Display Table (for all entries that have "Passed" the GATK filter) using widgets; IntSlider for Depth, Text Box for Allele Frequency, SelectMultiple dropdown for Clinical Significance and SIFT prediction

In [ ]:
def filter_variants(file):
    global rows_dis, filtered_df
    #def filtered_variants2(file):
    #print(file)
    #news['AF'] = news['AF'].astype(str)
    news99 = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Display_Table.xlsx')  
    news99.insert(loc=0, column='Select', value= '<input type="checkbox" />')
    #for row in news99.itertuples():
        #print(row[row['Select']])
    #news99.insert(loc=0, column='Select', value = True)
    news99['Depth'] = news99['Depth'].astype(int)
    news99['Allele Frequency'] = news99['Allele Frequency'].astype(str)
    news99['Mutation type']=news99['Mutation type'].str.strip()
    news99['Allele Frequency']=news99['Allele Frequency'].str.replace("%","")
    news99['Allele Frequency'] = news99['Allele Frequency'].astype(float)
    news99['Clinical Significance']=news99['Clinical Significance'].str.strip()
    cs=list(news99['Clinical Significance'].unique())
    news99['SIFT prediction']=news99['SIFT prediction'].str.strip()
    sp=list(news99['SIFT prediction'].unique())
    #news99.drop(news99.index[news99['NGS QC filter'] != 'PASS'], inplace=True)
    #fil['Mutation Type'] = fil['Mutation Type'].str.replace(' ', '')
    mtl=list(news99['Mutation type'].unique())
    ngf=list(news99['NGS QC filter'].unique())
    #print(mtl, ngf,news99['Mutation type'].unique(),news99['NGS QC filter'].unique())
    '''
    a = widgets.IntSlider(
        #value=100,
        min=0,
        max=news99['Depth'].max(),
        step=1,
        description='Depth',
        disabled=False,
        continuous_update=False,
        orientation='horizontal',
        readout=True,
        readout_format='d')
    ''' 
    a = widgets.BoundedIntText(
        value=0,
        min=0,
        max=news99['Depth'].max(),
        step=1,
        description='Depth',
        disabled=False)
    '''
    b = widgets.Text(
        value=news99['Allele Frequency'].min(),
        min=0,
        max=10,
        step=1,
        description='AF',
        disabled=False)
    '''
    b=widgets.BoundedFloatText(
        value=0,
        min=0,
        #max=10.0,
        step=0.1,
        description='AF',
        disabled=False)
    
    c = widgets.SelectMultiple(
            options= cs,
            value= cs,
            #rows=10,
            description='Clinical Significance',
            disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'}
        )
    d = widgets.SelectMultiple(
            options= sp,
            value= sp,
            #rows=10,
            description='SIFT prediction',
            disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'}
        )

    '''
    c = widgets.SelectMultiple(
        options=['All','Benign',
           'Conflicting interpretations of pathogenicity, other',
           'Likely benign', 'Uncertain significance', 'Benign/Likely benign',
           'Conflicting interpretations of pathogenicity', 'Pathogenic',
           'Pathogenic/Likely pathogenic', 'drug response',
           'Likely benign, drug response, other', '-', 'not provided'],
        value=['All','Benign',
           'Conflicting interpretations of pathogenicity, other',
           'Likely benign', 'Uncertain significance', 'Benign/Likely benign',
           'Conflicting interpretations of pathogenicity', 'Pathogenic',
           'Pathogenic/Likely pathogenic', 'drug response',
           'Likely benign, drug response, other', '-', 'not provided'],
        #rows=10,
        description='Clinical Significance',
        disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'}
    )
    d = widgets.SelectMultiple(
        options=['All','TOLERATED', 'nan', 'DELETERIOUS', 'damaging', 'neutral',
           'DELETERIOUS (*WARNING! Low confidence)', '-'],
        value=['All','TOLERATED', 'nan', 'DELETERIOUS', 'damaging', 'neutral',
           'DELETERIOUS (*WARNING! Low confidence)', '-'],
        #rows=10,
        description='SIFT prediction',
        disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    
    e=widgets.Dropdown(
    options=news99['Mutation type'].unique(),
    #value='AA_DELETION',
    description='Mutation type:',
    disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    
    f=widgets.Dropdown(
    options=news99['NGS QC filter'].unique(),
    #value='AA_DELETION',
    description='Filter:',
    disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    '''
    e = widgets.SelectMultiple(
        options= mtl, 
        value= mtl,
        #rows=10,
        description='Mutation type:',
        disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    f = widgets.SelectMultiple(
        options=ngf,
        value=ngf,
        #rows=10,
        description='Filter',
        disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    
    button_TMB = widgets.Button(description=" TMB calculator",button_style='info',style=dict(
                    font_weight='bold',
                    #font_variant="small-caps",
                    #text_color='red',
                    text_decoration='underline'))
    #display(button_TMB)
    print('\033[4m'+'\033[1m'+'TMB calculator')
    button = widgets.Button(description=" Display/Update TMB calculator",button_style='info',style=dict(
                    font_weight='bold',
                    #font_variant="small-caps",
                    #text_color='red',
                    text_decoration='underline'),display='flex',layout={'width': 'max-content'})
    output = widgets.Output()
    display(button, output)
   
    def on_button_clicked(b):
        with output:
            #print(len(filtered_df.index))
            #for i in range(len(filtered_df)):
                #display(filtered_df.loc[[i]])
            #display(filtered_df.iloc[[0],0:2])

            #print(filtered_df.loc[1:1])
            #for i in range(len(filtered_df)):
                #print(filtered_df['Select'][i])
            #print(len(filtered_df.loc[filtered_df['Select']==False].index))
            #test_df = filtered_df.drop(filtered_df.loc[filtered_df['Select']==False].index, inplace=True)
            #print(len(test_df.index))
            
            tmb=widgets.BoundedIntText(value=rows_dis,min=0,step=1,max=10000,
            description='Enter Number of Selected mutations:',
            disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
            display(tmb)
            mb=widgets.BoundedFloatText(value=1.4, step=1,description='Enter panel size in Mb:',
            disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
            display(mb)

            button = widgets.Button(description="Calculate TMB")
            output2 = widgets.Output()

            display(button, output2)

            def on_button_clicked(b):
                with output2:
                    clear_output(wait=True)
                    print('\033[1m'+'The TMB value =', (tmb.value/mb.value),'(mut/Mb)')

            button.on_click(on_button_clicked)
            
    #button.on_click(on_button_clicked1)
    button.on_click(on_button_clicked)
    
    items = [a,b,c,d]
    left_box = VBox([items[0], items[1]])
    right_box = VBox([items[2], items[3]])
    box_layout = Layout(display='flex',
                        flex_flow='row',
                        align_items='stretch',justify_content ='space-around', 
                        #border='solid'
                        width='100%')
    #a = HBox([left_box,right_box], layout=box_layout)
    ui = HBox([VBox([items[0], items[2],e]), VBox([items[1], items[3],f])], layout=box_layout)
    #ui = widgets.HBox([a, b, c,d])
    
    #clear_output(wait=True)
    
    
    def reset_button(defaults={}):
        def on_button_clicked(_):
            for k, v in defaults.items(): 
                k.value = v
        button = widgets.Button(description='Reset',button_style='info',style=dict(font_weight='bold'), layout={'width': 'max-content'})
        button.on_click(on_button_clicked)
        display(button)
    
    reset_button(defaults={a: 0, b: '0', c:c.options, d: d.options, e: e.options, f: f.options})
    

    #@interact(Depth=a, AF=b, Clinical_Sig=c, SIFT_Pred=d)
    def show_df(Depth, AF, Clinical_Sig, SIFT_Pred, mut, fil):
        global rows_dis,  filtered_df
        df = news99
        #show_df = news99.insert(loc=0, column='Select', value= '<input type="checkbox" />')
        #news99.drop(news99.index[news99['NGS QC filter'] != 'PASS'], inplace=True)
        df.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
        #print(Clinical_Sig)
        #df1 = df[(df['Depth'] > Depth) & (df['Allele Frequency'] > AF) & (df['Mutation type'] == mut) & (df['NGS QC filter'] == fil)]
        df1 = df[(df['Depth'] > Depth) & (df['Allele Frequency'] > AF)]
        dfe = pd.DataFrame(columns=['Clinical Significance'])
       
        
        l=[]
        for sig in Clinical_Sig:
            #print(sig)
            if sig == 'All':
                l=df['Clinical Significance'].unique()
                break
            else:
                
                #print(sig)
                l.append(sig)
        #print(l)

    
        l1=[]
        for sif in SIFT_Pred:
            #print(sif)
            if sif == 'All':
                l1=df['SIFT prediction'].unique()
                break
            else:
                l1.append(sif)
            #df1 = (df1[(df1['Clinical Significance']==sig)])
        #print(l1)
        
        l2=[]
        for mt in mut:
            #print(mt)
            if mt == 'All':
                l2=df['Mutation type'].unique()
                break
            else:
                l2.append(mt)
        #print(l2)
 
        l3=[]
        for fl in fil:

            #print(fl)
            l3.append(fl)
            #print(l3)

        #df2=((df1[df1['Clinical Significance'].isin(l)]) | (df1[df1['SIFT4G_prediction'].isin(l1)]))
        dff3=(df1[df1['NGS QC filter'].isin(l3)])
        dff=(dff3[dff3['Clinical Significance'].isin(l)])
        #print(dff.shape[0])
        #display(dff)
        #dff1=(df1[df1['SIFT prediction'].isin(l1)]) 
        dff1 = (dff[dff['SIFT prediction'].isin(l1)])
        #df4=pd.merge(dff, dff1, how="outer")
        #display(dff1)
        dff2=(dff1[dff1['Mutation type'].isin(l2)])
        #dff3=(df1[df1['NGS QC filter'].isin(l3)])
        #df5=pd.merge(df4, dff2, how="outer")
        #df2=pd.merge(df5, dff3, how="outer")
        #df2 = pd.concat([df4,dff2, dff3], axis=0)
        
        
        #df2 = df2.astype({'Chromosome Number':'int'})
        #df2.sort_values(by=['Chromosome Number'], inplace=True)
        pd.set_option('display.max_columns', 500)
        #df2 = df2.drop(['Unnamed: 0'], axis = 1)
        #display(df2)
        dff2.fillna('-', inplace=True)
        rows_dis = dff2.shape[0]
        
        print('\n',dff2.shape[0], 'entries are selected after applying the filters')
        dff2['Chromosome Position'] = dff2['Chromosome Position'].astype(int)
        dff2['Chromosome Number'] = dff2['Chromosome Number'].astype(str)
        x3 = dff2.loc[dff2['Chromosome Number'].str.contains('X', case=False, na=False)]
        dff2.drop(dff2.index[dff2['Chromosome Number'] == 'X'], inplace=True)
        dff2.loc[:,"Chromosome Number"] = dff2.loc[:,"Chromosome Number"].astype(str).apply(lambda x3: x3.replace('.0',''))
        dff2['Chromosome Number'] = dff2['Chromosome Number'].astype(int)
        dff2.sort_values(by=['Chromosome Number','Chromosome Position'], inplace=True)
        x3.sort_values(by=['Chromosome Position'], inplace=True)
        filtered_df=dff2.append(x3)
        filtered_df['Allele Frequency'] = filtered_df['Allele Frequency'].astype(str)+'%'
        display(HTML(filtered_df.to_html(render_links=True, escape=False, index=False)))
         
    
    w=interactive_output(show_df, {'Depth':a, 'AF':b, 'Clinical_Sig':c, 'SIFT_Pred':d, 'mut':e, 'fil':f})
    display(ui, w)
    

# Generate Display table of Variants using all the annotations and sources such as ClinVar and export it in .xlsx format or read the already existing Excel file and display dataframe 

In [ ]:

def displayVariants(file):
    
    '''
    token = input('Enter Token ID: ')
    with open(file_path+"token_filename_map.txt") as fin:
     rows = ( line.split('\t') for line in fin )
     d = { row[0]:row[1:] for row in rows }
    Matchfile = [val for key, val in d.items() if token in key]
    file = str(Matchfile[0][0]).strip()
    
    dffile = pd.read_csv('/home/ubuntu/clinical_reporting/s3/input_files/VCF_filename.txt', header=0)
    def f(file):

        dfl1 = dffile[(dffile.iloc[:,0]==file)]
        #print('{}*{}*{}={}'.format(a, b, c, a*b*c))
        #print(file)
        #print(dfl1)
        return file

    interact(f, file = widgets.Dropdown(
        options=dffile.iloc[:,0],
        #value='2',
        description='Select VCF file:',
        disabled=False,))
    
    #print(file)
    
    
    if path.exists(file_path+file+'_Display_Table.xlsx') is True:
        findf = pd.read_excel(file_path+file+'_Display_Table.xlsx')
        findf.fillna('-', inplace=True)
        print('\n No of variants: ',findf.shape[0])
        #display(findf.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index())
        return ((findf.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()), file)
    else:
        print("Please Wait..")
    
    
    try:
        #trying to open a file in read mode
        findf = pd.read_excel(file_path+file+'_Display_Table.xlsx')
        return (findf.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index(), file)
    except FileNotFoundError:
        print("Please Wait..")
        return
    except:
        print("Other error")
        return
   
    #print(file)
    '''
    
    if path.exists(USER_INPUT_FOLDER_PATH+file+'_Display_Table.xlsx') is True:
        findf = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Display_Table.xlsx')
        findf.fillna('-', inplace=True)
        #findf = findf.drop(['Unnamed: 0'], axis = 1)
        print('Reading the already existing Excel file ')
        def Filter_Button(b):
            clear_output(wait=True)
            #filter_variants(fileDropdown.value)
            filter_variants(file)

        buttonM = widgets.Button(description="Filter Variants")
        buttonM.on_click(Filter_Button)
        display(buttonM)
        print('\n No of variants: ',findf.shape[0])
        #with option_context("display.max_rows", 10000):
            #pd.options.display.max_colwidth=80
            #display(HTML(findf.to_html(render_links=True, escape=False,col_space=20,notebook=True,index=False)))
        #display(findf.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index().set_sticky(axis="columns"))
        display(HTML(findf.to_html(render_links=True, escape=False, index=False)))
        #display(HTML(findf.to_html(render_links=True, escape=False,col_space=20,notebook=True,index=False)))
        return (findf.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index().set_sticky(axis="columns"), file)
        #return
    else:
        print("Please Wait..")
    
    
    vcf_display=read_vcf(USER_INPUT_FOLDER_PATH+file)
    

    ref=list(vcf_display['REF'])
    alt=list(vcf_display['ALT'])
    pos=list(vcf_display['POS'])
    fil=list(vcf_display['FILTER'])
    chrom=list(vcf_display['#CHROM'])
    #tksh=vcf_display['TKHS210007670-1A']
    tksh=vcf_display.iloc[:,9]
    #print(tksh)
    dict = {'Chromosome Number': chrom, 'Chromosome Position': pos, 'Reference Nucleotide': ref,'Alternate Nucleotide':alt,'NGS QC filter':fil,'TKSH':tksh}
    df = pd.DataFrame(dict)
    df['Depth'] = ''
    df['Allele Frequency'] = ''
    df['Gene name'] = ''
    df['Amino acid change'] = ''
    df['Mutation Type'] = ''
    df['HGVS nomenclature'] = ''
    df['rs dbSNP'] = ''
    df['SIFT4G prediction'] = ''
    df['PolyPhen 2 prediction'] = ''
    df['Clinical significance'] = ''
    df['Functional consequence'] = ''
    df['Minor allele frequency'] = ''
    df['OMIM disease'] = ''
    df['Comments'] = ''
     
    


    #df[['UK-1', 'UK-2', 'AF','DP','UK-5','UK-6','UK-7','UK-8','UK-9','UK-10']] = df['TKSH'].str.split(':', expand=True)
    #df.drop(['UK-1','UK-2','UK-5','UK-6','UK-7','UK-8','UK-9','UK-10'], axis =1, inplace=True)

    if ('%' in df['TKSH'][0]) == False:
        df[['UK-1', 'UK-2', 'AF','DP','UK-5','UK-6','UK-7','UK-8','UK-9','UK-10']] = df['TKSH'].str.split(':', expand=True)
        df.drop(['UK-1','UK-2','UK-5','UK-6','UK-7','UK-8','UK-9','UK-10'], axis =1, inplace=True)
    else:
        df[['UK-1', 'UK-2', 'UK-3','DP','UK-5','UK-6','AF','UK-8','UK-9','UK-10','UK-11','UK-12','UK-13','UK-14']] = df['TKSH'].str.split(':', expand=True)
        df.drop(['UK-1','UK-2','UK-3','UK-5','UK-6','UK-8','UK-9','UK-10','UK-11','UK-12','UK-13','UK-14'], axis =1, inplace=True)

    

    df['Allele Frequency'] = df['AF']
    df['Depth']=df['DP']
    df.drop(['DP'], axis =1, inplace=True)
    #df_SIFT = pd.read_csv("op_indel_1_new.txt")
    
                
    
    try:
        #trying to open a file in read mode
        #df_SIFT = pd.read_csv(file_path+'SIFT_indel'+file+'.txt',sep='\t') 
        df_SIFT = pd.read_csv(USER_INPUT_FOLDER_PATH+'SIFT_indel_'+file+'.tsv',sep='\t') 
    except FileNotFoundError:
        print("SIFT Annotations in progress..")
        return
    except:
        print("Other error")
        return
    
    #df_SIFT = pd.read_csv(file_path+'SIFT_indel'+file+'.txt',sep='\t')
    #print(df_SIFT)

    coord_SIFT=list(df_SIFT['Coordinates'])
    #df_SIFT = pd.read_csv("op_SIFT_excel.txt",sep='\t',index_col = False)
    #print(df_SIFT)
    chr_no=list(df['Chromosome Number'])
    chr_pos_df=list(df['Chromosome Position'])
    print('Total Number of Variants:',len(chr_no))
    coord_SIFT=list(df_SIFT['Coordinates'])
    effect_SIFT=list(df_SIFT['Effect'])

    combine_no_pos=[]
    for (i,j) in zip(chr_no,chr_pos_df):
        combine_no_pos.append(str(i)+','+str(j))
    #print('combined df',len(combine_no_pos))
    df['snp_pos']=combine_no_pos
    #print(len(effect_SIFT))
    eff=[]
    df_SIFT[['N1', 'N2', 'UK1','UK2','UK3']] = df_SIFT['Coordinates'].str.split(',', expand=True)
    #print(df_SIFT)
    n1=list(df_SIFT['N1'])
    n2=list(df_SIFT['N2'])

    combine_no_pos_SIFT=[]
    for (i,j) in zip(n1,n2):
       combine_no_pos_SIFT.append(str(i)+','+str(j))#+','+str(k))
    #df = df[['snp_pos']]


    df_SIFT['snp_pos']=combine_no_pos_SIFT

    df_SIFT=df_SIFT[['snp_pos','Gene Name','Amino acid position change','Amino acid change','Substitution Type','Effect','Region']]

    #df_SIFT.drop(['Unnamed: 0','Transcript ID', 'Substitution Type','Region','Confidence Score','Classification Path','Causes NMD','Repeat detected','Transcript visualization'], axis = 1)

    merged = pd.merge(df,df_SIFT, how='outer', on='snp_pos')
    merged.drop(['AF','Gene name','Amino acid change_x','Mutation Type','SIFT4G prediction'], axis =1, inplace=True)
    merged.rename(columns = {'Amino acid position change':'Amino acid position','Amino acid change_y':'Amino acid change','Substitution Type':'Mutation type','Effect':'SIFT4G prediction'}, inplace = True)
    
    file_new=file.replace('.vcf','')
    
    try:
        file_11=USER_INPUT_FOLDER_PATH+file_new+'_SIFTannotations.xls'
    except FileNotFoundError:
        print("SIFT4G Annotations in progress..")
        return
    except:
        print("Other error")
        return

    df1= pd.read_csv(file_11, sep='\t')


    chrom=list(df1['CHROM'])
    pos=list(df1['POS'])
    variant=list(df1['VARIANT_TYPE'])
    GENE_NAME=list(df1['GENE_NAME'])
    REGION=list(df1['REGION'])
    SIFT_PREDICTION=list(df1['SIFT_PREDICTION'])
    dbSNP=list(df1['dbSNP'])
    AMINO_POS=list(df1['AMINO_POS'])
    REF_AMINO=list(df1['REF_AMINO'])
    ALT_AMINO=list(df1['ALT_AMINO'])
    reg=list(df1['REGION'])
    comb=[]
    gene_4G=[]
    region_4G=[]
    var=[]
    pred=[]
    aa_pos=[]
    aa_change=[]
    dbsnp=[]
    regi=[]
    #print(len(chrom))

    p=['FRAMESHIFT DELETION','FRAMESHIFT INSERTION','NONFRAMESHIFT DELETION','NONFRAMESHIFT INSERTION']

    for (i,j,k,l,m,n,o,s,q,r,u) in zip(chrom,pos,variant,GENE_NAME,REGION,SIFT_PREDICTION,dbSNP,AMINO_POS,REF_AMINO,ALT_AMINO,reg):
        if k not in p:
            comb.append(str(i)+','+str(j))
            var.append(k)
            gene_4G.append(l)
            region_4G.append(m)
            pred.append(n)
            dbsnp.append(str(o))
            aa_pos.append(str(s))
            aa_change.append(str(q)+str(s)+str(r))
            regi.append(str(u))


    #print(comb)
    #print(comb)
    #print(gene_4G)
    #print(region_4G)
    #print(pred)
    df2 = pd.DataFrame()
    df2['snp_pos']=comb
    df2['gene name']=gene_4G
    df2['mutation_type']=var
    #df2['region_4G']=region_4G
    df2['SIFT_prediction']=pred
    df2['_dbsnp']=dbsnp
    df2['aa_pos']=aa_pos
    df2['aa_change']=aa_change
    df2['region']=regi
    df2=df2[['snp_pos','gene name','mutation_type','SIFT_prediction','_dbsnp','aa_pos','aa_change','region']]
    #print(df2)




    merged_SIFT4G = pd.merge(merged,df2, how='left', on='snp_pos') 
    #print(merged_SIFT4G)
    #boolean = not merged_SIFT4G["Chromosome Position"].is_unique
    boolean = merged_SIFT4G.duplicated(subset=['Chromosome Position']).any() 
    #print(boolean)
    #duplicate_pos=list(merged_SIFT4G['Chromosome Position'])



    #merged_SIFT4G.drop_duplicates()
    merged_SIFT4G.drop_duplicates(subset=['Chromosome Position'],inplace = True)
    duplicate_pos=list(merged_SIFT4G['Chromosome Position'])
    #print(len(duplicate_pos))
    duplicate_no=list(merged_SIFT4G['Chromosome Number'])
    #print(len(duplicate_no))
    duplicate_snp=list(merged_SIFT4G['snp_pos'])
    #print(len(duplicate_snp))
    #merged_SIFT4G.replace(np.nan, '')
    merged_SIFT4G.replace(np.NaN, '', inplace=True)
    merged_SIFT4G.replace(np.nan, '', inplace=True)
    merged_SIFT4G.replace(to_replace =["nan", 'nannannan'], 
                                value ="", inplace=True)
    #merged_SIFT4G['Amino_acid_position'] = merged_SIFT4G['Amino_acid_position'].astype(str).apply(lambda x: x.replace('.0',''))
    merged_SIFT4G["Gene_Name"] = merged_SIFT4G["Gene Name"] + merged_SIFT4G["gene name"]
    merged_SIFT4G["Amino_acid_position"] = merged_SIFT4G["Amino acid position"] + merged_SIFT4G["aa_pos"]
    merged_SIFT4G["Amino_acid_change"] = merged_SIFT4G["Amino acid change"] + merged_SIFT4G["aa_change"]
    merged_SIFT4G["Mutation_type"] = merged_SIFT4G["Mutation type"] + merged_SIFT4G["mutation_type"]
    merged_SIFT4G["SIFT4G_prediction"] = merged_SIFT4G["SIFT4G prediction"] + merged_SIFT4G["SIFT_prediction"]
    merged_SIFT4G.drop(['TKSH','HGVS nomenclature','PolyPhen 2 prediction','Clinical significance','Functional consequence','Minor allele frequency','OMIM disease','Comments','rs dbSNP','Gene Name','Mutation type','Amino acid change','SIFT4G prediction','Amino acid position','gene name','mutation_type','SIFT_prediction','aa_pos','aa_change'], axis =1, inplace=True)
    merged_SIFT4G['HGVS nomenclature']=''
    merged_SIFT4G['PolyPhen 2 prediction']=''
    merged_SIFT4G['Clinical significance']=''
    merged_SIFT4G['Functional consequence']=''
    merged_SIFT4G['Minor allele frequency']=''
    merged_SIFT4G['OMIM disease']=''
    merged_SIFT4G['Comments']=''
    merged_SIFT4G['Amino_acid_position'] = merged_SIFT4G['Amino_acid_position'].astype(str).apply(lambda x: x.replace('.0',''))
    merged_SIFT4G['Amino_acid_change'] = merged_SIFT4G['Amino_acid_change'].astype(str).apply(lambda x: x.replace('.0',''))
    merged_SIFT4G.rename(columns = {'_dbsnp':'dbsnp'}, inplace = True)

    #data['result'] = data['result'].map(lambda x: x.lstrip('+-').rstrip('aAbBcC'))

    merged_SIFT4G['REGION']=merged_SIFT4G['Region']+merged_SIFT4G['region']
    #merged_SIFT4G=merged_SIFT4G[['Chromosome Number','Chromosome Position','Reference Nucleotide','Alternate Nucleotide','NGS QC filter','Depth','Allele Frequency','snp_pos','SIFT4G_prediction','dbSNP','Amino acid position','Gene Name','Amino acid change','REGION','Mutation type','HGVS nomenclature','PolyPhen 2 prediction','Clinical significance','Functional consequence','Minor allele frequency','OMIM disease','Comments']]

    with pd.option_context('display.max_rows', None,
                           'display.max_columns', None,
                           'display.max_colwidth', 400,
                           ):
        #print(df)
        HTML(merged_SIFT4G.to_html(classes='table table-striped'))
        #print(merged)
    #df.style.hide_index()
    merged_SIFT4G.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
    #print(merged_SIFT4G)
    all_freq=list(merged_SIFT4G['Allele Frequency'])
    a_f=[[el] for el in all_freq]
    #print(a_f)
    li=[]
    for i in a_f:
        t=[x for xs in i for x in xs.split(',')]
        li.append(t)
    #print(li)
    min_li=[]
    for i in li:
        min_li.append(min(i))

    merged_SIFT4G['Allele Frequency']=min_li
    #db_path=file_path+file_new+'.vcf.tsv'
    
    try:
        #db_path=file_path+file_new+'.vcf.tsv'
        db_path=USER_INPUT_FOLDER_PATH+file_new+'.vcf_dbNSFP.tsv'
    except FileNotFoundError:
        print("SIFT4G Annotations in progress..")
        return
    except:
        print("Other error")
        return

    
    display1=pd.read_csv(db_path, sep='\t')
    #mutation=list(db['MutationTaster_AAE'])
    #print(display1['MutationTaster_AAE'])
    display1['Mutation_Type'] = ''
    #display(display1)
    #print(display1['MutationTaster_AAE'])
    taster=list(display1['MutationTaster_AAE'])
    tas=[]
    tas1=[]
    tas2=[]
    for ele in taster:
        tas.append(str(ele))
    for i in tas:
        tas1.append(i.replace(".;", ""))
    for i in tas1:
        tas2.append(i.replace(";", " "))
    data_aa= {'dbNSFP Amino acid change':tas2}
    display2_aa= pd.DataFrame(data_aa)
    
    display2_aa['Chromosome Position']=list(display1['POS'])
    #print(tas2)
    #display1['dbNSFP Amino acid change']=list(display1['MutationTaster_AAE'])
    #print(display1['dbNSFP Amino acid change'])
    #display1['dbNSFP Amino acid change']=tas2
    #print("$$$$$$$$",display1['dbNSFP Amino acid change'])
    display1['MutationTaster_AAEE'] = display1['MutationTaster_AAE'].str.replace('.;', '')
    display1.loc[display1['MutationTaster_AAEE'].str.fullmatch('\.'), 'Mutation_Type'] = 'Splicesite'
    display1['MutationTaster_AAEE'] = display1['MutationTaster_AAE'].str.replace(' ', '')
    display1['MutationTaster_AAEE'] = display1['MutationTaster_AAE'].str.replace('  ', '')

    display1['MutationTaster_AAEE'] = display1['MutationTaster_AAE'].str.replace(' \.', '*')
    display1['MutationTaster_AAEE'] = display1['MutationTaster_AAE'].str.replace('  \.', '*')
    display1['MutationTaster_AAEE'] = display1['MutationTaster_AAE'].str.replace('   \.', '*')
    display1['MutationTaster_AAEE'] = display1['MutationTaster_AAE'].str.replace('\.', '.')
    display1['MutationTaster_AAEE'] = display1['MutationTaster_AAE'].str.replace(' \*', '.')
    #display1.loc[display1['MutationTaster_AAEE'].str.fullmatch('\.'), 'Mutation_Type'] = 'Splicesite'
    
    #print(display1['Mutation_Type'])
    #aapos,genename,MutationTaster_AAE,rs_dbSNP,mutationtype(I added this column)
    display1['aapos']=display1['aapos'].str.split(';').apply(set).str.join(',')
    display1['genename']=display1['genename'].str.split(';').apply(set).str.join(',')
    display1=display1[['#CHROM','POS','aapos','genename','MutationTaster_AAE','rs_dbSNP','Mutation_Type']]
    display1 = display1[display1['Mutation_Type'].str.contains('Splicesite')]
    display1.rename(columns = {'POS':'Chromosome Position'}, inplace = True)
    display2_aa.rename(columns = {'POS':'Chromosome Position'}, inplace = True)
    #print(display1)
    
    #merged_aa_dbNSFP= pd.merge(display2_aa,display1, how='left', on='Chromosome Position')
    #print(merged_aa_dbNSFP)
    all_columns = list(display1) # Creates list of all column headers
    display1[all_columns] = display1[all_columns].astype(str)
    all_columns1 = list(display2_aa) # Creates list of all column headers
    display2_aa[all_columns1] = display2_aa[all_columns1].astype(str)
    all_columns2 = list(merged_SIFT4G) # Creates list of all column headers
    merged_SIFT4G[all_columns2] = merged_SIFT4G[all_columns2].astype(str)
    merged_SIFT4G_dbNSFPP = pd.merge(merged_SIFT4G,display1, how='left', on='Chromosome Position')
    merged_SIFT4G_dbNSFP= pd.merge(merged_SIFT4G_dbNSFPP,display2_aa, how='left', on='Chromosome Position')
    #print(merged_SIFT4G_dbNSFP)
    merged_SIFT4G_dbNSFP.replace(np.NaN, '', inplace=True)
    merged_SIFT4G_dbNSFP.replace(np.nan, '', inplace=True)
    merged_SIFT4G_dbNSFP.replace(to_replace =["nan", 'nannannan'], 
                                value ="", inplace=True)
    merged_aa_dbNSFP= pd.merge(display2_aa,display1, how='left', on='Chromosome Position')
    merged_SIFT4G_dbNSFP["dbSNP"] = merged_SIFT4G_dbNSFP["dbsnp"] +' '+merged_SIFT4G_dbNSFP["rs_dbSNP"]
    merged_SIFT4G_dbNSFP["Amino acid position"] = merged_SIFT4G_dbNSFP["Amino_acid_position"]+' '+merged_SIFT4G_dbNSFP["aapos"]
    merged_SIFT4G_dbNSFP["Gene Name"] = merged_SIFT4G_dbNSFP["Gene_Name"]+' '+merged_SIFT4G_dbNSFP["genename"]
    merged_SIFT4G_dbNSFP["Amino acid change"] = merged_SIFT4G_dbNSFP["Amino_acid_change"]+' '+merged_SIFT4G_dbNSFP["MutationTaster_AAE"]
    merged_SIFT4G_dbNSFP["Mutation type"] = merged_SIFT4G_dbNSFP["Mutation_type"]+' '+merged_SIFT4G_dbNSFP["Mutation_Type"]
    
    merged_SIFT4G_dbNSFP['dbSNP']=merged_SIFT4G_dbNSFP['dbSNP'].str.split(' ').apply(set).str.join(' ')
    merged_SIFT4G_dbNSFP['Amino acid position']=merged_SIFT4G_dbNSFP['Amino acid position'].str.split(' ').apply(set).str.join(' ')
    merged_SIFT4G_dbNSFP['Gene Name']=merged_SIFT4G_dbNSFP['Gene Name'].str.split(' ').apply(set).str.join(' ')
    merged_SIFT4G_dbNSFP['Amino acid change']=merged_SIFT4G_dbNSFP['Amino acid change'].str.split(' ').apply(set).str.join(' ')
    merged_SIFT4G_dbNSFP['Mutation type']=merged_SIFT4G_dbNSFP['Mutation type'].str.split(' ').apply(set).str.join(' ')

    merged_SIFT4G_dbNSFP.drop(['#CHROM','dbsnp','rs_dbSNP','Gene_Name','Mutation_Type','Amino_acid_change','Amino_acid_position','genename','Mutation_type','aapos',"MutationTaster_AAE"], axis =1, inplace=True)
    merged_SIFT4G_dbNSFP=merged_SIFT4G_dbNSFP[['Chromosome Number','Chromosome Position','Reference Nucleotide','Alternate Nucleotide','NGS QC filter','Depth','Allele Frequency','snp_pos','SIFT4G_prediction','dbSNP','Amino acid position','Gene Name','Amino acid change','dbNSFP Amino acid change','dbNSFP Amino acid change','REGION','Mutation type','HGVS nomenclature','PolyPhen 2 prediction','Clinical significance','Functional consequence','Minor allele frequency','OMIM disease','Comments']]
    #print(merged_SIFT4G_dbNSFP)
    
    merged_SIFT4G_dbNSFP['Gene Name'] = merged_SIFT4G_dbNSFP['Gene Name'].str.replace(' ', '')
    db_fill_gene=merged_SIFT4G_dbNSFP.loc[merged_SIFT4G_dbNSFP['Gene Name'] == '']
    dbnsfp_fill=pd.read_csv(db_path, sep='\t')
    db_fill_gene1=pd.DataFrame()
    db_fill_gene1['Chromosome Number']=list(dbnsfp_fill['#CHROM'])
    db_fill_gene1['Chromosome Position']=list(dbnsfp_fill['POS'])
    db_fill_gene1['Amino acid position']=list(dbnsfp_fill['aapos'])
    db_fill_gene1['Gene Name']=list(dbnsfp_fill['genename'])
    db_fill_gene1['dbSNP']=list(dbnsfp_fill['rs_dbSNP'])
    db_fill_gene1['SIFT4G_prediction']=list(dbnsfp_fill['SIFT4G_pred'])
    db_fill_gene1['Gene Name'] = db_fill_gene1['Gene Name'].str.split(';').str[0]
    db_fill_gene1['SIFT4G_prediction'].replace({'D':'DELETERIOUS'}, regex=True, inplace=True)
    #display(db_fill_gene,db_fill_gene1)
    db_fill_gene['Chromosome Number']=db_fill_gene['Chromosome Number'].astype(str)
    db_fill_gene1['Chromosome Number']=db_fill_gene1['Chromosome Number'].astype(str)
    db_fill_gene1['Chromosome Position']=db_fill_gene1['Chromosome Position'].astype(str)
    db_fil = pd.merge(db_fill_gene,db_fill_gene1, how='left', on=['Chromosome Number','Chromosome Position'])
    db_fil.drop(['SIFT4G_prediction_x', 'dbSNP_x', 'Amino acid position_x','Gene Name_x'], axis = 1, inplace=True)
    db_fil.rename(columns = {'Amino acid position_y':'Amino acid position', 'Gene Name_y':'Gene Name', 'dbSNP_y':'dbSNP','SIFT4G_prediction_y':'SIFT4G_prediction'}, inplace = True)
    db_fil=db_fil[['Chromosome Number', 'Chromosome Position', 'Reference Nucleotide',
           'Alternate Nucleotide', 'NGS QC filter', 'Depth', 'Allele Frequency',
           'snp_pos', 'SIFT4G_prediction','dbSNP','Amino acid position', 'Gene Name','Amino acid change',
           'dbNSFP Amino acid change', 'REGION', 'Mutation type',
           'HGVS nomenclature', 'PolyPhen 2 prediction', 'Clinical significance',
           'Functional consequence', 'Minor allele frequency', 'OMIM disease',
           'Comments']]
    #db_fil['REGION']='CDS'
    #db_fil['Mutation type']='NONSYNONYMOUS'
    db_fil.loc[db_fil['Gene Name'].isnull() == False, 'REGION'] = 'CDS'                       
    db_fil.loc[db_fil['Gene Name'].isnull() == False, 'Mutation type'] = 'NONSYNONYMOUS'  
    #print(db_fill_gene.shape[0],db_fil.shape[0])
    db_fil
    merged_SIFT4G_dbNSFP1=merged_SIFT4G_dbNSFP.copy(deep=True)
    merged_SIFT4G_dbNSFP1.drop(merged_SIFT4G_dbNSFP1.index[merged_SIFT4G_dbNSFP1['Gene Name'] == ''], inplace=True)
    #merged_SIFT4G_dbNSFP2=pd.merge(merged_SIFT4G_dbNSFP1,db_fil,how='left', on=['Chromosome Number','Chromosome Position','Reference Nucleotide','Alternate Nucleotide'])
    merged_SIFT4G_dbNSFP2=merged_SIFT4G_dbNSFP1.append(db_fil)
    merged_SIFT4G_dbNSFP2['Chromosome Position'] = merged_SIFT4G_dbNSFP2['Chromosome Position'].astype(int)
    x = merged_SIFT4G_dbNSFP2.loc[merged_SIFT4G_dbNSFP2['Chromosome Number'].str.contains('X', case=False, na=False)]
    merged_SIFT4G_dbNSFP2.drop(merged_SIFT4G_dbNSFP2.index[merged_SIFT4G_dbNSFP2['Chromosome Number'] == 'X'], inplace=True)
    merged_SIFT4G_dbNSFP2['Chromosome Number'] = merged_SIFT4G_dbNSFP2['Chromosome Number'].astype(int)
    merged_SIFT4G_dbNSFP2.sort_values(by=['Chromosome Number','Chromosome Position'], inplace=True)
    x.sort_values(by=['Chromosome Position'], inplace=True)
    merged_SIFT4G_dbNSFP3=merged_SIFT4G_dbNSFP2.append(x)
    #display(db_fill_gene,merged_SIFT4G_dbNSFP3)
    
    '''
    my_cols = [str(i) for i in range(16)] # create some col names
    dfr1 = pd.read_csv(r'/home/ubuntu/clinical_reporting/s3/input_files/Polyphen_short.txt', sep="\t", names=my_cols, engine="python")
    #print(dfr1)
    r1 = pd.DataFrame(dfr1)
    #print(r1)
    new_header = dfr1.iloc[0] #grab the first row for the header
    dfr1 = dfr1[1:] #take the data less the header row
    dfr1.columns = new_header #set the header row as the df header
    dfr1.rename(columns=dfr1.iloc[0])
    dfr1.columns = dfr1.columns.str.replace(' ', '')
    newr1 = dfr1[~dfr1['#o_acc'].astype(str).str.startswith('##')]
    newr1.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}])

    newr1[None]
    sp = newr1[None].str.split(' ').str[1]
    sp1 = sp.str.split('|').str[0]
    #print(newr1)
    #print(dfr1)
    #print(dfr1.columns)
    newr1.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}])
    shcopy1 = newr1.copy(deep=True)
    shcopy1.columns = shcopy1.columns.str.replace(' ', '')
    shcopy1 = shcopy1.drop(['#o_acc',      'o_pos',      'o_aa1',      'o_aa2',       'rsid',        'pos',        'aa1',        'aa2',
            'pph2_prob',   'pph2_FPR',   'pph2_TPR', None], axis = 1)
    shcopy1['acc'] = shcopy1['acc'].str.replace(' ', '')
    shcopy1['snp_pos'] = sp1
    shcopy1.drop_duplicates(subset ='snp_pos', keep = 'first', inplace = True)
    #shcopy1 = shcopy1.drop('snp_pos', axis = 1)
    #shcopy1.sort_index(axis = 0)
    #shcopy1.reset_index(inplace = True, drop = True)
    shcopy1 = shcopy1.drop(shcopy1.columns[[0,2,3]], axis=1)
    shcopy1
    df4n = pd.read_csv(r"/home/ubuntu/clinical_reporting/s3/input_files/Polyphen_snps.txt", sep="\t", skiprows=0)
    #df4n1 = df[~df['0'].astype(str).str.startswith('##')]
    df4n.columns = df4n.columns.str.replace(' ', '')
    shcopy = df4n.copy(deep=True)
    shcopy['type'] = shcopy['type'].str.replace(' ', '')
    shcopy.drop(shcopy.loc[shcopy['type']=='missense'].index, inplace=True)
    shcopy.rename(columns = {'spacc':'acc'}, inplace = True)
    shcopy = shcopy.drop(['str', 'gene', 'transcript', 'ccid', 'ccds', 'cciden',
           'refa', 'type', 'ntpos', 'nt1', 'nt2', 'flanks', 'trv', 'cpg', 'jxdon',
           'jxacc', 'exon', 'cexon', 'jxc', 'dgn', 'cdnpos', 'frame', 'cdn1',
           'cdn2', 'aa1', 'aa2', 'aapos', 'spmap', 'spname', 'refs_acc',
           'dbrsid', 'dbobsrvd', 'dbavHet', 'dbavHetSE', 'dbRmPaPt', 'comments'], axis = 1)
    shcopy.sort_index(axis = 0)
    shcopy.reset_index(inplace = True, drop = True)
    shcopy['#snp_pos'].str.strip()
    shcopy.rename(columns = {'#snp_pos':'snp_pos'}, inplace = True)
    shcopy.drop_duplicates(subset ='snp_pos', keep = 'first', inplace = True)
    shcopy = shcopy[~shcopy['snp_pos'].astype(str).str.startswith('##')]
    shcopy['snp_pos'] = shcopy['snp_pos'].str.replace(' ', '')
    shcopy['acc'] = shcopy['acc'].str.replace(' ', '')
    shcopy
    snpsh = pd.merge(shcopy, shcopy1, how='outer', on='snp_pos')
    snpsh
    snpsh.drop(snpsh.loc[snpsh['acc'] == '?'].index, inplace=True)
    snpsh = snpsh.dropna()
    snpsh.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}])
    snpsh
    '''
    merged_SIFT4G_dbNSFP3["snp_pos1"] = 'chr'+merged_SIFT4G_dbNSFP3["Chromosome Number"].astype(str) +':'+ merged_SIFT4G_dbNSFP3["Chromosome Position"].astype(str)
    merged_SIFT4G_dbNSFP3.snp_pos1 = merged_SIFT4G_dbNSFP3.snp_pos1.str.replace(' ', '')
    merged_SIFT4G_dbNSFP3["snp_pos1"]
    '''
    snpsh.rename(columns = {'snp_pos':'snp_pos1'}, inplace = True)
    sif = pd.merge(merged_SIFT4G_dbNSFP3, snpsh, how='outer', on='snp_pos1')
    sif = sif.drop('acc', axis = 1)
    sif = sif.drop('PolyPhen 2 prediction', axis = 1)
    sif.rename(columns = {'prediction':'PolyPhen2 prediction'}, inplace = True)
    sif
    '''
    Clinvarfile = EXTERNAL_FILES_FOLDER_PATH+'variant_summary1.txt'   
    vs = pd.read_csv(Clinvarfile, sep="\t", skiprows=0, low_memory=False)
    vs = vs.drop(['Unnamed: 0','Type', 'RS# (dbSNP)', 'Cytogenetic'], axis = 1)
    vs["snp_pos"] = vs["Chromosome"].astype(str) +','+ vs["PositionVCF"].astype(str)
    vs.snp_pos = vs.snp_pos.str.replace(' ', '')
    vs.rename(columns = {'GeneSymbol':'Gene Name'}, inplace = True)
    vs.rename(columns = {'ReferenceAlleleVCF':'Reference Nucleotide'}, inplace = True)
    vs.rename(columns = {'AlternateAlleleVCF':'Alternate Nucleotide'}, inplace = True)
    vs
    merged_SIFT4G_dbNSFP3['Gene Name'] = merged_SIFT4G_dbNSFP3['Gene Name'].str.replace(' ', '')
    news = pd.merge(merged_SIFT4G_dbNSFP3, vs, how='left', on=['snp_pos', 'Gene Name', 'Reference Nucleotide','Alternate Nucleotide'])
    news.rename(columns = {'PhenotypeIDS':'OMIM id from ClinVar'}, inplace = True)
    news.rename(columns = {'PhenotypeList':'OMIM Disease from ClinVar'}, inplace = True)
    news = news.drop(['HGVS nomenclature', 'Clinical significance', 'OMIM disease', 'Chromosome', 'PositionVCF',  'Assembly'], axis = 1)
    base = 'https://cancer.sanger.ac.uk/cosmic/gene/'
    url1 = 'analysis?ln='
    gene = news['Gene Name']
    url2 = '#variants'
    url = base+url1+gene+url2
    '''
    if gene.isnull:
        merged2['COSMIC_link'] == ''
    else:'''
    news['COSMIC_link'] = url

    news['COSMIC_link'] = news['COSMIC_link'].str.replace(' ', '')
    news = news[['Chromosome Number', 'Chromosome Position', 'Reference Nucleotide',
           'Alternate Nucleotide', 'NGS QC filter', 'Depth', 'Allele Frequency',
           'snp_pos', 'dbSNP','Gene Name', 'Amino acid position', 'Amino acid change','dbNSFP Amino acid change','REGION','Mutation type','SIFT4G_prediction', 'Name',
            'ClinicalSignificance', 'Functional consequence', 'Minor allele frequency', 'OMIM id from ClinVar', 'OMIM Disease from ClinVar', 'OtherIDs',
            'Comments','COSMIC_link', 'snp_pos1','VariationID']]
    news.rename(columns = {'Name':'HGVS Nomenclature'}, inplace = True)
    news.rename(columns = {'ClinicalSignificance':'Clinical Significance'}, inplace = True)
    #import numpy as np
    news['OMIM id from ClinVar'] = np.where(~news['OMIM id from ClinVar'].str.contains("OMIM", na=False), "", news['OMIM id from ClinVar'])
    news.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
    #import requests

    #import lxml.html
    #from bs4 import *

    #from lxml import etree
    #import re
    #from IPython.display import HTML
    #import numpy as np

    df11 = news.copy(deep=True)
    copy1=df11.copy(deep=True)
    df11.replace(np.NaN, '-1', inplace=True)
    df11.replace(to_replace =["nan", 'nannannan','NA'], 
                                value ="-1", inplace=True)
    df11['VariationID'] = df11['VariationID'].astype('int')
    df11.replace(to_replace =[str('-1')], 
                                value ="Nan", inplace=True)
    op=list(df11['VariationID'])
    #print(op)
    p = [str(x) for x in op]

    # Printing output 
    #print(output)

    url_links=[]
    for j in range(len(p)):
        if p[j]=='-1':
            url_links.append('NONE')
        else:
            url_links.append('https://www.ncbi.nlm.nih.gov/clinvar/variation/'+str(p[j])+'/')
    #print(url_links)
    rosses=[]
    ros_1ses=[]
    linkt=[]
    for lin in url_links:
        if lin !='NONE':
            #print(lin)
            r = requests.get(lin)
            r.encoding = 'utf-8'

            html_content = r.text
            soups = BeautifulSoup(html_content, 'lxml')
            fissr= soups.find('dl',class_="dl-leftalign").find('dt',string="Allele frequency")
            #print(fissr)
            if fissr is not None:
                l=''
                te=[]
                for q in fissr.next_siblings:

                    #print(q)
                    #l=l+q.text.strip()
                    #ys = b.replace('\n','')
                    #mns= y.replace('  ','')

                    te.append(q.text.strip())
                #print(te)
                rosses.append(te)
            else:
                rosses.append('NONE')
            fiiss= soups.find_all('div',class_="dd-group-border")

            if len(fiiss)==0:
                ros_1ses.append(' ')
            else:
                #j=[]
                b=''
                for shot in fiiss:
                    #b=shot.find_all("dd")
                    j=[]
                    for k in shot.find_all("dd"):
                        b=b+";"+k.text.strip()
                        y = b.replace('\n','')
                        mn= y.replace('  ','')
                        #print(b)
                        #j.append(k.text.strip())
                ros_1ses.append(mn.strip())
        elif lin=='NONE':
            rosses.append(' ')
            ros_1ses.append(' ')


    df11['Minor_allele_frequency'] = rosses

    res = ['  '.join(ele) for ele in rosses]

    df11['min_af']=res
    df11=df11[df11.min_af.str.contains('1000 Genomes Project')]


    hj=list(df11['min_af'])
    hhi=[]
    for i in hj:
        li = list(i.split("  "))
        hhi.append(li)
    #print(hhi)
    tos=[]

    for x in hhi:
        for y in x:
            if '1000 Genomes Project' in y:
                tos.append(y)
    #print(tos)
    l_replace = [s.replace('1000 Genomes Project', '') for s in tos]
    #print(l_replace)
    df11['final_minaf']=l_replace
    min_a_f=list(df11['final_minaf'])
    poss=list(df11['Chromosome Position'])

    #df2=pd.DataFrame(poss)
    df2=pd.DataFrame(poss,columns =['Chromosome Position']) 
    #display(df2,poss,df2.columns,min_a_f)
    df2.columns =['Chromosome Position']
    df2['Minor_allele_frequency']=min_a_f
    copy1['Functional_consequence']= ros_1ses
    merged_df_minaf = pd.merge(copy1,df2, how='left', on='Chromosome Position')
    merged_df_minaf=merged_df_minaf[['Chromosome Number','Chromosome Position','Reference Nucleotide','Alternate Nucleotide','NGS QC filter','Depth','Allele Frequency','snp_pos','dbSNP','Gene Name','Amino acid position','Amino acid change','dbNSFP Amino acid change','REGION','Mutation type','SIFT4G_prediction','HGVS Nomenclature','Clinical Significance','VariationID','Minor_allele_frequency','Functional_consequence','OMIM id from ClinVar','OMIM Disease from ClinVar','OtherIDs','Comments','COSMIC_link']]
    merged_df_minaf.rename(columns = {'SIFT4G_prediction':'SIFT prediction'}, inplace = True)
    
    #merged_df_minaf['VariationID'] = merged_df_minaf['VariationID'].astype(str)
    merged_df_minaf.loc[:,"VariationID"] = merged_df_minaf.loc[:,"VariationID"].astype(str).apply(lambda x: x.replace('.0',''))
    base = 'https://www.ncbi.nlm.nih.gov/clinvar/variation/'
    varId = merged_df_minaf['VariationID']
    url1 = '/?new_evidence=false'
    url = base+varId+url1
    merged_df_minaf['ClinVar_link'] = url
    merged_df_minaf['ClinVar_link'] = merged_df_minaf['ClinVar_link'].str.replace(' ', '')
    merged_df_minaf['ClinVar_link'] = np.where(merged_df_minaf['ClinVar_link'].str.contains("/nan/", na=False), "", merged_df_minaf['ClinVar_link'])
    #merged_df_minaf = merged_df_minaf.drop(['Comments'], axis = 1)
    
    merged_df_minaf = merged_df_minaf.drop(['VariationID','Comments','snp_pos'], axis = 1)
    merged_df_minaf = merged_df_minaf.loc[:, ~merged_df_minaf.columns.duplicated()]
    merged_df_minaf.replace(np.NaN, ' ', inplace=True)
    merged_df_minaf.replace(to_replace =["nan", 'nannannan','NA','Nan'], 
                                value =" ", inplace=True)
    merged_df_minaf.fillna('-', inplace=True)
    merged_df_minaf.drop_duplicates(subset ="Chromosome Position", inplace = True)
    #print("##############")
    #print('\n No of variants: ',merged_df_minaf.shape[0])
    #with pd.option_context('display.max_rows', None, 'display.max_columns', None,'display.max_colwidth', 20):  
        #HTML(merged_df_minaf.to_html(classes='table table-striped'))
    merged_df_minaf['dbNSFP Amino acid change'] = merged_df_minaf['dbNSFP Amino acid change'].astype(str)
    merged_df_minaf['dbNSFP Amino acid change']=merged_df_minaf['dbNSFP Amino acid change'].str.split(' ').apply(set).str.join(', ')
    merged_df_minaf=merged_df_minaf.replace(r'^\s*$', np.nan, regex=True)
    #namess=list(merged_df_minaf.columns)
    #print(namess)
    #merged_df_minaf['dbNSFP Amino acid change'] = merged_df_minaf['dbNSFP Amino acid change'].astype(str)
    #merged_df_minaf['dbNSFP Amino acid change']=merged_df_minaf['dbNSFP Amino acid change'].str.split(' ').apply(set).str.join(', ')
    merged_df_minaf=merged_df_minaf.dropna(subset=['dbSNP', 'Gene Name', 'Amino acid position', 'Amino acid change', 'dbNSFP Amino acid change', 'REGION', 'Mutation type', 'SIFT prediction', 'HGVS Nomenclature', 'Clinical Significance', 'Minor_allele_frequency', 'Functional_consequence', 'OMIM id from ClinVar', 'OMIM Disease from ClinVar', 'OtherIDs'], how='all')
    merged_df_minaf = merged_df_minaf.drop(['Functional_consequence'], axis = 1)
    merged_df_minaf.replace(to_replace =["nan", 'nannannan','NA','Nan'], 
                                value =" ", inplace=True)
    merged_df_minaf.fillna('-', inplace=True)
    #merged_df_minaf.drop(merged_df_minaf.index[merged_df_minaf['Gene Name'] == '-'], inplace=True)
    print('No. of entries that have relevant information:',len(list(merged_df_minaf['dbSNP'])))
    if ('%' in merged_df_minaf['Allele Frequency'][0]) == False:
        merged_df_minaf['Allele Frequency'] = (100 * merged_df_minaf['Allele Frequency'].astype(float)).astype(str)+'%'
    file_name = USER_INPUT_FOLDER_PATH+file+'_Display_Table.xlsx'
    merged_df_minaf.to_excel(file_name, index=False)
    def Filter_Button(b):
            clear_output(wait=True)
            #filter_variants(fileDropdown.value)
            filter_variants(file)

    buttonM = widgets.Button(description="Filter Variants")
    buttonM.on_click(Filter_Button)
    display(buttonM)
    mdf = merged_df_minaf.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index().set_sticky(axis="columns")
    #display(mdf)
    display(HTML(merged_df_minaf.to_html(render_links=True, escape=False, index=False)))
    #display(HTML(merged_df_minaf.to_html(render_links=True, escape=False,col_space=20,notebook=True,index=False)))
    return (merged_df_minaf.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index().set_sticky(axis="columns"))

# Filter the Display Table using widgets; IntSlider for Depth, Text Box for Allele Freq, SelectMultiple dropdown for Clinical Significance and SIFT prediction and set the paramters to default using reset_button

# Buttons for Targeted Therapy section

In [ ]:
class color:
   PURPLE = '\033[95m'
   CYAN = '\033[96m'
   DARKCYAN = '\033[36m'
   BLUE = '\033[94m'
   GREEN = '\033[92m'
   YELLOW = '\033[93m'
   RED = '\033[91m'
   BOLD = '\033[1m'
   UNDERLINE = '\033[4m'
   END = '\033[0m'
    
def filtered_variants():
                       
    #def filtered_variants2(file):
    #print(file)
    #news['AF'] = news['AF'].astype(str)
    news = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Display_Table.xlsx')
    news['Depth'] = news['Depth'].astype(int)
    news['Allele Frequency'] = news['Allele Frequency'].astype(str)
    #news = news.drop(['Unnamed: 0'], axis = 1)
    news.drop(news.index[news['NGS QC filter'] != 'PASS'], inplace=True)
    news['Allele Frequency']=news['Allele Frequency'].str.replace("%","")
    news['Allele Frequency'] = news['Allele Frequency'].astype(float)
    news['Clinical Significance']=news['Clinical Significance'].str.strip()
    cs=list(news['Clinical Significance'].unique())
    news['SIFT prediction']=news['SIFT prediction'].str.strip()
    sp=list(news['SIFT prediction'].unique())
    news['Mutation type']=news['Mutation type'].str.strip()
    mtl=list(news['Mutation type'].unique())
    news['REGION']=news['REGION'].str.strip()
    reg=list(news['REGION'].unique())
    #news['Chromosome Number'] = news['Chromosome Number'].astype(int)
    #fil['Mutation Type'] = fil['Mutation Type'].str.replace(' ', '')

    '''
    a = widgets.IntSlider(
        value=100,
        min=0,
        max=news['Depth'].max(),
        step=1,
        description='Depth',
        disabled=False,
        continuous_update=False,
        orientation='horizontal',
        readout=True,
        readout_format='d'
    )
    '''
    a = widgets.BoundedIntText(
            value=100,
            min=0,
            max=news['Depth'].max(),
            step=1,
            description='Depth',
            disabled=False)
    '''
    b = widgets.Text(
        value='1',
        min=0,
        max=10,
        step=1,
        description='AF',
        disabled=False)
    '''
    b = widgets.BoundedFloatText(
            value=1,
            min=0,
            #max=10.0,
            step=0.1,
            description='AF',
            disabled=False)

    c = widgets.SelectMultiple(
            options= cs,
            value= cs,
            #rows=10,
            description='Clinical Significance',
            disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'}
        )
    d = widgets.SelectMultiple(
            options= sp,
            value= sp,
            #rows=10,
            description='SIFT prediction',
            disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'}
        )
    e = widgets.SelectMultiple(
        options= mtl, 
        value= mtl,
        #rows=10,
        description='Mutation type:',
        disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    f= widgets.SelectMultiple(
        options= reg, 
        value= reg,
        #rows=10,
        description='REGION:',
        disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    
    items = [a,b,c,d]
    left_box = VBox([items[0], items[1]])
    right_box = VBox([items[2], items[3]])
    box_layout = Layout(display='flex',
        flex_flow='row',
        align_items='stretch',justify_content ='space-around', 
        #border='solid'
        width='100%')
        #a = HBox([left_box,right_box], layout=box_layout)
    ui = HBox([VBox([items[0], items[2],e]), VBox([items[1], items[3],f])], layout=box_layout)
    #ui = widgets.HBox([a, b, c,d])

    #clear_output(wait=True)


    
    print(color.BOLD+'\nList of variants with clinical significance/potential clinical significance' + color.END, 
      '\n\nG-Knowme Default Somatic: All the annotated variants are further filtered for AF (allele frequency) > 1%, Depth >100, (AF>5% & Depth >500), Clinical significance: All the terms in display table except variants with "Bengin", "Likely Bengin", "Bengin/Likely Bengin" and SIFT predictions as "All", Variants with (Clinical significance "Uncertain Significance" and SIFT prediction as "Tolerated") and Mutation type "SYNONYMOUS" are elminated') 
    def on_button_clicked_somatic(_):
        df = news.drop(news[(news['Allele Frequency'] < 1)].index)
        #print('\n',df.shape[0],  'entries are selected after applying the filters')
        #display(df)
        df1 = df.drop(df[(df['Depth'] < 100 )].index)
        #print('\n',df1.shape[0],  'entries are selected after applying the filters')
        #display(df1)
        df2= df1.drop(df1[(df1['Depth'] < 500 ) & (df1['Allele Frequency'] < 5)].index)
        #print('\n',df2.shape[0],  'entries are selected after applying the filters')
        #display(df2)
        dff=(df2[~df2['Clinical Significance'].isin(['Benign', 'Likely benign', 'Benign/Likely benign'])])
        #print('\n',dff.shape[0],  'entries are selected after applying the filters')
        #display(dff)
        dff1= dff.drop(dff[(dff['Clinical Significance'] == 'Uncertain significance' ) & (dff['SIFT prediction'] == 'TOLERATED')].index)
        df_new = dff1.drop(dff1[(dff1['Mutation type'] == 'SYNONYMOUS' )].index)
        print( 'After applying Set G-KnowMe Default-Somatic')
        #sheet_new = from_dataframe(df_new)
        #sheet_new.row_headers=False
        #display(sheet_new)
        def reset_button(defaults={}):
            def on_button_clicked(_):
                for k, v in defaults.items(): 
                    k.value = v
            button = widgets.Button(description='Reset',layout={'width': 'max-content'})
            button.on_click(on_button_clicked)
            display(button)
    
        reset_button(defaults={c:c.options, d: d.options, e: e.options, f: f.options})
        
        def show_df(Depth, AF, Clinical_Sig, SIFT_Pred, mut, reg):

            df = df_new
            df.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
            #print(Clinical_Sig)
            df1 = df[(df['Depth'] > Depth) & (df['Allele Frequency'] > AF)]
            #display(df1)
            #print('df:',df.shape[0])
            #print('df1:' , df1.shape[0])
            dfe = pd.DataFrame(columns=['Clinical Significance'])
            l=[]
            for sig in Clinical_Sig:
                #print(sig)
                if sig == 'All':
                    l=df['Clinical Significance'].unique()
                    break
                else:
                    l.append(sig)

            #print(l)

            l1=[]
            for sif in SIFT_Pred:
                #print(sif)
                if sif == 'All':
                    l1=df['SIFT prediction'].unique()
                    break
                else:
                    l1.append(sif)
                #df1 = (df1[(df1['Clinical Significance']==sig)])
            #print(l1)
            l2=[]
            for mt in mut:
                #print(mt)
                if mt == 'All':
                    l2=df['Mutation type'].unique()
                    break
                else:
                    l2.append(mt)
            #print(l2)
            
            l3=[]
            for re in reg:
                #print(re)
                if mt == 'All':
                    l3=df['REGION'].unique()
                    break
                else:
                    l3.append(re)
            #print(l3)

            #df2=((df1[df1['Clinical Significance'].isin(l)]) | (df1[df1['SIFT4G_prediction'].isin(l1)]))
            dff=(df1[df1['Clinical Significance'].isin(l)])
            #print(dff.shape[0])
            #display(dff)
            dff1=(dff[dff['SIFT prediction'].isin(l1)])
            dff2=(dff1[dff1['Mutation type'].isin(l2)])
            dff3=(dff2[dff2['REGION'].isin(l3)])

            #df2=pd.merge(dff, dff1, how="outer")
            pd.set_option('display.max_columns', 500)
            print('\n',dff3.shape[0],  'entries are selected after applying the filters')
            dff3['Chromosome Position'] = dff3['Chromosome Position'].astype(int)
            dff3['Chromosome Number'] = dff3['Chromosome Number'].astype(str)
            x = dff3.loc[dff3['Chromosome Number'].str.contains('X', case=False, na=False)]
            dff3.drop(dff3.index[dff3['Chromosome Number'] == 'X'], inplace=True)
            dff3.loc[:,"Chromosome Number"] = dff3.loc[:,"Chromosome Number"].astype(str).apply(lambda x: x.replace('.0',''))
            dff3['Chromosome Number'] = dff3['Chromosome Number'].astype(int)
            dff3.sort_values(by=['Chromosome Number','Chromosome Position'], inplace=True)
            x.sort_values(by=['Chromosome Position'], inplace=True)
            df3=dff3.append(x)
            #df2.sort_values(by=['Chromosome Number'])
            #df2.sort_values(by=['Chromosome Number'], inplace=True)
            df3['Allele Frequency'] = df3['Allele Frequency'].astype(str)+'%'
            #display(df3.style.hide_index().set_sticky(axis="columns"))
            display(HTML(df3.to_html(render_links=True, escape=False, index=False)))
            #display(HTML(df3.to_html(render_links=True, escape=False,col_space=20,notebook=True,index=False)))
            #return df3.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
            #print('---saving filtered xlsx',len(df3.index))
            file_name = USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx'
            df3.to_excel(file_name, index=False)

        w=interactive_output(show_df, {'Depth':a, 'AF':b, 'Clinical_Sig':c, 'SIFT_Pred':d, 'mut':e, 'reg':f})
        display(ui, w)

        title='\nClick on the link below to download the table'
        display(Markdown('<strong><br/>{}'.format(title)))
        #download_file(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
        if (path.exists(file+'_Filtered.xlsx'))==True:
            display(FileLink(file+'_Filtered.xlsx'))
            #url=FileLink(file+'_Filtered.xlsx')
        else:
            os.symlink(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx',file+'_Filtered.xlsx')
            display(FileLink(file+'_Filtered.xlsx'))

        
    button = widgets.Button(description='Set G-KnowMe Default-Somatic',layout={'width': 'max-content'})
    button.on_click(on_button_clicked_somatic)
    display(button)

  
    print(color.BOLD+'\nList of variants with clinical significance/potential clinical significance' + color.END,'\n\n G-Knowme Default Germline: All the annotated variants are further filtered for AF (allele frequency) >20%, Depth >100, Clinical significance: All the terms in display table except variants with "Bengin", "Likely Bengin", "Bengin/Likely Bengin" and SIFT predictions as "All", Variants with (Clinical significance "Uncertain Significance" and SIFT prediction as "Tolerated") and Mutation type "SYNONYMOUS" are elminated')
    def on_button_clicked_germline(_):
        df = news.drop(news[(news['Allele Frequency'] < 20)].index)
        #print('\n',df.shape[0],  'entries are selected after applying the filters')
        #display(df)
        df1 = df.drop(df[(df['Depth'] < 100 )].index)
        #print('\n',df1.shape[0],  'entries are selected after applying the filters')
        #display(df1)
        dff=(df1[~df1['Clinical Significance'].isin(['Benign', 'Likely benign', 'Benign/Likely benign'])])
        #print('\n',dff.shape[0],  'entries are selected after applying the filters')
        #display(dff)
        dff1= dff.drop(dff[(dff['Clinical Significance'] == 'Uncertain significance' ) & (dff['SIFT prediction'] == 'TOLERATED')].index)
        df_new = dff1.drop(dff1[(dff1['Mutation type'] == 'SYNONYMOUS' )].index)
        print('After applying Set G-KnowMe Default- Germline')
        #sheet_news = from_dataframe(df_news)
        #sheet_new.row_headers=False
        #display(sheet_news)
        def reset_button(defaults={}):
            def on_button_clicked(_):
                for k, v in defaults.items(): 
                    k.value = v
            button = widgets.Button(description='Reset',layout={'width': 'max-content'})
            button.on_click(on_button_clicked)
            display(button)
    
        reset_button(defaults={c:c.options, d: d.options, e: e.options, f: f.options})
        def show_df(Depth, AF, Clinical_Sig, SIFT_Pred, mut, reg):

            df = df_new
            df.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
            #print(Clinical_Sig)
            df1 = df[(df['Depth'] > Depth) & (df['Allele Frequency'] > AF)]
            #display(df1)
            #print('df:',df.shape[0])
            #print('df1:' , df1.shape[0])
            dfe = pd.DataFrame(columns=['Clinical Significance'])
            l=[]
            for sig in Clinical_Sig:
                #print(sig)
                if sig == 'All':
                    l=df['Clinical Significance'].unique()
                    break
                else:
                    l.append(sig)

            #print(l)

            l1=[]
            for sif in SIFT_Pred:
                #print(sif)
                if sif == 'All':
                    l1=df['SIFT prediction'].unique()
                    break
                else:
                    l1.append(sif)
                #df1 = (df1[(df1['Clinical Significance']==sig)])
            #print(l1)
            l2=[]
            for mt in mut:
                #print(mt)
                if mt == 'All':
                    l2=df['Mutation type'].unique()
                    break
                else:
                    l2.append(mt)
            #print(l2)
            
            l3=[]
            for re in reg:
                #print(re)
                if mt == 'All':
                    l3=df['REGION'].unique()
                    break
                else:
                    l3.append(re)
            #print(l3)

            #df2=((df1[df1['Clinical Significance'].isin(l)]) | (df1[df1['SIFT4G_prediction'].isin(l1)]))
            dff=(df1[df1['Clinical Significance'].isin(l)])
            #print(dff.shape[0])
            #display(dff)
            dff1=(dff[dff['SIFT prediction'].isin(l1)])
            dff2=(dff1[dff1['Mutation type'].isin(l2)])
            dff3=(dff2[dff2['REGION'].isin(l3)])

            #df2=pd.merge(dff, dff1, how="outer")
            pd.set_option('display.max_columns', 500)
            print('\n',dff3.shape[0],  'entries are selected after applying the filters')
            dff3['Chromosome Position'] = dff3['Chromosome Position'].astype(int)
            dff3['Chromosome Number'] = dff3['Chromosome Number'].astype(str)
            x = dff3.loc[dff3['Chromosome Number'].str.contains('X', case=False, na=False)]
            dff3.drop(dff3.index[dff3['Chromosome Number'] == 'X'], inplace=True)
            dff3.loc[:,"Chromosome Number"] = dff3.loc[:,"Chromosome Number"].astype(str).apply(lambda x: x.replace('.0',''))
            dff3['Chromosome Number'] = dff3['Chromosome Number'].astype(int)
            dff3.sort_values(by=['Chromosome Number','Chromosome Position'], inplace=True)
            x.sort_values(by=['Chromosome Position'], inplace=True)
            df3=dff3.append(x)
            #df2.sort_values(by=['Chromosome Number'])
            #df2.sort_values(by=['Chromosome Number'], inplace=True)
            df3['Allele Frequency'] = df3['Allele Frequency'].astype(str)+'%'
            #display(df3.style.hide_index().set_sticky(axis="columns"))
            display(HTML(df3.to_html(render_links=True, escape=False, index=False)))
            #display(HTML(df3.to_html(render_links=True, escape=False,col_space=20,notebook=True,index=False)))
            #return df3.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
            #print('---saving filtered xlsx',len(df3.index))
            file_name = USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx'
            df3.to_excel(file_name, index=False)

        w=interactive_output(show_df, {'Depth':a, 'AF':b, 'Clinical_Sig':c, 'SIFT_Pred':d, 'mut':e, 'reg':f})
        display(ui, w)
        title='\nClick on the link below to download the table'
        display(Markdown('<strong><br/>{}'.format(title)))
        #download_file(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
        if (path.exists(file+'_Filtered.xlsx'))==True:
            #print('inside if',file)
            display(FileLink(file+'_Filtered.xlsx'))
            #url=FileLink(file+'_Filtered.xlsx')
        else:
            os.symlink(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx',file+'_Filtered.xlsx')
            #print('inside else')
            display(FileLink(file+'_Filtered.xlsx'))

    button = widgets.Button(description='Set G-KnowMe Default- Germline',layout={'width': 'max-content'})
    button.on_click(on_button_clicked_germline)
    display(button)

    '''
    def show_df(Depth, AF, Clinical_Sig, SIFT_Pred):

        df = news
        df.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
        #print(Clinical_Sig)
        df1 = df[(df['Depth'] > Depth) & (df['Allele Frequency'] > AF)]
        #display(df1)
        #print('df:',df.shape[0])
        #print('df1:' , df1.shape[0])
        dfe = pd.DataFrame(columns=['Clinical Significance'])
        l=[]
        for sig in Clinical_Sig:
            #print(sig)
            if sig == 'All':
                l=df['Clinical Significance'].unique()
                break
            else:
                l.append(sig)
           
        #print(l)

        l1=[]
        for sif in SIFT_Pred:
            #print(sif)
            if sif == 'All':
                l1=df['SIFT prediction'].unique()
                break
            else:
                l1.append(sif)
            #df1 = (df1[(df1['Clinical Significance']==sig)])
        #print(l)

        #df2=((df1[df1['Clinical Significance'].isin(l)]) | (df1[df1['SIFT4G_prediction'].isin(l1)]))
        dff=(df1[df1['Clinical Significance'].isin(l)])
        #print(dff.shape[0])
        #display(dff)
        dff1=(dff[dff['SIFT prediction'].isin(l1)]) 
    
        #df2=pd.merge(dff, dff1, how="outer")
        pd.set_option('display.max_columns', 500)
        print('\n',dff1.shape[0],  'entries are selected after applying the filters')
        dff1['Chromosome Position'] = dff1['Chromosome Position'].astype(int)
        dff1['Chromosome Number'] = dff1['Chromosome Number'].astype(str)
        x = dff1.loc[dff1['Chromosome Number'].str.contains('X', case=False, na=False)]
        dff1.drop(dff1.index[dff1['Chromosome Number'] == 'X'], inplace=True)
        dff1.loc[:,"Chromosome Number"] = dff1.loc[:,"Chromosome Number"].astype(str).apply(lambda x: x.replace('.0',''))
        dff1['Chromosome Number'] = dff1['Chromosome Number'].astype(int)
        dff1.sort_values(by=['Chromosome Number','Chromosome Position'], inplace=True)
        x.sort_values(by=['Chromosome Position'], inplace=True)
        df3=dff1.append(x)
        #df2.sort_values(by=['Chromosome Number'])
        #df2.sort_values(by=['Chromosome Number'], inplace=True)
        df3['Allele Frequency'] = df3['Allele Frequency'].astype(str)+'%'
        #display(df3.style.hide_index().set_sticky(axis="columns"))
        display(HTML(df3.to_html(render_links=True, escape=False, index=False)))
        #display(HTML(df3.to_html(render_links=True, escape=False,col_space=20,notebook=True,index=False)))
        #return df3.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
        file_name = USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx'
        df3.to_excel(file_name, index=False)
    #w=interactive(show_df, Depth=a, AF=b, Clinical_Sig=c, SIFT_Pred=d)
    
    w=interactive_output(show_df, {'Depth':a, 'AF':b, 'Clinical_Sig':c, 'SIFT_Pred':d})
    display(ui, w)
    
    title='\nClick on the link below to download the table'
    display(Markdown('<strong><br/>{}'.format(title)))
    #download_file(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
    if (path.exists(file+'_Filtered.xlsx'))==True:
        display(FileLink(file+'_Filtered.xlsx'))
        #url=FileLink(file+'_Filtered.xlsx')
    else:
        os.symlink(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx',file+'_Filtered.xlsx')
        display(FileLink(file+'_Filtered.xlsx'))
    '''

In [ ]:
def Matched_Variants_targeted_therapy1():
  
    def targeted_therapy_Button(b):

        Matched_Variants_targeted_therapy()

        def targeted_therapy(b):
            #clear_output(wait=True)
            identification_of_variants_for_targeted_therapy()
            def DecisionTable_targeted_therapy(b):

                decisiontable()

            buttonM = widgets.Button(description="Decision Table", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
            buttonM.on_click(DecisionTable_targeted_therapy)
            display(buttonM)


        buttonM = widgets.Button(description="View Information for matched Gene/Variants", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
        buttonM.on_click(targeted_therapy)
        display(buttonM)

    clear_output(wait=True)    
    buttonM = widgets.Button(description="View Matched Variants", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    buttonM.on_click(targeted_therapy_Button)
    display(buttonM)
    


# Returns matches for variants from Display Table and "FDA-approved targeted therapies" list

In [ ]:
def Matched_Variants_targeted_therapy():
    #print(file)
    #file = str(file[0][0]).strip()
    flt = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
    #flt = flt.drop(['Unnamed: 0'], axis = 1)
    fda = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'FDA-approved targeted therapies.xlsx', sheet_name = 'FDA-approved targeted updated')
    
    #commented this line as there is no extra column in sheet
    #fda = fda.drop(['Unnamed: 0'], axis = 1)
    
    #Stripping Extra spaceses from column values
    fda_lt = fda['Gene'].str.strip()
    flt_lt = flt['Gene Name'].str.strip()
    genelt = flt[flt['Gene Name'].isin(fda_lt)]
    #genelt.drop(['Unnamed: 0'], axis = 1)
    genelt.fillna('-', inplace=True)
    #pd.set_option('display.max_colwidth', 0)
    print('\n',genelt.shape[0],  'matches are found for possible targeted therapy')
    blankIndex=[''] * len(genelt)
    genelt.index=blankIndex
    #pd.set_option('display.max_columns', 500)
    #pd.set_option('display.width', 1000)
    #pd.set_option('display.max_columns', None)
    #pd.set_option('display.expand_frame_repr', False)
    #pd.set_option('max_colwidth', -1)
    #pd.set_option('display.large_repr', 'truncate')
    #pd.set_option('display.height', 10)
    #pd.reset_option("^display")
    display(HTML(genelt.to_html(render_links=True, escape=False, index=False)))
    
    return 

# Functions to export files in excel (.xlsx) and word doc (.docx) format

In [ ]:
def export_excel(df_list, sheets, file_name, spaces, title):
    
    writer = pd.ExcelWriter(file_name,engine='xlsxwriter')   
    row = 0
    for dataframe,t in zip(df_list,title):
        dataframe.to_excel(writer,sheet_name=sheets,startrow=row+1 , startcol=0, index=False)   
        worksheet = writer.sheets[sheets]
        worksheet.write_string(row, 0, t)
        row = row + len(dataframe.index) + spaces + 1

    writer.save()

def export_doc(df, tablename, doc_file):
            
    # open an existing document
    doc = docx.Document(doc_file)
    current_section = doc.sections[-1]
    #print(current_section.orientation)
    if current_section.orientation==0:
        new_width, new_height = current_section.page_height, current_section.page_width
        #new_section = doc.add_section(WD_SECTION.NEW_PAGE)
        new_section = doc.sections[-1]
        # Changing the orientation to landscape
        new_section.orientation = WD_ORIENT.LANDSCAPE
        new_section.page_width = new_width
        new_section.page_height = new_height
        new_section.top_margin = Inches(0.3)
        new_section.bottom_margin = Inches(0.3)
        new_section.left_margin = Inches(0.3)
        new_section.right_margin = Inches(0.3)
    else:
        new_section=current_section

    doc.add_heading(tablename, 0)
    t = doc.add_table(df.shape[0]+1, df.shape[1])
    t.style = 'Table Grid'
    # add the header rows.
    for j in range(df.shape[-1]):
        t.cell(0,j).text = df.columns[j]

    # add the rest of the data frame
    for i in range(df.shape[0]):
        for j in range(df.shape[-1]):
            t.cell(i+1,j).text = str(df.values[i,j])

    #t.columns[0].width = Cm(3.5)
    #t.columns[1].width = Cm(7.5)
    #t.columns[2].width = Cm(5.5)

    for row in t.rows:
        for cell in row.cells:
            paragraphs = cell.paragraphs
            for paragraph in paragraphs:
                for run in paragraph.runs:
                    font = run.font
                    font.size= Pt(9)

    # save the doc
    doc.save(doc_file)
            

# Identification of potential variants for FDA –approved Targeted therapy: Tables created for variants matched from filtered display table to the G-KnowMe curated database of FDA approved list, variant list and Gene description (dropdown to select gene to be viewed) and option to export all tables in excel and doc format using respective buttons

In [ ]:
dfslist = ['genelt', 'fdaflt', 'desg', 'varflt']
for x in dfslist:
    x=pd.DataFrame()
    
def identification_of_variants_for_targeted_therapy():
    global genelt, fdaflt, nccng, desg, varflt
    #file = str(file[0][0]).strip()
    #print(file_path+file+'_Filtered.xlsx')
    flt = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
    #flt = flt.drop(['Unnamed: 0'], axis = 1)
    fda = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'FDA-approved targeted therapies.xlsx', sheet_name = 'FDA-approved targeted updated')
    
    #commented this line as there is no extra column in sheet
    #fda = fda.drop(['Unnamed: 0'], axis = 1)
    
    #Stripping Extra spaceses from column values
    fda['Gene'] = fda['Gene'].str.replace('\n','')
    fda_lt = fda['Gene'].str.strip()
    flt_lt = flt['Gene Name'].str.strip()
    genelt = flt[flt['Gene Name'].isin(fda_lt)]
    fdaflt = fda[fda['Gene'].isin(flt_lt)]
    #display(fdaflt)
    nccn=pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'NCCN guidelines for targeted therapy.xlsx')
    nccn.columns = nccn.columns.str.strip()
    nccng = nccn[nccn['Gene name'].isin(genelt['Gene Name'])]
    #display(nccng)
    vard = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'Variant description.xlsx')
    vard.columns = vard.columns.str.strip()
    vard = vard.drop(['Variant Description                                                              Source: CKB core',
           'Source: Uniprot\nhttps://www.uniprot.org/uniprot/P42336',
           'Source: SIFT/Polyphen 2 predictions'], axis = 1)
    #new_header = vard.iloc[0] #grab the first row for the header
    #vard = vard[1:] #take the data less the header row
    #vard.columns = new_header #set the header row as the df header
    vard['Gene name'] = vard['Gene name'].str.replace(' ', '')
    vard['Variant\n(Amino acid change)'] = vard['Variant\n(Amino acid change)'].str.replace(' ', '')
    flt['Amino acid change'] = flt['Amino acid change'].str.replace(' ', '')
    aachn = flt['dbNSFP Amino acid change'].str.strip()
    l=list(aachn)
    t=[]
    for i in l:
        sub=i.split(' ')
        #print(sub)
        for k in sub:
                sub1=k.replace(',', '')
                t.append(sub1)

    varflt = vard[vard['Gene name'].isin(flt_lt) & vard['Variant\n(Amino acid change)'].isin(t)]
    varflt.reset_index(inplace = True, drop = True)
    #display(varflt)
    GeneDes = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'Gene description.xlsx')
    GeneDes.columns = GeneDes.columns.str.strip()
    GeneDes = GeneDes.drop(['Gene Description                                                                                                                                                                                        [Source : CKB core  https://ckb.jax.org/gene/grid]',
           'Description                                                                                                                                                                                                                Source: My Cancer Genome\nhttps://www.mycancergenome.org/content/biomarkers/',
           'Description                                                                                                                                                                                                                Source: NCBI                                                                 https://www.ncbi.nlm.nih.gov/gene'], axis = 1)
    desg = GeneDes[GeneDes['Gene name'].isin(genelt['Gene Name'])]
    #display(genelt, fdaflt, desg, varflt)
    
    flt = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
    #flt = flt.drop(['Unnamed: 0'], axis = 1)
    fda = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'FDA-approved targeted therapies.xlsx', sheet_name = 'FDA-approved targeted updated')
    #fda = fda.drop(['Unnamed: 0'], axis = 1)
    fda_lt = fda['Gene'].str.strip()
    flt_lt = flt['Gene Name'].str.strip()
    genelt = flt[flt['Gene Name'].isin(fda_lt)]
    dt = pd.DataFrame(columns = ['Gene Name', 'Variant effect (reported or probable)', 'Targeted therapy', 'confirm the tier', 'Want to include Variant in the report?', 'Need Clinical trial?'])
    Decision_Table = genelt.merge(dt, how='left')
    Decision_Table = Decision_Table.drop(['NGS QC filter', 'Depth', 'dbSNP', 'Amino acid position','REGION','Mutation type', 'SIFT prediction', 'Clinical Significance', 'Minor_allele_frequency',
       'OMIM id from ClinVar', 'OMIM Disease from ClinVar', 'OtherIDs','COSMIC_link','ClinVar_link'], axis = 1)
    Decision_Table.fillna('', inplace=True)
    
    def show_df(Matched_Variants):
        #print('#####',Matched_Variants,'######')
        #print(genelt['Gene Name'])
        disp_d = (genelt[genelt['Gene Name']==Matched_Variants])
        disp_d.fillna('-', inplace=True)
        blankIndex=[''] * len(disp_d)
        disp_d.index=blankIndex
        #print(disp_d)
        FDA_df = (fdaflt[fdaflt['Gene']==Matched_Variants])
        FDA_df.fillna('-', inplace=True)
        #display(FDA_df)
        Nccng = (nccn[nccn['Gene name']==Matched_Variants])
        Nccng.fillna('-', inplace=True)
        Desc = (GeneDes[GeneDes['Gene name']==Matched_Variants])
        Desc.fillna('-', inplace=True)
        var_list = (varflt[varflt['Gene name']==Matched_Variants])
        var_list.fillna('-', inplace=True)
        def varl(Matched_Variants):

            if (varflt[varflt['Gene name']==Matched_Variants]).empty:
                print('\033[1m' +'No matches found')

            else:
                display(var_list.style.hide_index())
    
        
        #pd.reset_option("^display")
        pd.set_option('display.max_columns', 500)
        #l = disp_d1, FDA_df1, Desc1, var_list1
        disp_d1 = disp_d.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
        FDA_df1 = FDA_df.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
        Nccng1 = Nccng.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
        Desc1 = Desc.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
        var_list1 = var_list.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
        def dist(Matched_Variants):

            if disp_d.empty:
                print('\n','\033[1m' +'No matches found','\n')

            else:
                display(HTML(disp_d.to_html(render_links=True, escape=False, index=False)))
        
        def fdm(Matched_Variants):

            if FDA_df.empty:
                print('\n','\033[1m' +'No matches found','\n')

            else:
                display(FDA_df1)
        def ncg(Matched_Variants):
            if Nccng.empty:
                print('\n','\033[1m' +'No matches found','\n')

            else:
                display(Nccng1)
                
        def gdes(Matched_Variants):

            if Desc.empty:
                print('\n','\033[1m' +'No matches found','\n')

            else:
                display(Desc1)
        
        return (print('\n              Information is displayed for selected ',Matched_Variants,'Variant\n\nVariant information from Display table'), dist(Matched_Variants), 
                print('Information about FDA-approved targeted therapy'), fdm(Matched_Variants), 
                print('Information about NCCN guidelines'), ncg(Matched_Variants), 
                print('Gene Description'), gdes(Matched_Variants), print('Matched Variants in Variant List''\n'), varl(Matched_Variants))

    interact(show_df, Matched_Variants=widgets.Dropdown(options=genelt['Gene Name'].unique(),description='Matched Variants:',style= {'description_width': 'initial'},layout={'width': 'max-content'}))
    

# Decision Table for FDA –approved Targeted therapy, automatic export of all Targeted therapy tables and links to download them in excel and word doc format

In [ ]:
tdt=pd.DataFrame()
DT_target_therapy=pd.DataFrame()
def decisiontable():
    global tdt,DT_target_therapy   
    flt = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
    #flt = flt.drop(['Unnamed: 0'], axis = 1)
    fda = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'FDA-approved targeted therapies.xlsx', sheet_name = 'FDA-approved targeted updated')
    #fda = fda.drop(['Unnamed: 0'], axis = 1)
    fda_lt = fda['Gene'].str.strip()
    flt_lt = flt['Gene Name'].str.strip()
    genelt = flt[flt['Gene Name'].isin(fda_lt)]
    dt = pd.DataFrame(columns = ['Gene Name', 'Variant effect (reported or probable)', 'Targeted therapies Tier I','Targeted therapies Tier II', 'Want to include Variant in the report?', 'Need Clinical trial?'])
    Decision_Table = genelt.merge(dt, how='left')
    Decision_Table = Decision_Table.drop(['NGS QC filter', 'Depth', 'dbSNP', 'Amino acid position','REGION','Mutation type', 'SIFT prediction', 'Clinical Significance', 'Minor_allele_frequency',
       'OMIM id from ClinVar', 'OMIM Disease from ClinVar', 'OtherIDs','COSMIC_link','ClinVar_link'], axis = 1)
    Decision_Table.fillna('', inplace=True)
    #Decision_Table['Allele Frequency'] = (100 * Decision_Table['Allele Frequency'].astype(float)).astype(str)+'%'
    
    df10=Decision_Table
    #df10=dfin
    df10[['Chromosome Number','Chromosome Position']] = df10[['Chromosome Number','Chromosome Position']].astype(str)
    sheet1 = ipysheet.sheet(columns=14,  rows = df10.shape[0],  
                       column_headers = ['Chromosome Number', 'Chromosome Position', 'Reference Nucleotide',
           'Alternate Nucleotide', 'Allele Frequency', 'Gene Name',
           'Amino acid change', 'dbNSFP Amino acid change', 'HGVS Nomenclature',
           'Variant effect (reported or probable)', 'Targeted therapies Tier I','Targeted therapies Tier II', 'Want to include Variant in the report?', 'Need Clinical trial?'], row_headers=False)
    sheet1.row_headers=False
    

    for i in range(df10.shape[0]):
        cell(i, 0, df10['Chromosome Number'].iloc[i])
        cell(i, 1, df10['Chromosome Position'].iloc[i])
        cell(i, 2, df10['Reference Nucleotide'].iloc[i])
        cell(i, 3, df10['Alternate Nucleotide'].iloc[i])
        cell(i, 4, df10['Allele Frequency'].iloc[i])
        cell(i, 5, df10['Gene Name'].iloc[i])
        cell(i, 6, df10['Amino acid change'].iloc[i])
        cell(i, 7, df10['dbNSFP Amino acid change'].iloc[i])
        cell(i, 8, df10['HGVS Nomenclature'].iloc[i])
        cell(i, 9, df10['Variant effect (reported or probable)'].iloc[i])
        cell(i, 10, df10['Targeted therapies Tier I'].iloc[i])
        cell(i, 11, df10['Targeted therapies Tier II'].iloc[i])
        #cell(i, 12, df10['Want to include Variant in the report?'].iloc[i], background_color = b.value)
        cell(i, 12, False)
        #cell(i, 13, df10['Need Clinical trial?'].iloc[i])
        cell(i, 13, False)
    
    display(current())
        
    button = widgets.Button(description="Save Decision Table")
    output2 = widgets.Output()
    print('\033[1m'+'Note: Please Save your final Decision Table. Only saved Decision Table will be exported')
    display(button, output2)

    def on_button_clicked(b):
        with output2:
            global tdt,DT_target_therapy   
            tdt=ipysheet.to_dataframe(current())
            DT_target_therapy=tdt.copy(deep=True)
            #display(DT_target_therapy)
            tdt.drop(tdt.loc[tdt['Want to include Variant in the report?']==False].index, inplace=True)
            #display(tdt)
            #return tdt
            title=["Matches found for possible targeted therapy", "Information about FDA-approved targeted therapy", "Information about NCCN guidelines", "Gene Description","Matched Variants in Variant List", "Decision_Table"]
            # list of dataframes
            dfs = [genelt, fdaflt, nccng, desg, varflt, tdt]
            excel_file_name = USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.xlsx'
            # run function
            export_excel(dfs, 'Targeted_Therapy_Variants', excel_file_name, 2, title)
            #file_name.to_csv(r'file_name', index = None, header=True)

            doc = docx.Document()
            doc_file=USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.docx'
            doc.save(doc_file)
            #export_doc(genelt, "Matches found for possible targeted therapy")
            export_doc(fdaflt, "Information about FDA-approved targeted therapy", doc_file)
            export_doc(nccng,"Information about NCCN guidelines", doc_file)
            export_doc(desg, "Gene Description", doc_file)
            export_doc(varflt, "Matched Variants in Variant List", doc_file)
            export_doc(tdt, "Decision Table", doc_file)
            
            title='Click on the links below to download all the tables in excel and doc format'
            display(Markdown('<strong><br/>{}'.format(title)))
            if (path.exists(file+'_Targeted Therapy_Filtered.xlsx'))==True:
                display(FileLink(file+'_Targeted Therapy_Filtered.xlsx'))
            else:
                os.symlink(USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.xlsx',file+'_Targeted Therapy_Filtered.xlsx')
                display(FileLink(file+'_Targeted Therapy_Filtered.xlsx'))
            if (path.exists(file+'_Targeted Therapy_Filtered.docx'))==True:
                display(FileLink(file+'_Targeted Therapy_Filtered.docx'))
            else:
                os.symlink(USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.docx',file+'_Targeted Therapy_Filtered.docx')
                display(FileLink(file+'_Targeted Therapy_Filtered.docx'))


    button.on_click(on_button_clicked)
    

# Buttons for Identification of dMMR Variants

In [ ]:
def MMR_Variants():
    
    def Display_button(b):
    
        MMR_Variants_in_displaytable()
            
        def MMR_Variants_Button1(b):

            MMR_Variants_in_VariantsList()
            '''
            def MMR_Variants_Button1(b):

            decisiontableMMR()
            #clear_output(wait=True)    
            buttonM = widgets.Button(description="Decision Table")
            buttonM.on_click(MMR_Variants_Button1)
            display(buttonM)
            '''
        buttonM = widgets.Button(description="View Information for matched Gene/Variants", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
        buttonM.on_click(MMR_Variants_Button1)
        display(buttonM)
            
    
    buttonM = widgets.Button(description="MMR Variants", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    buttonM.on_click(Display_button)
    display(buttonM)
        

    

In [ ]:
def decisiontableMMR1(file):
    def MMR_Variants_Button1(b):
            
        decisiontableMMR()
    clear_output(wait=True)    
    buttonM = widgets.Button(description="Decision Table")
    buttonM.on_click(MMR_Variants_Button1)
    display(buttonM)

# Identification of dMMR related variants: mutations in MMR genes, MLH1, MSH2, MSH6,PMS2 identified after AF>1 and Depth>100 cutoff - tables and decision table generated, similar to what is done for targeted therapy.

In [ ]:
m_list=pd.DataFrame()
m_list1=pd.DataFrame()
desgMMR=pd.DataFrame()
def MMR_Variants_in_displaytable():
    #file = str(file[0][0]).strip()
    global m_list, desgMMR,m_list1
    news3 = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Display_Table.xlsx')
    #news3 = news3.drop(['Unnamed: 0'], axis = 1)
    '''
    a = widgets.IntSlider(
        value=100,
        min=0,
        max=news3['Depth'].max(),
        step=1,
        description='Depth',
        disabled=False,
        continuous_update=False,
        orientation='horizontal',
        readout=True,
        readout_format='d')
    '''
    a = widgets.BoundedIntText(
        value=100,
        min=0,
        max=news3['Depth'].max(),
        step=1,
        description='Depth',
        disabled=False)
    '''
    b = widgets.Text(
        value='1',
        min=0,
        max=10,
        step=1,
        description='AF',
        disabled=False)
    '''
    b=widgets.BoundedFloatText(
        value=1,
        min=0,
        #max=10.0,
        step=0.1,
        description='AF',
        disabled=False)

    news3.drop(news3.index[news3['NGS QC filter'] != 'PASS'], inplace=True)
    list1 = ['MLH1', 'MSH2', 'MSH6', 'PMS2']
    m_list = news3[news3['Gene Name'].isin(list1)]
    m_list['Depth'] = m_list['Depth'].astype(int)
    m_list['Allele Frequency'] = m_list['Allele Frequency'].astype(str)
    m_list['Allele Frequency']=m_list['Allele Frequency'].str.replace("%","")
    m_list['Allele Frequency'] = m_list['Allele Frequency'].astype(float)
    GeneDes = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'Gene description.xlsx')
    GeneDes.columns = GeneDes.columns.str.strip()
    GeneDes = GeneDes.drop(['Gene Description                                                                                                                                                                                        [Source : CKB core  https://ckb.jax.org/gene/grid]',
           'Description                                                                                                                                                                                                                Source: My Cancer Genome\nhttps://www.mycancergenome.org/content/biomarkers/',
           'Description                                                                                                                                                                                                                Source: NCBI                                                                 https://www.ncbi.nlm.nih.gov/gene'], axis = 1)
    desgMMR = GeneDes[GeneDes['Gene name'].isin(m_list['Gene Name'])]
    #display(desgMMR)
    
    print('Following mutations in MMR genes, MLH1, MSH2, MSH6, PMS2 are identified after AF>1 and Depth>100 cutoff')
    '''
    def reset_button(defaults={}):
        def on_button_clicked(_):
            for k, v in defaults.items(): 
                k.value = v
        button = widgets.Button(description='Set G-KnowMe Default',layout={'width': 'max-content'})
        button.on_click(on_button_clicked)
        display(button)

    reset_button(defaults={a: 100, b: 1})
    #@interact(Depth=a, AF=b)
    '''
    print('\n\nG-Knowme Default Somatic: All the annotated variants are further filtered for AF (allele frequency) > 1%, Depth >100, (AF>5% & Depth >500), Clinical significance: All the terms in display table except variants with "Bengin", "Likely Bengin", "Bengin/Likely Bengin" and SIFT predictions as "All", Variants with (Clinical significance "Uncertain Significance" and SIFT prediction as "Tolerated") and Mutation type "SYNONYMOUS" are elminated') 

    def on_button_clicked_somatic(_):
        global m_list
        df = m_list.drop(m_list[(m_list['Allele Frequency'] < 1)].index)
        #print('\n',df.shape[0],  'entries are selected after applying the filters')
        #display(df)
        df1 = df.drop(df[(df['Depth'] < 100 )].index)
        #print('\n',df1.shape[0],  'entries are selected after applying the filters')
        #display(df1)
        df2= df1.drop(df1[(df1['Depth'] < 500 ) & (df1['Allele Frequency'] < 5)].index)
        #print('\n',df2.shape[0],  'entries are selected after applying the filters')
        #display(df2)
        dff=(df2[~df2['Clinical Significance'].isin(['Benign', 'Likely benign', 'Benign/Likely benign'])])
        #print('\n',dff.shape[0],  'entries are selected after applying the filters')
        #display(dff)
        dff1= dff.drop(dff[(dff['Clinical Significance'] == 'Uncertain significance' ) & (dff['SIFT prediction'] == 'TOLERATED')].index)
        df_new = dff1.drop(dff1[(dff1['Mutation type'] == 'SYNONYMOUS' )].index)
        print( 'After applying Set G-KnowMe Default-Somatic')
        def reset_button(defaults={}):
            def on_button_clicked(_):
                for k, v in defaults.items(): 
                    k.value = v
            button = widgets.Button(description='Reset',layout={'width': 'max-content'})
            button.on_click(on_button_clicked)
            display(button)
        reset_button(defaults={a: 100, b: 1})
        def show_df(Depth, AF):

            global m_list1
            df = df_new
            df['Allele Frequency'] = df['Allele Frequency'].astype(str)
            df['Allele Frequency']=df['Allele Frequency'].str.replace("%","")
            df['Allele Frequency'] = df['Allele Frequency'].astype(float)
            df.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
            df1 = df[(df['Depth'] > Depth) & (df['Allele Frequency'] > AF)]
            df1.fillna('-', inplace=True)
            print('\n',df1.shape[0],  'entries are selected after applying the filters')
            df1['Allele Frequency'] = df1['Allele Frequency'].astype(str)+'%'
            m_list1=df1
            #print(df[df['Mutation type']!=Mutation_type], df[df['Depth'] > Depth], df[df['AF'] > AF], df[df['Clinical Significance']==Clinical_Sig] )
            #(display(df1.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()))
            if df1.empty:
                print('\n','\033[1m' +'No matches found','\n')

            else:
                #print('\n',df1.shape[0],  'entries are selected after applying the filters')
                display(HTML(df1.to_html(render_links=True, escape=False, index=False)))
            return 

        interact(show_df, Depth = a, AF = b)



        def show_df1(Matched_Variants):

            Desc = (GeneDes[GeneDes['Gene name']==Matched_Variants])
            Desc.fillna('-', inplace=True)
            def gdes(Matched_Variants):

                if Desc.empty:
                     print('\n','\033[1m' +'No matches found','\n')

                else:
                    display(Desc.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index())

            return (print('Gene Description'), gdes(Matched_Variants))

        interact(show_df1, Matched_Variants=widgets.Dropdown(options=m_list['Gene Name'].unique(), description='Matched Variants:',
                                                             style= {'description_width': 'initial'}, layout={'width': 'max-content'}))
    button = widgets.Button(description='Set G-KnowMe Default-Somatic',layout={'width': 'max-content'})
    button.on_click(on_button_clicked_somatic)
    display(button)
    print('G-Knowme Default Germline: All the annotated variants are further filtered for AF (allele frequency) >20%, Depth >100, Clinical significance: All the terms in display table except variants with "Bengin", "Likely Bengin", "Bengin/Likely Bengin" and SIFT predictions as "All", Variants with (Clinical significance "Uncertain Significance" and SIFT prediction as "Tolerated") and Mutation type "SYNONYMOUS" are elminated')

    def on_button_clicked_germline(_):
        global m_list
        df = m_list.drop(m_list[(m_list['Allele Frequency'] < 20)].index)
        #print('\n',df.shape[0],  'entries are selected after applying the filters')
        #display(df)
        df1 = df.drop(df[(df['Depth'] < 100 )].index)
        #print('\n',df1.shape[0],  'entries are selected after applying the filters')
        #display(df1)
        dff=(df1[~df1['Clinical Significance'].isin(['Benign', 'Likely benign', 'Benign/Likely benign'])])
        #print('\n',dff.shape[0],  'entries are selected after applying the filters')
        #display(dff)
        dff1= dff.drop(dff[(dff['Clinical Significance'] == 'Uncertain significance' ) & (dff['SIFT prediction'] == 'TOLERATED')].index)
        df_new = dff1.drop(dff1[(dff1['Mutation type'] == 'SYNONYMOUS' )].index)
        print('After applying Set G-KnowMe Default- Germline')
        
        def reset_button(defaults={}):
            def on_button_clicked(_):
                for k, v in defaults.items(): 
                    k.value = v
            button = widgets.Button(description='Reset',layout={'width': 'max-content'})
            button.on_click(on_button_clicked)
            display(button)
        reset_button(defaults={a: 100, b: 1})
        
        def show_df(Depth, AF):

            global m_list1
            df = df_new
            df['Allele Frequency'] = df['Allele Frequency'].astype(str)
            df['Allele Frequency']=df['Allele Frequency'].str.replace("%","")
            df['Allele Frequency'] = df['Allele Frequency'].astype(float)
            df.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
            df1 = df[(df['Depth'] > Depth) & (df['Allele Frequency'] > AF)]
            df1.fillna('-', inplace=True)
            print('\n',df1.shape[0],  'entries are selected after applying the filters')
            df1['Allele Frequency'] = df1['Allele Frequency'].astype(str)+'%'
            m_list1=df1
            #print(df[df['Mutation type']!=Mutation_type], df[df['Depth'] > Depth], df[df['AF'] > AF], df[df['Clinical Significance']==Clinical_Sig] )
            #(display(df1.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()))
            if df1.empty:
                print('\n','\033[1m' +'No matches found','\n')

            else:
                #print('\n',df1.shape[0],  'entries are selected after applying the filters')
                display(HTML(df1.to_html(render_links=True, escape=False, index=False)))
            return 

        interact(show_df, Depth = a, AF = b)



        def show_df1(Matched_Variants):

            Desc = (GeneDes[GeneDes['Gene name']==Matched_Variants])
            Desc.fillna('-', inplace=True)
            def gdes(Matched_Variants):

                if Desc.empty:
                     print('\n','\033[1m' +'No matches found','\n')

                else:
                    display(Desc.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index())

            return (print('Gene Description'), gdes(Matched_Variants))

        interact(show_df1, Matched_Variants=widgets.Dropdown(options=m_list['Gene Name'].unique(),description='Matched Variants:', style= {'description_width': 'initial'},layout={'width': 'max-content'}))
        
    
    button = widgets.Button(description='Set G-KnowMe Default- Germline',layout={'width': 'max-content'})
    button.on_click(on_button_clicked_germline)
    display(button)
    

        #return m_list, desgMMR

In [ ]:
def MMR_Variants_in_VariantsList():
    #file = str(file[0][0]).strip()
    flt = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
    news2 = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Display_Table.xlsx')
    #news2 = news2.drop(['Unnamed: 0'], axis = 1)
    news2.drop(news2.index[news2['NGS QC filter'] != 'PASS'], inplace=True)
    news2['Depth'].astype(int)
    news2['Allele Frequency']=news2['Allele Frequency'].str.replace("%","")
    news2['Allele Frequency'] = news2['Allele Frequency'].astype(float)
    news2.drop(news2.index[news2['Depth'] < 100], inplace=True)
    news2.drop(news2.index[news2['Allele Frequency'] < 1], inplace=True)
    news2['Allele Frequency'] = news2['Allele Frequency'].astype(str)+'%'
    news_gene = news2['Gene Name'].unique()
    MMR_variants = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'MMR gene variants.xlsx', sheet_name = 'Variants')
    MMR_variants['Variant'] = MMR_variants['Variant'].replace('\n','', regex=True)
    MMR_variants['Variant'] = MMR_variants['Variant'].str.replace(' ', '')
    flt['Amino acid change'] = flt['Amino acid change'].str.replace(' ', '')
    aachn = flt['Amino acid change'] 
    MSH2_gene = news2[news2['Gene Name'].isin(MMR_variants['Gene'].unique()) & news2['Gene Name'].isin(MMR_variants['Variant'].unique())]
    MMR_variants_match = MMR_variants[MMR_variants['Gene'].isin(news2['Gene Name'].unique()) & MMR_variants['Variant'].isin(aachn)]
    
    if MSH2_gene.empty:
        print('\033[1m' +'Additional information for MMR gene variants could not be found in the G-KnowMe database.')
            
    else:
    
        @interact
        def show_df(column=['Gene'], Matched_Variants=MSH2_gene['Gene Name'].unique()):
            disp_df = MMR_variants[MMR_variants[column] == Matched_Variants]
            disp_df.fillna('', inplace=True)
            print('Matched variants in MMR Variants list')
            print('\n',disp_df.shape[0],  'entries are selected after applying the filters')
            return disp_df


# dMMR decision Table, automatic export of all dMMR Variants tables and links to download them in excel and word doc format

In [ ]:
dtMMR_MS=pd.DataFrame()
dtMMR_fill=pd.DataFrame()
def decisiontableMMR():
    global dtMMR_MS,dtMMR_fill
    #display(m_list1)
    news4 = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Display_Table.xlsx')
    news4.drop(news4.index[news4['NGS QC filter'] != 'PASS'], inplace=True)
    list1 = ['MLH1', 'MSH2', 'MSH6', 'PMS2']
    m_list = news4[news4['Gene Name'].isin(list1)]
    #m_list['Depth'] = m_list['Depth'].astype(int)
    m_list['Allele Frequency'] = m_list['Allele Frequency'].astype(str)
    df = m_list
    df['Allele Frequency']=df['Allele Frequency'].str.replace("%","")
    df['Allele Frequency'] = df['Allele Frequency'].astype(float)
    df1 = df[(df['Depth'] > 100) & (df['Allele Frequency'] > 1)]
    df1['Allele Frequency'] = df1['Allele Frequency'].astype(str)+'%'
    df1[['Depth', 'Chromosome Number','Chromosome Position']] = df1[['Depth', 'Chromosome Number','Chromosome Position']].astype(str)
    idx = 0
    new_col = ''  # can be a list, a Series, an array or a scalar   
    df1.insert(loc=idx, column='Select', value=new_col)
    #df1.drop(['Unnamed: 0'], axis = 1)
    df1.fillna('-', inplace=True)
    
    sheet1 = sheet(columns=24,  rows = df1.shape[0],
                   column_headers = ['Select', 'Chromosome Number', 'Chromosome Position', 'Reference Nucleotide',
       'Alternate Nucleotide', 'NGS QC filter', 'Depth', 'Allele Frequency', 'dbSNP', 'Gene Name', 'Amino acid position',
       'Amino acid change', 'dbNSFP Amino acid change', 'REGION',
       'Mutation type', 'SIFT prediction', 'HGVS Nomenclature',
       'Clinical Significance', 'Minor_allele_frequency',
       'OMIM id from ClinVar',
       'OMIM Disease from ClinVar', 'OtherIDs', 'COSMIC_link','ClinVar_link'], row_headers=False)
    sheet1.row_headers=False


    for i in range(df1.shape[0]):
        #cell(i, 0, [i])
        #print(dtMMR_MS['MMR gene'].iloc[i])
        cell(i, 0, False )  
        cell(i, 1, df1['Chromosome Number'].iloc[i])
        cell(i, 2, df1['Chromosome Position'].iloc[i])
        cell(i, 3, df1['Reference Nucleotide'].iloc[i])
        cell(i, 4, df1['Alternate Nucleotide'].iloc[i])
        cell(i, 5, df1['NGS QC filter'].iloc[i])
        cell(i, 6, df1['Depth'].iloc[i])
        cell(i, 7, df1['Allele Frequency'].iloc[i])
        cell(i, 8, df1['dbSNP'].iloc[i])
        cell(i, 9, df1['Gene Name'].iloc[i])
        cell(i, 10, df1['Amino acid position'].iloc[i])
        cell(i, 11, df1['Amino acid change'].iloc[i])
        cell(i, 12, df1['dbNSFP Amino acid change'].iloc[i])
        cell(i, 13, df1['REGION'].iloc[i])
        cell(i, 14, df1['Mutation type'].iloc[i])
        cell(i, 15, df1['SIFT prediction'].iloc[i])
        cell(i, 16, df1['HGVS Nomenclature'].iloc[i])
        #cell(i, 15, df1['PolyPhen2 prediction'].iloc[i])
        cell(i, 17, df1['Clinical Significance'].iloc[i])
        cell(i, 18, df1['Minor_allele_frequency'].iloc[i])
        cell(i, 19, df1['OMIM id from ClinVar'].iloc[i])
        cell(i, 20, df1['OMIM Disease from ClinVar'].iloc[i])
        cell(i, 21, df1['OtherIDs'].iloc[i])
        cell(i, 22, df1['COSMIC_link'].iloc[i])
        cell(i, 23, df1['ClinVar_link'].iloc[i])

    display(current())

    #df=to_dataframe(current())
    #display(df)

    def on_update_button_clicked(b):
            global dtMMR_MS, dtMMR_fill
            df=to_dataframe(current())
            #print(df['Checkbox'][0])
            #if df['Checkbox']==False
            if not 'Select' in df.columns:
                print('MMR genes not found')
            elif df[df['Select']==True].empty: 
                print('MMR gene not selected')
            else:
                df.drop(df.loc[df['Select']==False].index, inplace=True)
                dtMMR = pd.DataFrame(columns = ['MMR gene', 'Allele Frequency'])
                dtMMR_MS = df.merge(dtMMR, how='left')
                dtMMR_MS = dtMMR_MS.drop(['Chromosome Number', 'Chromosome Position', 'Reference Nucleotide',
                       'Alternate Nucleotide', 'NGS QC filter', 'Depth', 'Amino acid change', 'dbNSFP Amino acid change', 'REGION',
                       'dbSNP', 'Amino acid position', 'SIFT prediction', 'Clinical Significance',
                       'Minor_allele_frequency',
                       'OMIM id from ClinVar', 'OMIM Disease from ClinVar', 'OtherIDs', 'COSMIC_link'], axis = 1)
                dtMMR_MS = dtMMR_MS.drop(['MMR gene'], axis = 1)
                dtMMR_MS.rename(columns = {'HGVS Nomenclature':'Variant', 'Allele Frequency':'VAF(Variant allele frequency)', 'Gene Name': 'MMR gene' }, inplace = True)
                dtMMR_MS = dtMMR_MS[['MMR gene', 'Variant', 'VAF(Variant allele frequency)', 'Mutation type']]
                #dtMMR_MS['VAF(Variant allele frequency)'] = (100 * dtMMR_MS['VAF(Variant allele frequency)'].astype(float)).astype(str)+'%'
                dtMMR_MS.fillna('', inplace=True)
                #sheet1.row_headers=False
                sheet2 = from_dataframe(dtMMR_MS)
                sheet2.row_headers=False
                clear_output(wait=True)
                print('Select variants for dMMR') 
                display(current())
                print('\n',df.shape[0],  'entries are selected')
                display(updateButton)
                #display(df)
                print('\033[1m' + '\nDecision table for dMMR:')
                display(sheet2)

                #if sheet2 needs to be edited and saved/converted to df, this button
                button = widgets.Button(description="Save Decision Table")
                output2 = widgets.Output()
                print('\033[1m'+'Note: Please Save your final Decision Table. Only saved Decision Table will be exported')
                display(button, output2)

                def on_button_clicked(b):
                    with output2:
                        global dtMMR_MS, dtMMR_fill   
                        dtMMR_fill=ipysheet.to_dataframe(sheet2) 
                        #display(dtMMR_fill)
                        title=["MMR Variants", "Gene Description", "Decision_Table"]
                        # list of dataframes
                        dfs = [m_list1, desgMMR, dtMMR_fill]
                        #dfs=[m_list, desg]
                        excel_file_name = USER_INPUT_FOLDER_PATH+file+'_dMMR_Variants_Filtered.xlsx'
                        # run function
                        export_excel(dfs, 'Targeted_Therapy_Variants', excel_file_name, 2, title)
                        #file_name.to_csv(r'file_name', index = None, header=True)

                        doc = docx.Document()
                        doc_file=USER_INPUT_FOLDER_PATH+file+'_dMMR_Variants_Filtered.docx'
                        doc.save(doc_file)
                        export_doc(desgMMR, "MMR Variants", doc_file)
                        export_doc(dtMMR_fill, "Decision Table", doc_file)
                        #export_doc(m_list, "MMR Variants", doc_file)
                        #export_doc(MMR_variants_match, "Matched variants in MMR Variants list", doc_file)

                        title='Click on the links below to download all the tables in excel and doc format'
                        display(Markdown('<strong><br/>{}'.format(title)))
                        if (path.exists(file+'_dMMR_Variants_Filtered.xlsx'))==True:
                            display(FileLink(file+'_dMMR_Variants_Filtered.xlsx'))
                        else:
                            os.symlink(USER_INPUT_FOLDER_PATH+file+'_dMMR_Variants_Filtered.xlsx',file+'_dMMR_Variants_Filtered.xlsx')
                            display(FileLink(file+'_dMMR_Variants_Filtered.xlsx'))
                        if (path.exists(file+'_dMMR_Variants_Filtered.docx'))==True:
                            display(FileLink(file+'_dMMR_Variants_Filtered.docx'))
                        else:
                            os.symlink(USER_INPUT_FOLDER_PATH+file+'_dMMR_Variants_Filtered.docx',file+'_dMMR_Variants_Filtered.docx')
                            display(FileLink(file+'_dMMR_Variants_Filtered.docx'))


                button.on_click(on_button_clicked)
                        #doc_file=USER_INPUT_FOLDER_PATH+file+'_dMMR Variants_Filtered.docx'
                        #export_doc(dtMMR_MS, "Decision Table", doc_file)


    updateButton = widgets.Button(description = "Show Selection in G-KnowMe Reporting format", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    updateButton.on_click(on_update_button_clicked)
    display(updateButton)
    

# buttons for exporting all targeted therapy and dMMR Variants tables in excel and word doc format

In [ ]:
def Export_all_Tables_in_Excel_wordDoc_format():
    
    print('Click on the links below to download all the tables in excel and doc format:')
    download_file(USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.xlsx')
    download_file(USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.docx')
    def Export_excel_button(b):
        
        title=["Matches found for possible targeted therapy", "Information about FDA-approved targeted therapy", "Gene Description","Matched Variants in Variant List", "Decision_Table"]
        # list of dataframes
        dfs = [genelt, fdaflt, nccng, desg, varflt, tdt]
        excel_file_name = USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.xlsx'
        # run function
        export_excel(dfs, 'Targeted_Therapy_Variants', excel_file_name, 2, title)
        #file_name.to_csv(r'file_name', index = None, header=True)
    
    buttonM = widgets.Button(description="Export Targeted Therapy tables as .xlsx", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    buttonM.on_click(Export_excel_button)
    display(buttonM)
    
    def Export_doc_button(b):
        
        doc = docx.Document()
        doc_file=USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.docx'
        doc.save(doc_file)
        #export_doc(genelt, "Matches found for possible targeted therapy")
        export_doc(fdaflt, "Information about FDA-approved targeted therapy", doc_file)
        export_doc(nccng,"Information about NCCN guidelines", doc_file)
        export_doc(desg, "Gene Description", doc_file)
        export_doc(varflt, "Matched Variants in Variant List", doc_file)
        export_doc(tdt, "Decision Table", doc_file)
    
    buttonM = widgets.Button(description="Export Targeted Therapy tables as .docx", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    buttonM.on_click(Export_doc_button)
    display(buttonM)
    
    def Export_excel_button(b):

        title=["MMR Variants", "Gene Description", "Decision_Table"]
        # list of dataframes
        dfs = [m_list, desgMMR, dtMMR_fill]
        #dfs=[m_list, desg]
        excel_file_name = USER_INPUT_FOLDER_PATH+file+'_dMMR_Variants_Filtered.xlsx'
        # run function
        export_excel(dfs, 'Targeted_Therapy_Variants', excel_file_name, 2, title)
        #file_name.to_csv(r'file_name', index = None, header=True)

    buttonM = widgets.Button(description="Export dMMR Variants tables as .xlsx", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    buttonM.on_click(Export_excel_button)
    display(buttonM)

    def Export_doc_button(b):

        doc = docx.Document()
        doc_file=USER_INPUT_FOLDER_PATH+file+'_dMMR_Variants_Filtered.docx'
        doc.save(doc_file)
        export_doc(desgMMR, "MMR Variants", doc_file)
        export_doc(dtMMR_fill, "Decision Table", doc_file)
        #export_doc(m_list, "MMR Variants", doc_file)
        #export_doc(MMR_variants_match, "Matched variants in MMR Variants list", doc_file)

    buttonN = widgets.Button(description="Export dMMR Variants tables as .docx", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    buttonN.on_click(Export_doc_button)
    display(buttonN)
    

# Identification of variants for Drug Resistance

In [ ]:
DT_drug_res=pd.DataFrame()
def drug_resistance():
    global DT_drug_res
    drug_res = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'Drug resistance information.xlsx')
    fitd = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
    print('\033[1m' + 'List of variants with clinical significance/potential clinical significance (Obtained after applying G-KnowMe filters)')
    #display(fitd)
    display(HTML(fitd.to_html(render_links=True, escape=False, index=False)))

    drug_res.fillna('-', inplace=True)
    l3=fitd['Gene Name'].unique()
    matches_regex = "|".join(l3)
    matches_regex
    drm1 = drug_res.loc[drug_res['Gene'].str.contains(matches_regex, case=False, na=False)]
    drm=drm1.copy()
    #drm['Year of publication'] = drm['Year of publication'].astype(str).apply(lambda x: x.replace('.0',''))
    #drm['Year of publication'] = drm.iloc[:,6].astype(str).apply(lambda x: x.replace('.0',''))
    drm.loc[:,"Year of publication"] = drm.loc[:,"Year of publication"].astype(str).apply(lambda x: x.replace('.0',''))

    tt = widgets.SelectMultiple(
        options=drug_res['Tumor type'].unique(),
        na=False,value=['Lung Cancer', 'Breast Cancer'],description='Choose Tumor Type:',
        #rows=10,
        disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})


    def show_df(Tumor_type):
            global DT_drug_res
            df = drm
            #df1 = drug_res.loc[drug_res['Gene'].str.contains(Tumor_type, case=False, na=False)]
            #df1 = df[(df['Tumor type'] == Tumor_type)]
            print('\033[1m' + 'Relevant Drug resistance information for identified variants')
            l=[]
            for tm in Tumor_type:
                #print(sif)
                l.append(tm)

            DT_drug_res=(df[df['Tumor type'].isin(l)])
            print(DT_drug_res.shape[0],'entries are displayed')
            if DT_drug_res.empty:
                print('\n','\033[1m' +'No matches found','\n')

            else:
                display(DT_drug_res.style.hide_index())
    interact(show_df, Tumor_type=tt)


In [ ]:

def select_drug_res():
    global DT_drug_res
    drug_res = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'Drug resistance information.xlsx')
    fitd = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
    #display(fitd)
    def update_button(_):
        global DT_drug_res
        
    button = widgets.Button(description="UPDATE", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    button.on_click(update_button)
    display(button)
    
    drug_res.fillna('-', inplace=True)
    l3=fitd['Gene Name'].unique()
    matches_regex = "|".join(l3)
    matches_regex
    drm1 = drug_res.loc[drug_res['Gene'].str.contains(matches_regex, case=False, na=False)]
    drm=drm1.copy()
    drm.loc[:,"Year of publication"] = drm.loc[:,"Year of publication"].astype(str).apply(lambda x: x.replace('.0',''))

    l4=drug_res['Gene'].unique()
    matches_regex4 = "|".join(l4)
    matches_regex4
    flm = fitd.loc[fitd['Gene Name'].str.contains(matches_regex4, case=False, na=False)]
    #df=drm.loc[drm['Tumor type'].str.contains('Lung cancer', case=False, na=False)]
    l5=flm['Gene Name'].unique()
    matches_regex5 = "|".join(l5)
    matches_regex5

    df1=flm.copy()
    #df1=fitd
    tumor_types=DT_drug_res['Tumor type'].unique()
    matches_dt='|'.join(tumor_types)
    
    drm2=drm.loc[drm['Tumor type'].str.contains(matches_dt, case=False, na=False)]
    df2=drm2.copy()
    df1['join'] = 1
    df2['join'] = 1
    dl = df1.merge(df2, on='join').drop('join', axis=1)
    df2.drop('join', axis=1, inplace=True)


    dt = dl.loc[dl['Gene Name'].str.contains(matches_regex5, case=False, na=False)]
    dt['M'] = dt.apply(lambda x: x['Gene Name'] in x.Gene, axis=1)
    #dt = dt[dt.M != 'False', inplace=True]
    dt.drop(dt.loc[dt['M']==False].index, inplace=True)
    #print(dt.shape[0])
    #display(dt)
    #dt['Allele Frequency'] = (100 * dt['Allele Frequency'].astype(float)).astype(str)+'%'

    a=widgets.ColorPicker(concise=True,description='Represents columns from Display Table:',value='pink',disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    display(a)
    b=widgets.ColorPicker(concise=True,description='Gene matched drug resistance information from G-KnowMe database:',value='cyan',disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    display(b)
    c=widgets.ColorPicker(concise=True,description='Variant Matches:',value='yellow',disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    display(c)

    df10=dt
    df10[['Chromosome Number','Chromosome Position', 'Year of publication']] = df10[['Chromosome Number','Chromosome Position','Year of publication']].astype(str)
    sheet1 = sheet(columns=18,  rows = df10.shape[0],
                       column_headers = ['Select', 'Chromosome Number', 'Chromosome Position', 'Reference Nucleotide',
           'Alternate Nucleotide', 'Allele Frequency', 'Gene Name',
           'Amino acid change', 'dbNSFP Amino acid change', 'HGVS Nomenclature',
           'Tumor type', 'Potential drug \nresistance', 'Gene', 'Mutation',
           'Description', 'reference', 'Year of publication','Variant Matches'], row_headers=False)
    sheet1.row_headers=False

    for i in range(df10.shape[0]):
        cell(i, 0, False )  
        cell(i, 1, df10['Chromosome Number'].iloc[i], background_color = a.value)
        cell(i, 2, df10['Chromosome Position'].iloc[i], background_color = a.value)
        cell(i, 3, df10['Reference Nucleotide'].iloc[i], background_color = a.value)
        cell(i, 4, df10['Alternate Nucleotide'].iloc[i], background_color = a.value)
        cell(i, 5, df10['Allele Frequency'].iloc[i], background_color = a.value)
        cell(i, 6, df10['Gene Name'].iloc[i], background_color = a.value)
        cell(i, 7, df10['Amino acid change'].iloc[i], background_color = a.value)
        cell(i, 8, df10['dbNSFP Amino acid change'].iloc[i], background_color = a.value)
        cell(i, 9, df10['HGVS Nomenclature'].iloc[i], background_color = a.value)
        cell(i, 10, df10['Tumor type'].iloc[i], background_color = b.value)
        cell(i, 11, df10['Potential drug \nresistance'].iloc[i], background_color = b.value)
        cell(i, 12, df10['Gene'].iloc[i], background_color = b.value)
        cell(i, 13, df10['Mutation'].iloc[i], background_color = b.value)
        cell(i, 14, df10['Description'].iloc[i], background_color = b.value)
        cell(i, 15, df10['reference'].iloc[i], background_color = b.value)
        cell(i, 16, df10['Year of publication'].iloc[i], background_color = b.value)
        cell(i, 17, df10.apply(lambda x: x['Mutation'] in x['Amino acid change'], axis=1).iloc[i])

        v_match=cell(i, 17, df10.apply(lambda x: x['Mutation'] in x['Amino acid change'], axis=1).iloc[i])
        #print(v_match.value)
        if v_match.value==True:
        #if v_match.value==True & v_match.value != '-':
            cell(i, 7,df10['Amino acid change'].iloc[i], background_color = 'yellow')
            cell(i, 13, df10['Mutation'].iloc[i], background_color = 'yellow')
            #cell(i, 7,df10['Amino acid change'].iloc[i], color="purple", background_color=a.value, font_style="times new roman", font_weight="bold")
            #cell(i, 13, df10['Mutation'].iloc[i], color="purple", background_color=b.value, font_style="times new roman", font_weight="bold")

    display(current())

    #df=to_dataframe(current())
    #display(df)

    def on_update_button_clicked(b):
        global DT_drug_res
        df10=to_dataframe(current())
        if df10[df10['Select']==True].empty:
            print('No entries have been selected in the Drug Resistance Table')
        else:
            df10.drop(df10.loc[df10['Select']==False].index, inplace=True)
            df10.drop(['Select', 'Gene', 'Mutation','Description', 'Year of publication', 'Variant Matches'], axis=1, inplace=True)
            sheet2 = from_dataframe(df10)
            sheet2.row_headers=False
            clear_output(wait=True)
            print('Select variants') 
            display(current())
            print('\n',df10.shape[0],  'entries are selected')
            display(updateButton)
            #display(df10)
            print('\033[1m' + '\nDecision table for Drug Resistance:')
            display(sheet2)
            DT_drug_res=ipysheet.to_dataframe(sheet2)
            def save_table(b):
                global DT_drug_res
                new1=tdt.copy()
                new1.drop(new1.loc[new1['Want to include Variant in the report?']==False].index, inplace=True)
                #display(new1)
                title=["Matches found for possible targeted therapy", "Information about FDA-approved targeted therapy", "Information about NCCN guidelines", "Gene Description","Matched Variants in Variant List", "Decision_Table", "Decision table for Drug Resistance"]
                # list of dataframes
                dfs = [genelt, fdaflt, nccng, desg, varflt,tdt, DT_drug_res ]
                excel_file_name = USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.xlsx'
                # run function
                export_excel(dfs, 'Targeted_Therapy_Variants', excel_file_name, 2, title)
                #file_name.to_csv(r'file_name', index = None, header=True)

                doc = docx.Document()
                doc_file=USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.docx'
                doc.save(doc_file)
                #export_doc(genelt, "Matches found for possible targeted therapy")
                export_doc(fdaflt, "Information about FDA-approved targeted therapy", doc_file)
                export_doc(nccng,"Information about NCCN guidelines", doc_file)
                export_doc(desg, "Gene Description", doc_file)
                export_doc(varflt, "Matched Variants in Variant List", doc_file)
                export_doc(tdt, "Decision Table", doc_file)
                export_doc(DT_drug_res, "Decision table for Drug Resistance", doc_file)

                title='Click on the links below to download all the tables in excel and doc format'
                display(Markdown('<strong><br/>{}'.format(title)))
                if (path.exists(file+'_Targeted Therapy_Filtered.xlsx'))==True:
                    display(FileLink(file+'_Targeted Therapy_Filtered.xlsx'))
                else:
                    os.symlink(USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.xlsx',file+'_Targeted Therapy_Filtered.xlsx')
                    display(FileLink(file+'_Targeted Therapy_Filtered.xlsx'))
                if (path.exists(file+'_Targeted Therapy_Filtered.docx'))==True:
                    display(FileLink(file+'_Targeted Therapy_Filtered.docx'))
                else:
                    os.symlink(USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.docx',file+'_Targeted Therapy_Filtered.docx')
                    display(FileLink(file+'_Targeted Therapy_Filtered.docx'))

            button = widgets.Button(description="Save Decision Table", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
            button.on_click(save_table)
            display(button)
    updateButton = widgets.Button(description = "Show Selection in G-KnowMe Reporting format", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    updateButton.on_click(on_update_button_clicked)
    display(updateButton)
    



# Prognostic information about identified variants

In [ ]:
prog_select=pd.DataFrame()
def prognostic_information():
    global prog_select
    prog_info = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'Information related to prognosis.xlsx')
    fitd = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
    print('\033[1m' + 'List of variants with clinical significance/potential clinical significance (Obtained after applying G-KnowMe filters)')
    #display(fitd)
    display(HTML(fitd.to_html(render_links=True, escape=False, index=False)))
    
  
    prog_info.fillna('-', inplace=True)
    l3=fitd['Gene Name'].unique()
    matches_regex = "|".join(l3)
    matches_regex
    #display(matches_regex)
    prm1 = prog_info.loc[prog_info['Gene'].str.contains(matches_regex, case=False, na=False)]
    #display(prm1)
    #prm1 = prm1.iloc[: , 1:]
    prm=prm1.copy()
    
    prm.loc[:,"Year of publication"] = prm.loc[:,"Year of publication"].astype(str).apply(lambda x: x.replace('.0',''))
    #display(prm)
    
    tt = widgets.SelectMultiple(
        options=prog_info['Tumor type'].unique(),
        na=False,value=['Lung Cancer','Breast Cancer'],description='Choose Tumor Type:',
        #rows=10,
        disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})


    def show_df(Tumor_type):
            global prog_select
            df = prm
            #df1 = drug_res.loc[drug_res['Gene'].str.contains(Tumor_type, case=False, na=False)]
            #df1 = df[(df['Tumor type'] == Tumor_type)]
            print('\033[1m' + 'Relevant prognostic information for identified variants')
            l=[]
            for tm in Tumor_type:
                #print(sif)
                l.append(tm)

            prog_select=(df[df['Tumor type'].isin(l)])
            print(prog_select.shape[0],'entries are displayed')
            if prog_select.empty:
                print('\n','\033[1m' +'No matches found','\n')

            else:
                display(prog_select.style.hide_index())
            
    interact(show_df, Tumor_type=tt)
    

In [ ]:
def select_entries_for_prognosis():
    global prog_select
    prog_info = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'Information related to prognosis.xlsx')
    fitd = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
    #print('\033[1m' + 'List of variants with clinical significance/potential clinical significance (Obtained after applying G-KnowMe filters)')
    #display(fitd)
    def update_button(_):
        global prog_select
        
    button = widgets.Button(description="UPDATE", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    button.on_click(update_button)
    display(button)
    
    prog_info.fillna('-', inplace=True)
    l3=fitd['Gene Name'].unique()
    matches_regex = "|".join(l3)
    matches_regex
    prm1 = prog_info.loc[prog_info['Gene'].str.contains(matches_regex, case=False, na=False)]
    #prm1 = prm1.iloc[: , 1:]
    prm=prm1.copy()
    prm.loc[:,"Year of publication"] = prm.loc[:,"Year of publication"].astype(str).apply(lambda x: x.replace('.0',''))
    

    l4=prog_info['Gene'].unique()
    matches_regex4 = "|".join(l4)
    matches_regex4
    flm = fitd.loc[fitd['Gene Name'].str.contains(matches_regex4, case=False, na=False)]
    #df=prm.loc[prm['Tumor type'].str.contains('Lung cancer', case=False, na=False)]
    l5=flm['Gene Name'].unique()
    matches_regex5 = "|".join(l5)
    matches_regex5

    df1=flm.copy()
    #df1=fitd
    
    tumor_types=prog_select['Tumor type'].unique()
    matches_tt = '|'.join(tumor_types)
    prm2=prm.loc[prm['Tumor type'].str.contains(matches_tt, case=False, na=False)]
    df2=prm2.copy()
    df1['join'] = 1
    df2['join'] = 1
    dl = df1.merge(df2, on='join').drop('join', axis=1)
    df2.drop('join', axis=1, inplace=True)
    
    #display(df2)
    if df2.empty:
        print('\n','\033[1m' +'No matches found','\n')

    else:
        dt = dl.loc[dl['Gene Name'].str.contains(matches_regex5, case=False, na=False)]
        dt['match'] = dt.apply(lambda x: x['Gene Name'] in x.Gene, axis=1)
        #dt = dt[dt.M != 'False', inplace=True]
        dt.drop(dt.loc[dt['match']==False].index, inplace=True)
        #print(dt.shape[0])
        #display(dt)
        #dt['Allele Frequency'] = (100 * dt['Allele Frequency'].astype(float)).astype(str)+'%'

        a=widgets.ColorPicker(concise=True,description='Represents columns from Display Table:',value='pink',disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
        display(a)
        b=widgets.ColorPicker(concise=True,description='Gene matched drug resistance information from G-KnowMe database:',value='cyan',disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
        display(b)
        c=widgets.ColorPicker(concise=True,description='Variant Matches:',value='yellow',disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
        display(c)
        df10=dt
        df10[['Chromosome Number','Chromosome Position', 'Year of publication']] = df10[['Chromosome Number','Chromosome Position','Year of publication']].astype(str)
        sheet1 = sheet(columns=16,  rows = df10.shape[0],
                           column_headers = ['Select', 'Chromosome Number', 'Chromosome Position', 'Reference Nucleotide',
               'Alternate Nucleotide', 'Allele Frequency', 'Gene Name','Amino acid change', 'dbNSFP Amino acid change', 'HGVS Nomenclature',
               'Tumor type', 'Mutations', 'Prognostic significance','references/PubMed', 'Year of publication','Variant Matches'], row_headers=False)
        sheet1.row_headers=False

        for i in range(df10.shape[0]):
            cell(i, 0, False )  
            cell(i, 1, df10['Chromosome Number'].iloc[i], background_color = a.value)
            cell(i, 2, df10['Chromosome Position'].iloc[i], background_color = a.value)
            cell(i, 3, df10['Reference Nucleotide'].iloc[i], background_color = a.value)
            cell(i, 4, df10['Alternate Nucleotide'].iloc[i], background_color = a.value)
            cell(i, 5, df10['Allele Frequency'].iloc[i], background_color = a.value)
            cell(i, 6, df10['Gene Name'].iloc[i], background_color = a.value)
            cell(i, 7, df10['Amino acid change'].iloc[i], background_color = a.value)
            cell(i, 8, df10['dbNSFP Amino acid change'].iloc[i], background_color = a.value)
            cell(i, 9, df10['HGVS Nomenclature'].iloc[i], background_color = a.value)
            cell(i, 10, df10['Tumor type'].iloc[i], background_color = b.value)
            #cell(i, 11, df10['Potential drug \nresistance'].iloc[i], background_color = 'orange')
            #cell(i, 11, df10['Gene'].iloc[i], background_color = 'orange')
            cell(i, 11, df10['Mutations'].iloc[i], background_color = b.value)
            cell(i, 12, df10['Prognostic significance'].iloc[i], background_color = b.value)
            cell(i, 13, df10['references/PubMed'].iloc[i], background_color = b.value)
            cell(i, 14, df10['Year of publication'].iloc[i], background_color = b.value)
            cell(i, 15, df10.apply(lambda x: x['Mutations'] in x['Amino acid change'], axis=1).iloc[i])

            v_match=cell(i, 15, df10.apply(lambda x: x['Mutations'] in x['Amino acid change'], axis=1).iloc[i])
            #print(v_match.value)
            if v_match.value==True:
            #if v_match.value==True & v_match.value != '-':
                cell(i, 7,df10['Amino acid change'].iloc[i], background_color = 'yellow')
                cell(i, 11, df10['Mutations'].iloc[i], background_color = 'yellow')
                #cell(i, 7,df10['Amino acid change'].iloc[i], color="purple", background_color=a.value, font_style="times new roman", font_weight="bold")
                #cell(i, 11, df10['Mutations'].iloc[i], color="purple", background_color=b.value, font_style="times new roman", font_weight="bold")


        display(current())

        def on_update_button_clicked(b):
            global prog_select
            df10=to_dataframe(current())
            if df10[df10['Select']==True].empty:
                print('No entries have been selected in the Prognostic significance Table') 
            else:
                df10.drop(df10.loc[df10['Select']==False].index, inplace=True)
                df10.drop(['Select', 'Mutations', 'Year of publication','Variant Matches'], axis=1, inplace=True)
                sheet2 = from_dataframe(df10)
                sheet2.row_headers=False
                clear_output(wait=True)
                print('Select variants') 
                display(current())
                print('\n',df10.shape[0],  'entries are selected')
                display(updateButton)
                #display(df10)
                print('\033[1m' + '\nDecision table for Prognostic significance:')
                display(sheet2)
                prog_select=ipysheet.to_dataframe(sheet2)
            def save_table(b):
                global prog_select
                new1=tdt.copy()
                new1.drop(new1.loc[new1['Want to include Variant in the report?']==False].index, inplace=True)
                #display(new1)
                title=["Matches found for possible targeted therapy", "Information about FDA-approved targeted therapy", "Information about NCCN guidelines", "Gene Description","Matched Variants in Variant List", "Decision_Table", "Decision table for Drug Resistance", "Decision table for Prognostic significance"]
                # list of dataframes
                dfs = [genelt, fdaflt, nccng, desg, varflt,tdt, DT_drug_res, prog_select]
                excel_file_name = USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.xlsx'
                # run function
                export_excel(dfs, 'Targeted_Therapy_Variants', excel_file_name, 2, title)
                #file_name.to_csv(r'file_name', index = None, header=True)

                doc = docx.Document()
                doc_file=USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.docx'
                doc.save(doc_file)
                #export_doc(genelt, "Matches found for possible targeted therapy")
                export_doc(fdaflt, "Information about FDA-approved targeted therapy", doc_file)
                export_doc(nccng,"Information about NCCN guidelines", doc_file)
                export_doc(desg, "Gene Description", doc_file)
                export_doc(varflt, "Matched Variants in Variant List", doc_file)
                export_doc(tdt, "Decision Table", doc_file)
                export_doc(DT_drug_res, "Decision table for Drug Resistance", doc_file)
                export_doc(DT_drug_res, "Decision table for Prognostic significance", doc_file)

                title='Click on the links below to download all the tables in excel and doc format'
                display(Markdown('<strong><br/>{}'.format(title)))
                if (path.exists(file+'_Targeted Therapy_Filtered.xlsx'))==True:
                    display(FileLink(file+'_Targeted Therapy_Filtered.xlsx'))
                else:
                    os.symlink(USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.xlsx',file+'_Targeted Therapy_Filtered.xlsx')
                    display(FileLink(file+'_Targeted Therapy_Filtered.xlsx'))
                if (path.exists(file+'_Targeted Therapy_Filtered.docx'))==True:
                    display(FileLink(file+'_Targeted Therapy_Filtered.docx'))
                else:
                    os.symlink(USER_INPUT_FOLDER_PATH+file+'_Targeted Therapy_Filtered.docx',file+'_Targeted Therapy_Filtered.docx')
                    display(FileLink(file+'_Targeted Therapy_Filtered.docx'))

            button = widgets.Button(description="Save Decision Table", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
            button.on_click(save_table)
            display(button)
                
        updateButton = widgets.Button(description = "Show Selection in G-KnowMe Reporting format", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
        updateButton.on_click(on_update_button_clicked)
        display(updateButton)


# Evidence based variant categorization and Targeted therapy: Tables in G-KnowMe report format

In [ ]:
def categorization():
    #display(tdt,DT_target_therapy)
    #tab = tdt.copy()
    new=tdt.copy()
    if not 'Chromosome Number' in new.columns:
        print('Targeted therpay decision Table has not been saved')
    else:
        gk=new.drop(['Chromosome Number', 'Chromosome Position', 'Reference Nucleotide','Alternate Nucleotide','Amino acid change', 'dbNSFP Amino acid change', 'Variant effect (reported or probable)','Want to include Variant in the report?',
               'Need Clinical trial?'], axis = 1)
        #sh1['']='Tier II'
        gk=gk[['Gene Name', 'HGVS Nomenclature','Allele Frequency','Targeted therapies Tier I', 'Targeted therapies Tier II']]
        #gk['Allele Frequency'] = (100 * gk['Allele Frequency']).astype(str)+'%'
        gk
        gk.loc[:,"Allele Frequency"] = gk.loc[:,"Allele Frequency"].astype(str)
        if not 'Allele Frequency' in DT_drug_res.columns:
            print('No entries have been selected in the Drug Resistance Decision Table')
        else:
            DT_drug_res.loc[:,"Allele Frequency"] = DT_drug_res.loc[:,"Allele Frequency"].astype(str)
            rep1 = pd.merge(gk, DT_drug_res, how='left', on=['Gene Name', 'HGVS Nomenclature','Allele Frequency'])
            rep2=rep1.drop(['Chromosome Number', 'Chromosome Position', 'Reference Nucleotide','Alternate Nucleotide','Amino acid change', 'dbNSFP Amino acid change','Tumor type', 
               'reference'], axis = 1)
            #print(rep2.columns)
            #rep2['Allele Frequency'] = (100 * rep2['Allele Frequency'].astype(float)).astype(str)+'%'
            rep2.rename(columns = {'Gene Name':'Gene', 'HGVS Nomenclature':'Variant', 'Allele Frequency':'VAF', 'Targeted therapy':'Targeted therapies','Potential drug \nresistance':'Potential drug resistance'}, inplace = True)
            rep2.fillna('-', inplace=True)
            rep2 = rep2.replace('\n',', ', regex=True)
            pd.set_option('display.max_colwidth', None)
            #display(rep2.style.hide_index())
            dft=rep2.copy()
            dft1=dft.loc[dft['Targeted therapies Tier I'] != '']
            dft1['']='Tier I'
            dft11=dft1.drop(['Targeted therapies Tier II'], axis = 1)

            dft11.rename(columns = {'Targeted therapies Tier I':'Targeted therapies'},inplace=True)
            dft2=dft.loc[dft['Targeted therapies Tier II'] != '']
            dft2['']='Tier II'
            #display(dft2)
            dft22=dft2.drop(['Targeted therapies Tier I'], axis = 1)

            dft22.rename(columns = {'Targeted therapies Tier II':'Targeted therapies'},inplace=True)
            dft3=dft11.append(dft22)
            dft3=dft3[['','Gene', 'Variant','VAF','Targeted therapies','Potential drug resistance']]
            if dft22.empty:
                dft3.loc[len(dft3.index)] = ['Tier II','','','','',''] 
            if dft11.empty:  
                new_row = pd.DataFrame({'':'Tier I', 'Gene':'', 'Variant':'','VAF':'', 'Targeted therapies':'', 'Potential drug resistance':''}, index =[0])
                # Concatenate new_row with df
                dft3 = pd.concat([new_row, dft3[:]]).reset_index(drop = True)

            #t3d1= dft3.groupby(['Gene','Variant', 'VAF','Targeted therapies','','Potential drug resistance']).sum()
            t3d1= dft3.groupby(['Gene','Variant', 'VAF','','Targeted therapies','Potential drug resistance'],sort=False).count()
            #df = t3d1.drop_duplicates('', keep='first')
            #display(t3d1)
            t3d = dft3.groupby(['','Gene', 'Variant','VAF','Targeted therapies','Potential drug resistance']).sum()
            #display(t3d)
            df=t3d.copy()
            display(HTML(df.to_html()))
           

# Identification of variants for Clinical Trials

In [ ]:
def merge_information():
    global genelt, gene_info_fitd, var_info
    gene_info = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'Gene description.xlsx')
    fitd = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
    gene_info.columns = gene_info.columns.str.strip()
    gene_info = gene_info.drop(['Gene Description                                                                                                                                                                                        [Source : CKB core  https://ckb.jax.org/gene/grid]',
           'Description                                                                                                                                                                                                                Source: My Cancer Genome\nhttps://www.mycancergenome.org/content/biomarkers/',
           'Description                                                                                                                                                                                                                Source: NCBI                                                                 https://www.ncbi.nlm.nih.gov/gene', 'Associated Pathways                              Source: My Cancer Genome'], axis = 1)
    gene_info['Gene name'] = gene_info['Gene name'].str.replace('\n','')
    gene_info_lt = gene_info['Gene name'].str.strip()
    #display(gene_info_lt)
    fitd_lt = fitd['Gene Name'].str.strip()
    #display(fitd_lt)
    genelt = fitd[fitd['Gene Name'].isin(gene_info_lt)]
    #display(genelt)
    gene_info_fitd =gene_info[gene_info['Gene name'].isin(fitd_lt)]
    #print(len(gene_info_fitd))
    #display(gene_info_fitd)
    gene_info_fitd.rename(columns = {'Gene name':'Gene Name', 'Gene Role' :'Oncogene/tumor suppressor', 'Gene Description from G-KnowMe Database':'Gene description'}, inplace = True)
    #print(list(gene_info_fitd))

    vard2 = pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'Variant description.xlsx', sheet_name='Variant Description')
    vard2.columns = vard2.columns.str.strip()
    vard2 = vard2.drop(['Variant Description                                                              Source: CKB core',
           'Source: Uniprot\nhttps://www.uniprot.org/uniprot/P42336',
           'Source: SIFT/Polyphen 2 predictions', 'Variant \nHGVS', 'Variant \ndbSNP ID', 'G-KnowMe category of the variant'], axis = 1)
    #display(vard2)

    vard2['Gene name'] = vard2['Gene name'].str.replace(' ', '')
    vard2['Variant\n(Amino acid change)'] = vard2['Variant\n(Amino acid change)'].str.replace(' ', '')
    fitd['Amino acid change'] = fitd['Amino acid change'].str.replace(' ', '')
    aachn = fitd['dbNSFP Amino acid change'].str.strip()
    l=list(aachn)
    t=[]
    for i in l:
        sub=i.split(' ')
        #print(sub)
        for k in sub:
                sub1=k.replace(',', '')
                t.append(sub1)

    var_info = vard2[vard2['Gene name'].isin(fitd_lt) & vard2['Variant\n(Amino acid change)'].isin(t)]
    var_info.reset_index(inplace = True, drop = True)
    #display(genelt, var_info, gene_info)
    var_info.rename(columns = {'Gene name':'Gene Name', 'Variant description from G-KnowMe  database' : 'Variant description'}, inplace = True)
    
    #print(list(var_info))



In [ ]:
def table_for_variant_for_clinical_trials():
    global genelt, gene_info_fitd, var_info, df12
    fitd = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
    dt_vcl= pd.DataFrame(columns = ['Gene Name', 'Interpretation', 'Do you want to search clinical trails for this variant ?'])
    Decision_Table = var_info.merge(dt_vcl, how='left')
    Decision_Table = Decision_Table.drop (['Variant\n(Amino acid change)', 'Protein Effect'], axis = 1)
    #display(Decision_Table)
    Decision_Table = gene_info_fitd.merge(Decision_Table, how='left')
    #display(Decision_Table)
    Decision_Table = genelt.merge(Decision_Table, how='left')
    #display(Decision_Table)
    Decision_Table = Decision_Table.drop (['NGS QC filter', 'Depth', 'dbSNP', 'Amino acid position', 'Amino acid change', 'REGION', 'Minor_allele_frequency', 'OMIM id from ClinVar', 'OMIM Disease from ClinVar', 'OtherIDs'], axis = 1)
    Decision_Table.fillna('', inplace=True)
    df11 = Decision_Table
    df11[['Chromosome Number','Chromosome Position']] = df11[['Chromosome Number','Chromosome Position']].astype(str)
    #display(df11)
    sheet1 = ipysheet.sheet(columns=19,  rows = df11.shape[0],  
            column_headers = ['Chromosome Number', 'Chromosome Position','Reference Nucleotide','Alternate Nucleotide', 'Gene Name', 'G-knowMe category of genes', 'Allele Frequency', 'Mutation type','dbNSFP Amino acid change', 'HGVS Nomenclature','Oncogene/tumor suppressor', 'Gene description', 'Variant description',  'SIFT prediction', 'Clinical Significance', 'ClinVar_link', 'COSMIC_link', 'Interpretation', 'Do you want to search clinical trails for this variant ?'], row_headers=False)
    sheet1.row_headers=False

    for i in range(df11.shape[0]):
        cell(i, 0, df11['Chromosome Number'].iloc[i])
        cell(i, 1, df11['Chromosome Position'].iloc[i])
        cell(i, 2, df11['Reference Nucleotide'].iloc[i])
        cell(i, 3, df11['Alternate Nucleotide'].iloc[i])
        cell(i, 4, df11['Gene Name'].iloc[i])
        cell(i, 5, df11['G-knowMe category of genes'].iloc[i])
        cell(i, 6, df11['Allele Frequency'].iloc[i])
        cell(i, 7, df11['Mutation type'].iloc[i])
        cell(i, 8, df11['dbNSFP Amino acid change'].iloc[i])
        cell(i, 9, df11['HGVS Nomenclature'].iloc[i])
        cell(i, 10, df11['Oncogene/tumor suppressor'].iloc[i])
        cell(i, 11, df11['Gene description'].iloc[i]) 
        cell(i, 12, df11['Variant description'].iloc[i])
        cell(i, 13, df11['SIFT prediction'].iloc[i])
        cell(i, 14, df11['Clinical Significance'].iloc[i])
        cell(i, 15, df11['ClinVar_link'].iloc[i])
        cell(i, 16, df11['COSMIC_link'].iloc[i])
        cell(i, 17, df11['Interpretation'].iloc[i])
        cell(i, 18, False)

    current().row_headers=False
    display(current())
    def save_table(b):
        global df12
        df10=to_dataframe(current())
        if df10[df10['Do you want to search clinical trails for this variant ?']==True].empty:
            print('No entries have been selected in the Prognostic significance Table') 
        else:
            df10.drop(df10.loc[df10['Do you want to search clinical trails for this variant ?']==False].index, inplace=True)
            df10.drop(['Do you want to search clinical trails for this variant ?'], axis=1, inplace=True)
            sheet2 = from_dataframe(df10)
            sheet2.row_headers=False  
            #print('\n',df10.shape[0], 'entries are selected')
            #print('\033[1m' + '\nIdentification of variants for Clinical trial:')
            #display(sheet2)
            df12=ipysheet.to_dataframe(sheet2)
            #display(HTML(df12.to_html(render_links=True, escape=False, index=False)))
            file_name = USER_INPUT_FOLDER_PATH+file+'_Clinical_trials.xlsx'
            df12.to_excel(file_name, index=False)
            title='\nClick on the link below to download the table'
            display(Markdown('<strong><br/>{}'.format(title)))
            #download_file(USER_INPUT_FOLDER_PATH+file+'_Filtered.xlsx')
            if (path.exists(file+'_Clinical_trials.xlsx'))==True:
                display(FileLink(file+'_Clinical_trials.xlsx'))
                #url=FileLink(file+'_Filtered.xlsx')
            else:
                os.symlink(USER_INPUT_FOLDER_PATH+file+'_Clinical_trials.xlsx',file+'_Clinical_trials.xlsx')
                display(FileLink(file+'_Clinical_trials.xlsx'))

    button = widgets.Button(description="Save Table", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    button.on_click(save_table)
    display(button)

   
    

# Clinical Trials

In [ ]:
'''
def style():
    style = {'description_width': 'initial'}
    flt= pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Clinical_trials.xlsx')
    flt['Gene Name']=flt['Gene Name'].astype(str)
    gene= list(flt['Gene Name'].unique())
    Gene_name=widgets.SelectMultiple(
        options = gene,
        value= gene,
        description='Gene Name:',
        disabled=False
    )
    Gender=widgets.Dropdown(
        options=['All','Female','Male'],
        description='Gender:',
        disabled=False,
    )
    Bio_marker=widgets.Dropdown(
        options=['NONE','TMB', 'MSI', 'dMMR'],
        value='NONE',
        description='Other predictive Biomarkers:',
        disabled=False,
        style=style,
    )
    Tumor=widgets.SelectMultiple( 
        options=[('All','all'), ('Breast Cancer','breast'), ('Lung Cancer','lung'), ('All Solid tumors','solid'), ('Spindle Cell tumor','Spindle'), ('Malignant Peripheral nerve sheath tumor (MPNST)','nerve sheath tumor ,MPNST')],
        value=['all'],
        description='Tumor/Cancer:',
        disabled=False,
        style=style,
    )
    Age=widgets.Dropdown(
        options=[('Child (Birth-17 yrs)', 'child'), ('Adult (18-64 yrs)', 'adult'), ('Older Adult (65+ yrs)', 'older'),('All', 'all')],
        value='all',
        description='Age:',
        style=style,
    )
    Medical_condition=widgets.Text(
        value='Enter Medical Condition',
        placeholder='Enter Medical Condition',
        description='Medical Condition:',
        disabled=False,
        style=style,
    )


    conn = pymysql.connect(user='admin', password='gkmadmin', host='gknominer-instance.c2vk6g6oesho.us-east-2.rds.amazonaws.com', database='gknominer')

    #Creating a cursor object using the cursor() method
    cursor = conn.cursor()
    def extract_trials():
        gene_name_string, tumor_string, bio_marker_string,gender_string, age_string = get_search_strings()

        #trials_df=pd.read_excel('CT_data_5.xlsx')
        sql = "SELECT DISTINCT ct.nct_id 'NCTID', official_title 'Official Title', intervention_treatment 'Intervention/Treatment',condition_disease 'Condition/Disease', "\
                "phase 'Phase',location 'Trial Location',GROUP_CONCAT(g2.gene_name SEPARATOR ' , ') 'Gene Name' "\
              "FROM clinical_trials ct, clinical_trial_genes g1,  clinical_trial_genes g2 "\
              "WHERE ct.nct_id=g1.nct_id AND g1.nct_id=g2.nct_id "

        if len(gene_name_string)>0 :
            sql = sql + " AND " + gene_name_string 

        if len(bio_marker_string)>0 :
            sql = sql + " AND " + bio_marker_string 

        if len(tumor_string)>0 :
            sql = sql + " AND " + tumor_string 

        if len(gender_string)>0 :
            sql = sql + " AND " + gender_string 

        if len(age_string)>0 :
            sql = sql + " AND " + age_string 

        sql = sql + " GROUP BY ct.nct_id ORDER BY ct.nct_id"
        print("SQL :",sql)

        cursor.execute(sql)
        trials_df = pd.DataFrame(cursor.fetchall())
        #print(trials_df)

        if len(trials_df) > 0 :
            trials_df.columns = [i[0] for i in cursor.description]
            #print(trials_df.columns)

        return trials_df
    def on_button_clicked(b):
        with output:
            output.clear_output()
            global trials
            trials = extract_trials()
            if len(trials) > 0 :
                trials = trials.replace(r'\n',' ', regex=True) 
                name = 'Filtered_Clinical_Trials.xlsx'
                trials.to_excel(name,index=False)

                print("\033[1mClick on the below link to download Clinical Trials data")
                display(FileLink(name))

                print('\n\033[1mDisplaying ',len(trials), ' Interventional and Recruting clinical trials for Genes:', Gene_name.value, \
                      ' Gender: ', Gender.value,' Age: ' , Age.value ,' Tumor: ',Tumor.value, ' Bio Marker: ',Bio_marker.value)

                display(trials.style.hide_index())

            else :
                print('\033[1m', 'No data found')

    button=widgets.Button(
    value=False,
    description='Search Clinical Trials',
    disabled=False,
    button_style='warning', 
    tooltip='Search Clinical Trials',
    )

    button.on_click(on_button_clicked)
    trials = pd.DataFrame()

    def get_search_strings():
    
        #print(Gene_name.value)
        gene_name_string = "(" 
        for val in Gene_name.value:
            if len(gene_name_string) >1:
                gene_name_string = gene_name_string + " OR "
            gene_name_string = gene_name_string + " lower(g1.gene_name)='"+val.strip().lower()+"'" 
        gene_name_string = gene_name_string + ")"

        #hack: if none is selected in gene name list, then reurn empty string
        if gene_name_string.find('none') != -1 :
            gene_name_string=''
        #print(gene_name_string)

        #print(Gender.value)
        gender_string = ''
        if Gender.value.lower().find('all') == -1:
            gender_string="lower(ct.eligibility_gender)='"+Gender.value.strip().lower()+"'"
        #print(gender_string)

        #print(Bio_marker.value)
        bio_marker_string=''
        if Bio_marker.value.lower().find('none') == -1:
            bio_marker_string= " lower(g1.gene_name)='"+Bio_marker.value.strip().lower()+"'" 
        #print(gene_name_string)

        #print(Tumor.value)
        tumor_string = '('
        for val in Tumor.value:
            #print(val)
            for term in val.strip().lower().split(','):
                #print('\t',term)
                if len(tumor_string) >1:
                    tumor_string = tumor_string + " OR "
                tumor_string =tumor_string + " lower(ct.official_title) like '%"+term.lower()+"%' OR lower(ct.condition_disease) like '%" +term.lower()+"%'" 
        tumor_string = tumor_string + ")"

        #hack: if all is selected in tumor, then reurn empty string
        if tumor_string.find('all') != -1 :
            tumor_string=''
        #print(tumor_string)

        #print(Age.value)
        age_string = ''
        if Age.value.find('all') == -1:
            age_string="lower(ct.eligibility_age) like '%"+Age.value.strip().lower()+"%'"
        #print(age_string)

        #print(Medical_condition.value)

        #print('gene_name_string:',gene_name_string,'\ntumor_string:',tumor_string, '\ngender_string:', gender_string, '\nage_string:',age_string)
        return gene_name_string, tumor_string, bio_marker_string, gender_string, age_string

    output = widgets.Output()
    left_box = VBox([Gene_name,Gender,Bio_marker])
    right_box = VBox([Tumor, Age, Medical_condition])
    display(HBox([left_box, right_box]),button,output)
'''

In [ ]:
def selection_table():
    global wsMultiple, sheet2, df11
    trials_add = pd.read_excel('Filtered_Clinical_Trials.xlsx')
    flt= pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Clinical_trials.xlsx')
    df_data = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Clinical_trials.xlsx', usecols= ['Gene Name', 'HGVS Nomenclature'])
    display(HTML(flt.to_html(render_links=True, escape=False, index=False)))
    
    t=[]
    for ind in trials_add.index:
        trials_gene=trials_add['Gene Name'][ind]
        trials_gene_list=trials_gene.split()
        #print(trials_gene)
        gene_HGVS=[]

        for gname in df_data.index:
            data_gene= df_data['Gene Name'][gname]
            data_gene_list=data_gene.split()
            if data_gene in trials_gene_list:
                gene_HGVS.append(df_data['HGVS Nomenclature'][gname])
        #print(gene_HGVS)
        t.append(gene_HGVS)
    trials_add.insert(7,"HGVS Nomenclature", t)
    trials_add.rename(columns ={'HGVS Nomenclature':'HGVS nomenclature of the variant'}, inplace = True)
    new = ''
    trials_add.insert(loc = 0,column = 'Select',value=False)
    trials_add.style
    new=''
    #trials_add.insert(loc=8, column='HGVS nomenclature of the variant', value = new)
    trials_add.insert(loc=9, column='Does this trial include variant of your interest?', value = new)
    trials_add.insert(loc=10, column='Type/Copy relevant intervention', value = new)
    trials_add[['NCTID','Official Title']] = trials_add[['NCTID','Official Title']].astype(str)
    sheet1 = ipysheet.sheet(columns=9,  rows = trials_add.shape[0],  
                           column_headers = ['NCTID','Official Title','Intervention/Treatment','Condition/Disease','Phase','Trial Location','Gene Name','Does this trial include variant of your interest?','Type/Copy relevant intervention'], row_headers=False)
    chkboxes=[]
    for i in range(len(trials_add)):
        chkboxes.append(False)
    wsMultiple = ipysheet.sheet(ipysheet.from_dataframe(trials_add))


    column0 = ipysheet.column(0, chkboxes)  # New column
    column1 = wsMultiple.cells[1] # Keep the "old" second column

    column2 = wsMultiple.cells[2]
    column3 = wsMultiple.cells[3]
    column4 = wsMultiple.cells[4]
    column5 = wsMultiple.cells[5]
    column6 = wsMultiple.cells[6]
    column7 = wsMultiple.cells[7]
    column8 = wsMultiple.cells[8]
    column9 = wsMultiple.cells[9]
    column10 = wsMultiple.cells[10]

    wsMultiple.cells = (column0,column1, column2,column3,column4,column5,column6,column7,column8,column9,column10) # override all the columns
    wsMultiple.row_headers=False
    display(wsMultiple)
    
    def on_update_button_clicked(b):
        global wsMultiple, sheet2, df11
        wsMultiple.row_headers=False
        df10=to_dataframe(wsMultiple)
        #display(wsMultiple)
        if df10[df10['Select']==True].empty:
            print('No entries have been selected in the clinical trials table') 
        else:
            df10.drop(df10.loc[df10['Select']==False].index, inplace=True)
            df10.drop(['Select'], axis=1, inplace=True)
            print('\033[1m' + '\nDecision table:')
            sheet2 = from_dataframe(df10)
            sheet2.row_headers=False
            display(HTML(df10.to_html(index=False)))
        
        def formated_table(b):
            global sheet2, df11
            df11= to_dataframe(sheet2)
            df11=df11.drop (['Intervention/Treatment', 'Condition/Disease', 'Does this trial include variant of your interest?'], axis = 1)
            df11.rename(columns = {'Gene Name':'Gene', 'HGVS nomenclature of the variant': 'Variant', 'Official Title':'Clinical trial', 'Type/Copy relevant intervention':'Intervention', 'Phase of the study':'Phase', 'NCTID':'Trial Identifier'}, inplace = True)
            df11= df11[['Gene', 'Variant', 'Clinical trial', 'Intervention', 'Phase', 'Trial Identifier', 'Trial Location']]
            display(HTML(df11.to_html(index=False)))
    
            def save_table(b):
                global df11
                title=['Information on Potential Clinical Trials']
                fds=[df11]
                excel_file_name = USER_INPUT_FOLDER_PATH+file+'_Potential_CT.xlsx'
                export_excel(fds, '_Potential_CT', excel_file_name, 2, title)
                title='Click on the links below to download all the tables in excel and word'
                display(Markdown('<strong><br/>{}'.format(title)))
                if (path.exists(file+'_Potential_CT'))==True:
                    display(FileLink(file+'_Potential_CT.xlsx'))
                else:
                    os.symlink(USER_INPUT_FOLDER_PATH+file+'_Potential_CT.xlsx',file+'_Potential_CT.xlsx')
                    display(FileLink(file+'_Potential_CT.xlsx'))
                doc = docx.Document()
                doc_file=USER_INPUT_FOLDER_PATH+file+'_Potential_CT.docx'
                doc.save(doc_file)
                export_doc(df11, "Information on Potential Clinical Trials", doc_file)
                if (path.exists(file+'_Potential_CT.docx'))==True:
                    display(FileLink(file+'_Potential_CT.docx'))
                else:
                    os.symlink(USER_INPUT_FOLDER_PATH+file+'_Potential_CT.docx',file+'_Potential_CT.docx')
                    display(FileLink(file+'_Potential_CT.docx'))
                    
            button = widgets.Button(description="Save Table", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
            button.on_click(save_table)
            display(button)
            
        button = widgets.Button(description="Show the list in GknowMe reporting format", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
        button.on_click(formated_table)
        display(button)
      
    updateButton = widgets.Button(description = "Show Selected Entries", style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'})
    updateButton.on_click(on_update_button_clicked)
    display(updateButton)
    
        

# Chemotherapy Response (only for germline variants)

In [ ]:
clinical_info_news5=pd.DataFrame()
df3 = pd.DataFrame()
def displaytable():
    global clinical_info_news5, df3
    news5 = pd.read_excel(USER_INPUT_FOLDER_PATH+file+'_Display_Table.xlsx')
    news5.drop(news5.index[news5['NGS QC filter'] != 'PASS'], inplace=True)
    #display(news5)
    news5['Depth'] = news5['Depth'].astype(int)
    news5['Allele Frequency'] = news5['Allele Frequency'].astype(str)
    news5['Allele Frequency']=news5['Allele Frequency'].str.replace("%","")
    news5['Allele Frequency'] = news5['Allele Frequency'].astype(float)
    news5 = news5.drop(news5[(news5['Depth'] < 100 )].index)
    news5 = news5.drop(news5[(news5['Allele Frequency'] < 20)].index)
    #display(news5)
    news5 = news5.drop(['Chromosome Number', 'Chromosome Position', 'NGS QC filter', 'Depth', 
                            'Amino acid position', 'Amino acid change', 'dbNSFP Amino acid change', 
                            'REGION', 'Mutation type', 'SIFT prediction', 'HGVS Nomenclature', 'Clinical Significance', 
                            'Minor_allele_frequency', 'OMIM id from ClinVar', 'OMIM Disease from ClinVar', 'OtherIDs', 
                            'COSMIC_link', 'ClinVar_link'], axis = 1)
    news5.rename(columns = {'Gene Name':'Gene'}, inplace = True)

    clinical_info= pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'Clinical annotations15062022.xlsx')
    
    clinical_info['Clinical Annotation ID']=clinical_info['Clinical Annotation ID'].astype(str)
    cl_anno = clinical_info['Clinical Annotation ID'].str.strip()
    #display(cl_anno)
    pmid_info= pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'Clinical_ann_evidence.xlsx')
    pmid_info.columns = pmid_info.columns.str.strip()
    pmid_info = pmid_info.drop(['Evidence ID', 'Evidence Type', 'Evidence URL', 'Summary', 'Score'], axis = 1)
    #display(pmid_info)
    pmid_info.replace(np.NaN, 0, inplace=True)
    pmid_info.replace(np.nan, 0, inplace=True)
    pmid_info.replace(to_replace =["nan", 'nannannan'], 
                                value=0, inplace=True)
    pmid_info['PMID']=pmid_info['PMID'].astype(int)
    #display(pmid_info)
    pmid_info=pmid_info.drop(pmid_info[(pmid_info['PMID']==0)].index)
    pmid_info = pmid_info.drop_duplicates(subset=['PMID','Clinical Annotation ID'], keep='first')
    #display(pmid_info)
    pmid_info['PMID'] = 'PMID:' + pmid_info['PMID'].astype(str)
    pmid_info = pmid_info.groupby(['Clinical Annotation ID'])['PMID'].apply(', '.join).reset_index()
    pmid_info['Clinical Annotation ID']=pmid_info['Clinical Annotation ID'].astype(str)
    pmid_info_lt = pmid_info['Clinical Annotation ID'].str.strip()
    pmid_infolt = pmid_info[pmid_info['Clinical Annotation ID'].isin(cl_anno)]
    #print(len(pmid_infolt))
    #display(HTML(pmid_infolt.to_html(render_links=True, escape=False, index=False)))
    clinical_info_news5= clinical_info.merge(pmid_infolt, how='left')
    
    clinical_info_news5 = clinical_info_news5.merge(news5, how='left')
    
    genotype=[]
    #ind=news5lt.index()
    for ind in clinical_info_news5.index:
        #print(ind)
        if clinical_info_news5['Allele Frequency'][ind]< 80:
            alt= clinical_info_news5['Reference Nucleotide'][ind] + clinical_info_news5['Alternate Nucleotide'][ind]
            
            genotype.append(''.join(sorted(alt)))
        else:
            genotype.append(clinical_info_news5['Alternate Nucleotide'][ind] +  clinical_info_news5['Alternate Nucleotide'][ind])
    
    #print(genotype)
    clinical_info_news5.insert(5,"Genotype", value= genotype)
    clinical_info_news5= clinical_info_news5.drop(['Reference Nucleotide', 'Alternate Nucleotide', 'Allele Frequency', 'dbSNP'], axis = 1)
    #display(clinical_info_news5)
    
    anno_info= pd.read_excel(EXTERNAL_FILES_FOLDER_PATH+'Clinical_ann_alleles.xlsx')
    anno_info.columns = anno_info.columns.str.strip()
    anno_info = anno_info.drop(['Allele Function'], axis = 1)
    anno_info['Clinical Annotation ID']=anno_info['Clinical Annotation ID'].astype(str)
    anno_info_lt=anno_info['Clinical Annotation ID'].str.strip()

    clinical_info_news5['Clinical Annotation ID']=clinical_info_news5['Clinical Annotation ID'].astype(str)
    cl_anno_lt= clinical_info_news5['Clinical Annotation ID'].str.strip()


    anno_info['Genotype/Allele'] = anno_info['Genotype/Allele'].astype(str)
    clinical_info_news5['Genotype'] = clinical_info_news5['Genotype'].astype(str)
    aachn = clinical_info_news5['Genotype'].str.strip()
    l=list(aachn)
    t=[]
    for i in l:
        sub=i.split(' ')
        #print(sub)
        for k in sub:
                sub1=k.replace(',', '')
                t.append(sub1)

    anno_text = anno_info[anno_info['Clinical Annotation ID'].isin(cl_anno_lt) & anno_info['Genotype/Allele'].isin(t)]
    anno_text.reset_index(inplace = True, drop = True)
    anno_text.rename(columns = {'Genotype/Allele':'Genotype'}, inplace = True)
    #display(anno_text)
    df_merged = pd.merge(anno_text, clinical_info_news5, on=['Clinical Annotation ID', 'Genotype'], how='left')
    #display(df_merged)
    
    df_merged['Variant/Haplotypes'] = df_merged['Variant/Haplotypes'].str.replace('\n','')
    clinical_info_lt = df_merged['Variant/Haplotypes'].str.strip()
    #display(news5)
    news5['dbSNP']=news5['dbSNP'].str.strip()
    #display(clinical_info_lt)
    news5_lt = news5['dbSNP'].str.strip()
    news5lt = news5[news5['dbSNP'].isin(clinical_info_lt)]
    clinical_info_news5 =df_merged[df_merged['Variant/Haplotypes'].isin(news5_lt)]
    #display(clinical_info_news5)

In [ ]:
def chemo_filter():
    global clinical_info_news5, df3
    #display(clinical_info_news5)
    clinical_info_news5['Phenotype Category'] = clinical_info_news5['Phenotype Category'].astype(str)
    #clinical_info_news5['Phenotype(s)'] = clinical_info_news5['Phenotype(s)'].str.replace(' ', '-')
    clinical_info_news5['Level of Evidence'] = clinical_info_news5['Level of Evidence'].astype(str)
    clinical_info_news5['Level of Evidence']=clinical_info_news5['Level of Evidence'].str.replace("%","")
    #clinical_info_news5.replace(to_replace =['nan'], value =" ", inplace=True)
    clinical_info_news5.replace(np.NaN, '', inplace=True)
    clinical_info_news5.replace(np.nan, '', inplace=True)
    clinical_info_news5.replace(to_replace =["nan", 'nannannan'], 
                                value ="", inplace=True)
    clinical_info_news5['Phenotype(s)']=clinical_info_news5['Phenotype(s)'].str.strip()
    phe_type=list(clinical_info_news5['Phenotype(s)'].unique())
    clinical_info_news5['Specialty Population']=clinical_info_news5['Specialty Population'].str.strip()
    spe_pop=list(clinical_info_news5['Specialty Population'].unique())
    clinical_info_news5['Phenotype Category']=clinical_info_news5['Phenotype Category'].str.strip()
    phe_cat=list(clinical_info_news5['Phenotype Category'].unique())
    #print(phe_cat)
    clinical_info_news5['Level of Evidence']=clinical_info_news5['Level of Evidence'].str.strip()
    evd_level=list(clinical_info_news5['Level of Evidence'].unique())
    #print(evd_level) 

    a= widgets.SelectMultiple(
            options= phe_cat,
            value= phe_cat,
            #rows=10,
            description='Efficacy/Toxicity:',
            disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'}
        )
    b = widgets.SelectMultiple(
        options= evd_level, 
        value= evd_level,
        #rows=10,
        description='Level of evidence:',
        disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'}
        )
    
    c=widgets.SelectMultiple(
        options= phe_type,
        #nan=False, 
        value=phe_type,
        #rows=10,
        description='Phenotypes:',
        disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'}
        )
    d=widgets.SelectMultiple(
        options= spe_pop,
        #nan=False, 
        value=spe_pop,
        #rows=10,
        description='Specialty Population:',
        disabled=False,style= {'description_width': 'initial'},display='flex',layout={'width': 'max-content'}
        )

    
    items = [a,b,c,d]
    left_box = VBox([items[0], items[3]])
    right_box = VBox([items[1], items[2]])
    box_layout = Layout(display='flex',
        flex_flow='row',
        align_items='stretch',justify_content ='space-around', 
        #border='solid'
        width='100%')
        #a = HBox([left_box,right_box], layout=box_layout)
    ui = HBox([VBox([items[0], items[2], items[3]]), VBox([items[1]])], layout=box_layout)
    def reset_button(defaults={}):
        def on_button_clicked(_):
            for k, v in defaults.items(): 
                k.value = v
        button = widgets.Button(description='Reset',layout={'width': 'max-content'})
        button.on_click(on_button_clicked)
        display(button)
    
    reset_button(defaults={ a:a.options, b: b.options, c: c.options, d:d.options})
    
    def show_df(Phe_type, Phe_Cat, Level_Evi, Spec_Pop):
        global clinical_info_news5, df3
        df = clinical_info_news5
        df.style.set_table_styles([{'selector':'','props':[('border','4px solid #7a7')]}]).hide_index()
        dfe = pd.DataFrame(columns=['Phenotype(s)'])
        l=[]
        for pheno in Phe_type:
            
            l.append(pheno)

        #print(l)

        l1=[]
        for cat in Phe_Cat:
            l1.append(cat)

        l2=[]
        for loe in Level_Evi:
            l2.append(loe)
        #print(l2)
        l3=[]
        for sp in Spec_Pop:
            l3.append(sp)
        #print(l3)

        dff=(df[df['Phenotype(s)'].isin(l)])
            #print(dff.shape[0])
            #display(dff)  
        dff1=(dff[dff['Phenotype Category'].isin(l1)])
        dff2=(dff1[dff1['Level of Evidence'].isin(l2)])
        dff3=(dff2[dff2['Specialty Population'].isin(l3)])
        pd.set_option('display.max_columns', 500)
        print('\n',dff3.shape[0],  'entries are selected after applying the filters')
        #dff2['Variant/Haplotypes'] = dff2['Variant/Haplotypes'].astype(str)
        dff3['Clinical Annotation ID'] = dff3['Clinical Annotation ID'].astype(str)
        x = dff3.loc[dff3['Clinical Annotation ID'].str.contains('X', case=False, na=False)]
        dff3.drop(dff3.index[dff3['Clinical Annotation ID'] == 'X'], inplace=True)
        dff3.loc[:,"Clinical Annotation ID"] = dff3.loc[:,"Clinical Annotation ID"].astype(str).apply(lambda x: x.replace('.0',''))
        dff3['Clinical Annotation ID'] = dff3['Clinical Annotation ID'].astype(int)
        dff3.sort_values(by=['Clinical Annotation ID'], inplace=True)
        x.sort_values(by=['Clinical Annotation ID'], inplace=True)
        df3=dff3.append(x)
        display(HTML(df3.to_html(render_links=True, escape=False, index=False)))
    w=interactive_output(show_df, {'Phe_Cat':a, 'Level_Evi':b, 'Phe_type':c, 'Spec_Pop':d  })
    display(ui, w)

    

# Tables for Chemotherapy (Toxicity and Efficacy)

In [ ]:
def summary_toxicity():
    global df3, toxicity_chemo, det_toxicity, efficacy_chemo, det_efficacy 
    toxicity_chemo = df3
    #display(toxicity_chemo)
    toxicity_chemo['Phenotype Category']=toxicity_chemo['Phenotype Category'].astype(str)
    df12= toxicity_chemo['Phenotype Category'].str.contains('Toxicity', case=False)
    #df12= df12[df12['Phenotype Category']==True].index
    toxicity_chemo=toxicity_chemo[df12]
    toxicity_chemo= toxicity_chemo.drop(['Clinical Annotation ID', 'Annotation Text', 'Level Override', 'Level Modifiers', 'Score', 'Phenotype Category', 'PMID Count', 'Evidence Count', 'Phenotype(s)', 'Latest History Date (YYYY-MM-DD)', 'URL', 'Specialty Population', 'PMID'], axis = 1)
    toxicity_chemo.rename(columns = {'Variant/Haplotypes':'Mutation site', 'Drug(s)':'Chemotherapy', 'Level of Evidence': 'PharmGKB Evidence Level for associationbetween genotype and chemotherapy toxicity'}, inplace = True)
    #print(toxicity_chemo.columns.tolist())
    toxicity_chemo= toxicity_chemo[['Gene', 'Mutation site', 'Genotype', 'Chemotherapy', 'PharmGKB Evidence Level for associationbetween genotype and chemotherapy toxicity']]
    print(color.BOLD+'\n Summary for Chemotherapy Toxicity' + color.END)
    #print(len(toxicity_chemo))
    display(HTML(toxicity_chemo.to_html(render_links=True, escape=False, index=False))) 
    
    det_toxicity = df3
    det_toxicity['Phenotype Category']=det_toxicity['Phenotype Category'].astype(str)
    df12= det_toxicity['Phenotype Category'].str.contains('Toxicity', case=False)
    #df12= df12[df12['Phenotype Category']==True].index
    det_toxicity=det_toxicity[df12]
    det_toxicity= det_toxicity.drop(['Clinical Annotation ID', 'Level Override', 'Level Modifiers', 'Score', 'Phenotype Category', 'PMID Count', 'Evidence Count', 'Phenotype(s)', 'Latest History Date (YYYY-MM-DD)', 'URL', 'Specialty Population'], axis = 1)
    det_toxicity.rename(columns = {'Variant/Haplotypes':'Mutation site', 'Drug(s)':'Chemotherapy', 'Level of Evidence': 'Evidence level'}, inplace = True)
    alt= det_toxicity['Annotation Text'] + det_toxicity['PMID'].astype(str)
    det_toxicity.insert(loc=4, column='Potential relevance to chemotherapy toxicity', value= alt)
    det_toxicity= det_toxicity[['Gene', 'Mutation site', 'Genotype', 'Chemotherapy', 'Potential relevance to chemotherapy toxicity', 'Evidence level']]
    print(color.BOLD+'\n Detected mutation and their relavance to chemotherapy toxicity' + color.END)
    #print(len(det_toxicity))
    display(HTML(det_toxicity.to_html(render_links=True, escape=False, index=False))) 
    
    
def summary_efficacy():
    global df3, toxicity_chemo, det_toxicity, efficacy_chemo, det_efficacy 
    efficacy_chemo = df3
    #display(toxicity_chemo)
    efficacy_chemo['Phenotype Category']=efficacy_chemo['Phenotype Category'].astype(str)
    df12= efficacy_chemo['Phenotype Category'].str.contains('Efficacy', case=False)
    #df12= df12[df12['Phenotype Category']==True].indeefficacy
    efficacy_chemo=efficacy_chemo[df12]
    efficacy_chemo= efficacy_chemo.drop(['Clinical Annotation ID', 'Annotation Text', 'Level Override', 'Level Modifiers', 'Score', 'Phenotype Category', 'PMID Count', 'Evidence Count', 'Phenotype(s)', 'Latest History Date (YYYY-MM-DD)', 'URL', 'Specialty Population', 'PMID'], axis = 1)
    efficacy_chemo.rename(columns = {'Variant/Haplotypes':'Mutation site', 'Drug(s)':'Chemotherapy', 'Level of Evidence': 'PharmGKB Evidence Level for associationbetween genotype and chemotherapy toxicity'}, inplace = True)
    #print(toxicity_chemo.columns.tolist())
    efficacy_chemo= efficacy_chemo[['Gene', 'Mutation site', 'Genotype', 'Chemotherapy', 'PharmGKB Evidence Level for associationbetween genotype and chemotherapy toxicity']]
    print(color.BOLD+'\n Summary for Chemotherapy Efficacy' + color.END)
    #print(len(efficacy_chemo))
    display(HTML(efficacy_chemo.to_html(render_links=True, escape=False, index=False))) 
    
    det_efficacy = df3
    det_efficacy['Phenotype Category']=det_efficacy['Phenotype Category'].astype(str)
    df12= det_efficacy['Phenotype Category'].str.contains('Efficacy', case=False)
    #df12= df12[df12['Phenotype Category']==True].index
    det_efficacy=det_efficacy[df12]
    det_efficacy= det_efficacy.drop(['Clinical Annotation ID', 'Level Override', 'Level Modifiers', 'Score', 'Phenotype Category', 'PMID Count', 'Evidence Count', 'Phenotype(s)', 'Latest History Date (YYYY-MM-DD)', 'URL', 'Specialty Population'], axis = 1)
    det_efficacy.rename(columns = {'Variant/Haplotypes':'Mutation site', 'Drug(s)':'Chemotherapy', 'Level of Evidence': 'Evidence level'}, inplace = True)
    alt= det_efficacy['Annotation Text'] + det_efficacy['PMID'].astype(str) 
    det_efficacy.insert(loc=4, column='Potential relevance to chemotherapy toxicity', value= alt)
    det_efficacy= det_efficacy[['Gene', 'Mutation site', 'Genotype', 'Chemotherapy', 'Potential relevance to chemotherapy toxicity', 'Evidence level']]
    print(color.BOLD+'\n Detected mutation and their relavance to chemotherapy efficacy' + color.END)
    #print(len(det_efficacy))
    display(HTML(det_efficacy.to_html(render_links=True, escape=False, index=False)))

# Export chemotherapy tables

In [ ]:
def export_chemo_table():
    global df3, toxicity_chemo, det_toxicity, efficacy_chemo, det_efficacy 
    title=['Summary for Chemotherapy Toxicity', 'Detected mutation and their relavance to chemotherapy toxicity', 'Summary for Chemotherapy Efficacy', 'Detected mutation and their relavance to chemotherapy efficacy']
    #display(toxicity_chemo, det_toxicity, efficacy_chemo, det_efficacy)
    fds=[toxicity_chemo, det_toxicity, efficacy_chemo, det_efficacy]
    excel_file_name = USER_INPUT_FOLDER_PATH+file+'_Chemotherapy_Tables.xlsx'
    export_excel(fds, '_Chemotherapy_Tables', excel_file_name, 2, title)
    title='Click on the links below to download all the tables in excel'
    display(Markdown('<strong><br/>{}'.format(title)))
    if (path.exists(file+'_Chemotherapy_Tables.xlsx'))==True:
        display(FileLink(file+'_Chemotherapy_Tables.xlsx'))
    else:
        os.symlink(USER_INPUT_FOLDER_PATH+file+'_Chemotherapy_Tables.xlsx',file+'_Chemotherapy_Tables.xlsx')
        display(FileLink(file+'_Chemotherapy_Tables.xlsx'))